# minecu_tg: TinyGrad vs the hand-written CUDA, on this T4

The same voxel DDA, written twice: once by hand in CUDA
(`minecu_ref/src/kernels.cu`, 3,245 lines, vendored unmodified)
and once as plain tensor code in TinyGrad (`minecu_tg/`, the
payload in the cell below). This notebook measures the second
against the first on this card, in this session.

The TinyGrad side is not tuned. It is what a person writes when
handed a specification and asked to use array operations and a
loop -- no tile sizes, no `Context(...)`, no custom kernel, no
hand-unrolling. Read `minecu_tg/render.py`'s docstring for the
exact list of things it is not allowed to do.

## What this produces

1. `SUMMARY.txt` -- the full log, the primary artefact.
2. `ref_timing.json` -- the hand-written side, per batch.
3. `tg/report.json` -- the TinyGrad side: census, memory, timing.
4. `obs_equivalence.json` -- the two observations compared on the
   state the reference itself produced.
5. `tg_codegen/` -- the generated CUDA, compiled for sm_75, with
   the SASS and the `ptxas -v` resource report.
6. `log_bench_dda.txt` -- the reference harness's own DDA
   statistics, for the workload comparison.

No number here is copied from a notes file. Every figure is
produced by an instrument in this notebook.

In [ ]:

import base64, hashlib, json, os, re, subprocess, sys, time

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
EXPECT = {"c6ed8f06ee64d40a8a8ab7ebce3b3ceea1e3ac93f9ee50cfc1e63d3f090d878f": "kernels.cu", "d3a0931d642d67bd5e6fbe22e77a732da72edc3502ca8296f87f615a224965e8": "refdrv.cu", "990fdfc069ff8e59dcce5ee85ec2068a99b7b575b5800b1429185b066d3964a5": "bench.cu", "6f376400341dd1015acd5f32e687e2fed7a85b4a26d4269b71680ec1f0c0d7d5": "minecu_tg/__init__.py", "752ffaea2028475df25e81eaa186ce8ae089e36eed53d96eeecd531663dabed9": "minecu_tg/blocks.py", "6c75c0e1e9cfc105e152f06c63c7695eff08781b9b8aa3e49ba098fe299ffbc3": "minecu_tg/world.py", "fe85581f80ed7fdba57398a6c04fe744a7fd59917d99b132ea0eb2dcfa457952": "minecu_tg/render.py", "d3d6356552db2be483430751924970d288cb1c128a56709aa074d7e18a908108": "minecu_tg/cpu_reference.py", "a79e7cdcfbbb10c7b4adef880f87a27b3d471489c283c6d5c07d74aa80843caf": "minecu_tg/verify.py", "f926389a8c3064d17224da3f6152330468712e74f27e95a0ff628d6e97a05fb3": "minecu_tg/codegen.py", "7ec6927c711e0eec2b4de27d8eb03bb6c2f7e0d660edd8dfa969dba61434c539": "minecu_tg/bench.py", "b363a884cfad373559af625926a29d34443e628002bb00a752d453d3ce3f96f8": "minecu_tg/battery.py", "f1957fb652054f63b93e9bbf0d5629391504f7bc2116eef1cc4192f874a97d0b": "minecu_tg/capture_cuda.py", "0972d45d060a1f6426c1bc6376d59788d145a5563c0cc0c66e8931ddc7382ec7": "minecu_tg/analyse_codegen.py", "dedf5437ebfbaf0b574203f7dd5960d895cf2c6aeb260936b4834aed795a23af": "minecu_tg/sass_matrix.py", "1db65d800a1f4ac2820a273a6e6a221de87650f3365fe2784a8182eea0f35708": "tools/archlab/inspect_ptx.py"}
PAYLOADS = [["kernels.cu", "..."], ["refdrv.cu", "..."], ["bench.cu", "..."], ["minecu_tg/__init__.py", "IiIibWluZWN1X3RnIC0tIHRoZSBzYW1lIHZveGVsIEREQSwgd3JpdHRlbiBhcyBwbGFpbiBUaW55R3JhZC4KCkEgaGFuZC13cml0dGVuIENVREEgdm94ZWwgcmF5bWFyY2hlciBleGlzdHMgYXQgYG1pbmVjdV9yZWYvc3JjL2tlcm5lbHMuY3VgCigzLDI0NSBsaW5lcywgdW5tb2RpZmllZCkuIFRoaXMgcGFja2FnZSBpcyB0aGUgc2FtZSBjb21wdXRhdGlvbiB3cml0dGVuIHRoZSB3YXkKYSBwZXJzb24gd3JpdGVzIGl0IHdpdGggYXJyYXkgb3BlcmF0aW9ucyBhbmQgYSBsb29wLCBzbyB0aGUgc3R1ZHkgY2FuIG1lYXN1cmUKd2hlcmUgYSBjb2RlIGdlbmVyYXRvcidzIG91dHB1dCBsYW5kcyByZWxhdGl2ZSB0byBoYW5kLXdyaXR0ZW4gY29kZS4KCiAgICByZW5kZXIucHkgICAgICAgICB0aGUgRERBIHJlbmRlciwgYXMgdGVuc29yIG9wcyAgICAgICAgICA8LSB0aGUgc3ViamVjdAogICAgYmxvY2tzLnB5ICAgICAgICAgYmxvY2sgaWRzIGFuZCB0aGUgY29sb3VyIHRhYmxlLCBmcm9tIHRoZSByZWZlcmVuY2UKICAgIHdvcmxkLnB5ICAgICAgICAgIHRoZSB3b3JsZGdlbiBrZXJuZWxzLCBhcyB0ZW5zb3Igb3BzCiAgICBjcHVfcmVmZXJlbmNlLnB5ICBhbiBpbmRlcGVuZGVudCBmbG9hdDY0IENQVSB0cmFuc2NyaXB0aW9uCiAgICB2ZXJpZnkucHkgICAgICAgICB0aGUgY29ycmVjdG5lc3MgZ2F0ZQogICAgY29kZWdlbi5weSAgICAgICAga2VybmVsLWNvdW50IGNlbnN1cyBhbmQgQ1VEQSBzb3VyY2UgY2FwdHVyZQogICAgYmVuY2gucHkgICAgICAgICAgdGltaW5nIG9uIGEgcmVhbCBjYXJkCiAgICBrYWdnbGUvICAgICAgICAgICB0aGUgcHVzaCByZWNpcGUKClJlYWQgYHJlbmRlci5weWAncyBkb2NzdHJpbmcgYmVmb3JlIHF1b3RpbmcgYW55dGhpbmcgYWJvdXQgd2hhdCB0aGlzIGNvZGUgZG9lczsKdGhlIGNvbnN0cmFpbnRzIHRoYXQgc2hhcGVkIGl0IGFyZSBwYXJ0IG9mIHRoZSByZXN1bHQuCgpOT1RFIE9OIFRIRSBOQU1FUy4gYHJlbmRlcmAgaXMgdGhlICptb2R1bGUqLCBhbmQgYHJlbmRlci5yZW5kZXJgIGlzIHRoZQpmdW5jdGlvbi4gVGhpcyBwYWNrYWdlIGRlbGliZXJhdGVseSBkb2VzIG5vdCByZS1leHBvcnQgdGhlIGZ1bmN0aW9uIHVuZGVyIHRoZQpiYXJlIG5hbWUgYHJlbmRlcmAsIGJlY2F1c2UgdGhhdCB3b3VsZCBvdmVyd3JpdGUgdGhlIG1vZHVsZSBhdHRyaWJ1dGUgYW5kIG1ha2UKYG1pbmVjdV90Zy5yZW5kZXIuY2FtZXJhX2Jhc2lzYCBhIGNvbmZ1c2luZyBBdHRyaWJ1dGVFcnJvciBsYXRlci4gVXNlCgogICAgZnJvbSBtaW5lY3VfdGcucmVuZGVyIGltcG9ydCByZW5kZXIsIGNhbWVyYV9iYXNpcwoKb3IgdGhlIGByZW5kZXJfb2JzYCBhbGlhcyBiZWxvdy4KIiIiCgpmcm9tIC5yZW5kZXIgaW1wb3J0IGNhbWVyYV9iYXNpcwpmcm9tIC5yZW5kZXIgaW1wb3J0IHJlbmRlciBhcyByZW5kZXJfb2JzCmZyb20gLndvcmxkIGltcG9ydCBleHBhbmRfd29ybGQsIGdlbmVyYXRlX3dvcmxkLCBncm91bmRfb2YKCl9fYWxsX18gPSBbImNhbWVyYV9iYXNpcyIsICJyZW5kZXJfb2JzIiwgImdlbmVyYXRlX3dvcmxkIiwgImV4cGFuZF93b3JsZCIsICJncm91bmRfb2YiXQo=", "6f376400341dd1015acd5f32e687e2fed7a85b4a26d4269b71680ec1f0c0d7d5"], ["minecu_tg/blocks.py", "IiIiQmxvY2sgaWRzIGFuZCB0aGUgY29sb3VyIHRhYmxlcywgdHJhbnNjcmliZWQgZnJvbSB0aGUgcmVmZXJlbmNlLgoKRXZlcnl0aGluZyBoZXJlIGlzIGNvcGllZCBmcm9tIGBtaW5lY3VfcmVmL3NyYy9rZXJuZWxzLmN1YCwgd2hpY2ggaXMgdGhlCmhhbmQtd3JpdHRlbiBzaWRlIGFuZCBpcyBuZXZlciBtb2RpZmllZC4gTGluZSBudW1iZXJzIGluIHRoZSBjb21tZW50cyBhcmUgdGhlCnJlZmVyZW5jZSdzLgoKICAgIGtlcm5lbHMuY3U6MTEtMjQgICAgdGhlIDEzIGJsb2NrIGlkcywgQUlSID0gLTEKICAgIGtlcm5lbHMuY3U6NDc5LTQ5MyAgQkxPQ0tfQ09MT1JTX1U4CiAgICBrZXJuZWxzLmN1OjQ5NSAgICAgIFNLWV9DT0xPUl9VOAoiIiIKCmZyb20gdGlueWdyYWQgaW1wb3J0IFRlbnNvciwgZHR5cGVzCgpBSVIgPSAtMQpHUkFTUywgRElSVCwgU1RPTkUsIE9BS0xPRywgTEVBVkVTLCBTQU5ELCBXQVRFUiwgR0xBU1MsIEJSSUNLLCBDT0JCTEVTVE9ORSwgXAogICAgUExBTktTLCBTTk9XLCBCRURST0NLID0gcmFuZ2UoMTMpCgpCTE9DS19OQU1FUyA9ICgiR1JBU1MiLCAiRElSVCIsICJTVE9ORSIsICJPQUtMT0ciLCAiTEVBVkVTIiwgIlNBTkQiLCAiV0FURVIiLAogICAgICAgICAgICAgICAiR0xBU1MiLCAiQlJJQ0siLCAiQ09CQkxFU1RPTkUiLCAiUExBTktTIiwgIlNOT1ciLCAiQkVEUk9DSyIpCgojIGtlcm5lbHMuY3U6NDc5CkJMT0NLX0NPTE9SU19VOCA9IFsKICAgIFs3NiwgMTY2LCA1MV0sICAgICAgIyBHUkFTUwogICAgWzE0MCwgODksIDUxXSwgICAgICAjIERJUlQKICAgIFsxMjcsIDEyNywgMTI3XSwgICAgIyBTVE9ORQogICAgWzE0MCwgMTAyLCA2NF0sICAgICAjIE9BS0xPRwogICAgWzUxLCAxNDAsIDM4XSwgICAgICAjIExFQVZFUwogICAgWzIyOSwgMjE3LCAxNTNdLCAgICAjIFNBTkQKICAgIFs1MSwgMTAyLCAyMDRdLCAgICAgIyBXQVRFUgogICAgWzIxNywgMjI5LCAyNDJdLCAgICAjIEdMQVNTCiAgICBbMTc4LCA4OSwgNzZdLCAgICAgICMgQlJJQ0sKICAgIFsxMDIsIDEwMiwgMTAyXSwgICAgIyBDT0JCTEVTVE9ORQogICAgWzE5MSwgMTUzLCAxMDJdLCAgICAjIFBMQU5LUwogICAgWzI0MiwgMjQyLCAyNTBdLCAgICAjIFNOT1cKICAgIFszOCwgMzgsIDM4XSwgICAgICAgIyBCRURST0NLCl0KCiMga2VybmVscy5jdTo0OTUKU0tZX0NPTE9SX1U4ID0gWzEzNSwgMjA2LCAyMzVdCgoKZGVmIGNvbG91cl9sdXQoKToKICAgICIiIlRoZSBjb2xvdXIgdGFibGUgYXMgb25lIFsxNCwgM10gdWludDggdGFibGUuCgogICAgUm93IDAgaXMgR1JBU1MgLi4gcm93IDEyIGlzIEJFRFJPQ0ssIGV4YWN0bHkgYXMgYEJMT0NLX0NPTE9SU19VOGAgaXMKICAgIGluZGV4ZWQgaW4gdGhlIHJlZmVyZW5jZSAoYEJMT0NLX0NPTE9SU19VOFtoaXRfYmxvY2tdYCwgd2l0aCBoaXRfYmxvY2sgaW4KICAgIFswLCAxMl0pLiBSb3cgMTMgaXMgdGhlIHNreSwgd2hpY2ggdGhlIHJlZmVyZW5jZSByZWFjaGVzIGJ5IGEgKmJyYW5jaCoKICAgIHJhdGhlciB0aGFuIGFuIGluZGV4OiBgaGl0X2Jsb2NrYCBpcyBpbml0aWFsaXNlZCB0byBBSVIgPT0gLTEgYW5kCiAgICBgb3V0cHV0ID0gU0tZX0NPTE9SX1U4YCB3aGVuIHRoZSBEREEgZmluZHMgbm90aGluZy4gQXBwZW5kaW5nIHRoZSBza3kgYXMKICAgIHJvdyAxMyB0dXJucyB0aGF0IGJyYW5jaCBpbnRvIHRoZSBzYW1lIHRhYmxlIGxvb2t1cCB0aGUgaGl0IHBhdGggYWxyZWFkeQogICAgZG9lcywgc28gImRpZCBub3QgaGl0IiBhbmQgImhpdCIgYmVjb21lIG9uZSBnYXRoZXIuIFRoYXQgaXMgYSBjaGFuZ2Ugb2YKICAgICpzaGFwZSosIG5vdCBvZiBzY2hlZHVsZTogbm8gY29kZWdlbiBrbm9iLCBubyBmdXNpb24gaGludCwgbm8gdGlsZSBzaXplLgogICAgSXQgaXMgd2hhdCBhIHBlcnNvbiB3cml0aW5nIHRoZSB0ZW5zb3IgdmVyc2lvbiB3b3VsZCBkbyB0byBhdm9pZCBhbgogICAgYGlmYCBvbiBhIHRlbnNvci4KICAgICIiIgogICAgcm93cyA9IFtsaXN0KG1hcChpbnQsIGMpKSBmb3IgYyBpbiBCTE9DS19DT0xPUlNfVThdICsgW2xpc3QobWFwKGludCwgU0tZX0NPTE9SX1U4KSldCiAgICByZXR1cm4gVGVuc29yKHJvd3MsIGR0eXBlPWR0eXBlcy51aW50OCkK", "752ffaea2028475df25e81eaa186ce8ae089e36eed53d96eeecd531663dabed9"], ["minecu_tg/world.py", "IiIiVGhlIHZveGVsIHdvcmxkLCBhcyB0ZW5zb3Igb3BzLgoKVHJhbnNjcmliZWQgZnJvbSB0aHJlZSBoYW5kLXdyaXR0ZW4ga2VybmVscywgdW5tb2RpZmllZDoKCiAgICBnZW5lcmF0ZV93b3JsZF9rZXJuZWwgICBrZXJuZWxzLmN1OjE3OTMKICAgIHBsYWNlX3RyZWVfa2VybmVsICAgICAgIGtlcm5lbHMuY3U6MTgzMQogICAgcGxhY2VfaG91c2Vfa2VybmVsICAgICAga2VybmVscy5jdToxODY4CgpXcml0dGVuIHRoZSB3YXkgYSBwZXJzb24gd3JpdGVzIGl0IHdpdGggYXJyYXkgb3BlcmF0aW9uczogYnJvYWRjYXN0IHRoZSB0aHJlZQpjb29yZGluYXRlIGF4ZXMgYWdhaW5zdCBlYWNoIG90aGVyLCBidWlsZCB0aGUgYmxvY2sgaWQgd2l0aCBuZXN0ZWQgYHdoZXJlYCwKdGhlbiBzdGFtcCB0aGUgdHJlZSBhbmQgdGhlIGhvdXNlIGJ5IHNjYXR0ZXJpbmcgYSBoYW5kZnVsIG9mIGNvbnN0YW50IHZveGVscy4KClR3byBwcm9wZXJ0aWVzIG9mIHRoZSByZWZlcmVuY2UgYXJlIHdvcnRoIHN0YXRpbmcgYmVjYXVzZSB0aGV5IGFyZSBsb2FkLWJlYXJpbmcKZm9yIHRoZSBjb21wYXJpc29uIGFuZCBuZWl0aGVyIGlzIGEgY29kZWdlbiBjaG9pY2U6CgogICogKipUaGUgd29ybGQgaXMgaWRlbnRpY2FsIGZvciBldmVyeSBlbnYuKiogYGdlbmVyYXRlX3dvcmxkX2tlcm5lbGAgYWNjZXB0cyBhCiAgICBgc2VlZGAgYXJndW1lbnQgYW5kIG5ldmVyIHJlYWRzIGl0OyB0aGUgdHJlZSBhbmQgdGhlIGhvdXNlIGFyZSBwbGFjZWQgYXQKICAgIGZpeGVkIG9mZnNldHMuIFNvIGBnZW5lcmF0ZV93b3JsZGAgcmV0dXJucyBvbmUgW1ksIFgsIFpdIHZvbHVtZSBhbmQKICAgIGBleHBhbmRgIGdpdmVzIGV2ZXJ5IGVudiB0aGUgc2FtZSBvbmUuIFRoZSByZWZlcmVuY2UgbWF0ZXJpYWxpc2VzIEIKICAgIGlkZW50aWNhbCBjb3BpZXMgYW5kIHRoZSByZW5kZXIga2VybmVsIHJlYWRzIEIgc2VwYXJhdGUgYWRkcmVzc2VzOyB0aGlzCiAgICBtYXR0ZXJzIGZvciB0aGUgbWVtb3J5LXRyYWZmaWMgaGFsZiBvZiB0aGUgY29tcGFyaXNvbiBhbmQgaXMgY2FsbGVkIG91dAogICAgdGhlcmUgcmF0aGVyIHRoYW4gYmVpbmcgcGFwZXJlZCBvdmVyLgoKICAqICoqQXQgd29ybGQ9MTYgdGhlIGhvdXNlIGRvZXMgbm90IGZpdC4qKiBJdHMgY29ybmVyIGlzCiAgICBgKHNwYXduX3ggKyA2LCBncm91bmQsIHNwYXduX3ogLSAyKWAgPSBgKDE0LCBncm91bmQsIDYpYCBhbmQgaXQgaXMgNSB3aWRlLAogICAgc28gYGR4YCBpbiB7MiwgMywgNH0gbGFuZCBhdCB4IGluIHsxNiwgMTcsIDE4fSBhbmQgYXJlIGRyb3BwZWQgYnkgdGhlCiAgICBib3VuZHMgdGVzdC4gYGRkYV9tb2RlbC5weTo3MS0xMDNgIHNheXMgdGhlIHNhbWUgdGhpbmcgYWJvdXQgdGhlIHNhbWUKICAgIHdvcmxkLiBUaGUgaGFsZi1ob3VzZSBpcyB3aGF0IHRoZSByZWZlcmVuY2UgcmVuZGVycywgc28gdGhlIHRlbnNvciB2ZXJzaW9uCiAgICBidWlsZHMgYSBoYWxmLWhvdXNlIHRvbzsgdGhlIGJvdW5kcyB0ZXN0IGlzIHdyaXR0ZW4gb3V0IHJhdGhlciB0aGFuIGFzc3VtZWQuCiIiIgoKZnJvbSB0aW55Z3JhZCBpbXBvcnQgVGVuc29yLCBkdHlwZXMKCmZyb20gLmJsb2NrcyBpbXBvcnQgQUlSLCBCRURST0NLLCBESVJULCBHUkFTUywgT0FLTE9HLCBMRUFWRVMsIENPQkJMRVNUT05FLCBQTEFOS1MsIEJSSUNLCgoKZGVmIGdyb3VuZF9vZih3b3JsZDogaW50LCBncm91bmQ6IGludCB8IE5vbmUgPSBOb25lKSAtPiBpbnQ6CiAgICAiIiJUaGUgZ3JvdW5kIGhlaWdodCwgZGVyaXZlZCBmcm9tIHRoZSB3b3JsZCBzaXplIHdoZW4gbm90IGdpdmVuLgoKICAgIGBwaHlzaWNzX2tlcm5lbGAgZmxvb3JzIHRoZSBhZ2VudCBvbiBhIHBsYW5lIGF0IGB3b3JsZF95IC8gNGAgKmludGVnZXIqCiAgICBkaXZpc2lvbiAoa2VybmVscy5jdToxNTY2KSBhbmQgYGdlbmVyYXRlX3dvcmxkX2tlcm5lbGAgcHV0cyB0aGUgZ3Jhc3MgYXQKICAgIGBncm91bmRfaGVpZ2h0YCAoa2VybmVscy5jdToxODIzKS4gVGhlIHR3byBhZ3JlZSBvbmx5IHdoZW4KICAgIGBncm91bmRfaGVpZ2h0ID09IHdvcmxkX3kgLyA0YCwgYW5kIGRpc2FncmVlaW5nIGJ1cmllcyB0aGUgYWdlbnQgaW5zaWRlCiAgICBpdHMgb3duIHN1cmZhY2UsIHdoaWNoIGNvbGxhcHNlcyB0aGUgRERBIHRvIH4xLjMgc3RlcHMvcGl4ZWwuIERlcml2aW5nIGl0CiAgICBpcyB0aGUgY29uZmlndXJhdGlvbiB0aGUgYmFzZWxpbmUgd2FzIG1lYXN1cmVkIGluCiAgICAoYG1pbmVjdV9yZWYvYnVyaWVkX2FnZW50X2ZpeC5tZGApLgogICAgIiIiCiAgICByZXR1cm4gd29ybGQgLy8gNCBpZiBncm91bmQgaXMgTm9uZSBlbHNlIGdyb3VuZAoKCmRlZiBfYXhpcyhuOiBpbnQsIGRpbTogaW50KSAtPiBUZW5zb3I6CiAgICByZXR1cm4gVGVuc29yKGxpc3QocmFuZ2UobikpLCBkdHlwZT1kdHlwZXMuaW50MzIpLnJlc2hhcGUoCiAgICAgICAgbiBpZiBkaW0gPT0gMCBlbHNlIDEsIG4gaWYgZGltID09IDEgZWxzZSAxLCBuIGlmIGRpbSA9PSAyIGVsc2UgMSkKCgpkZWYgZ2VuZXJhdGVfd29ybGQod29ybGQ6IGludCwgZ3JvdW5kOiBpbnQgfCBOb25lID0gTm9uZSkgLT4gVGVuc29yOgogICAgIiIiT25lIFtZLCBYLCBaXSBpbnQ4IHZvbHVtZSwgZXhhY3RseSBhcyB0aGUgdGhyZWUgd29ybGRnZW4ga2VybmVscyBsZWF2ZSBpdC4KCiAgICBnZW5lcmF0ZV93b3JsZF9rZXJuZWwgaXMgYSB0aHJlZS13YXkgYnJhbmNoIG9uIHRoZSB5IGNvb3JkaW5hdGU6CgogICAgICAgIHkgPT0gMCAgICAgICAgICAgICAgICAgLT4gQkVEUk9DSwogICAgICAgIHkgPCAgZ3JvdW5kX2hlaWdodCAtIDEgLT4gRElSVAogICAgICAgIHkgPT0gZ3JvdW5kX2hlaWdodCAtIDEgLT4gR1JBU1MKICAgICAgICBvdGhlcndpc2UgICAgICAgICAgICAgIC0+IEFJUgoKICAgIHdoaWNoIGFzIHRlbnNvciBvcHMgaXMgdHdvIG5lc3RlZCBgd2hlcmVgcyBvdmVyIHRoZSB5IGF4aXMuIEVhY2ggYXhpcyBpcyBhCiAgICBbTiwgMSwgMV0gLyBbMSwgTiwgMV0gLyBbMSwgMSwgTl0gcm93IHNvIHRoZSB0aHJlZSBicm9hZGNhc3QgYWdhaW5zdCBlYWNoCiAgICBvdGhlciBpbnRvIFtZLCBYLCBaXS4KICAgICIiIgogICAgZyA9IGdyb3VuZF9vZih3b3JsZCwgZ3JvdW5kKQogICAgeSwgeCwgeiA9IF9heGlzKHdvcmxkLCAwKSwgX2F4aXMod29ybGQsIDEpLCBfYXhpcyh3b3JsZCwgMikKCiAgICBibGsgPSAoeSA9PSAwKS53aGVyZShCRURST0NLLCAoeSA8IGcgLSAxKS53aGVyZShESVJULCAoeSA9PSBnIC0gMSkud2hlcmUoR1JBU1MsIEFJUikpKQogICAgdm94ID0gKGJsayArIHggKiAwICsgeiAqIDApLmNhc3QoZHR5cGVzLmludDgpICAgIyBicm9hZGNhc3QgdG8gW1ksIFgsIFpdCgogICAgdm94ID0gX3BsYWNlX3RyZWUodm94LCB3b3JsZCwgZykKICAgIHZveCA9IF9wbGFjZV9ob3VzZSh2b3gsIHdvcmxkLCBnKQogICAgcmV0dXJuIHZveC5jb250aWd1b3VzKCkKCgpkZWYgX2F0KHZveDogVGVuc29yLCB5OiBpbnQsIHg6IGludCwgejogaW50LCBibG9jazogaW50KSAtPiBUZW5zb3I6CiAgICAiIiJPbmUgdm94ZWwgd3JpdGUsIGluIGJvdW5kcyBvciBub3QuCgogICAgVGhlIHJlZmVyZW5jZSdzIFNFVF9WT1hFTCBtYWNybyBib3VuZHMtY2hlY2tzIGJlZm9yZSB3cml0aW5nCiAgICAoa2VybmVscy5jdToxODk2KSwgc28gYSBob3VzZSB0aGF0IGRvZXMgbm90IGZpdCB3cml0ZXMgbm90aGluZyBvdXRzaWRlIHRoZQogICAgd29ybGQuIFNjYXR0ZXJpbmcgYSB3aG9sZSBbWSwgWCwgWl0gbWFzayBvZiBgYmxvY2tgIGludG8gYHZveGAgYW5kCiAgICBgd2hlcmVgLWluZyBpdCBpbiByZXByb2R1Y2VzIHRoYXQ6IGFuIG91dC1vZi1yYW5nZSBjb29yZGluYXRlIHNpbXBseSBuZXZlcgogICAgbWF0Y2hlcyBhbiBlbGVtZW50IG9mIHRoZSB2b2x1bWUuCiAgICAiIiIKICAgIHd5LCB3eCwgd3ogPSB2b3guc2hhcGUKICAgIHl5LCB4eCwgenogPSBfYXhpcyh3eSwgMCksIF9heGlzKHd4LCAxKSwgX2F4aXMod3osIDIpCiAgICBtID0gKCh5eSA9PSB5KSAmICh4eCA9PSB4KSAmICh6eiA9PSB6KSkKICAgIHJldHVybiBtLndoZXJlKGJsb2NrLCB2b3gpCgoKZGVmIF9wbGFjZV90cmVlKHZveDogVGVuc29yLCB3b3JsZDogaW50LCBncm91bmQ6IGludCkgLT4gVGVuc29yOgogICAgIiIicGxhY2VfdHJlZV9rZXJuZWwgKGtlcm5lbHMuY3U6MTgzMSk6IDMgT0FLTE9HIGF0IHkgPSBncm91bmQrZHkgLi4gKzIsIG9uZQogICAgTEVBVkVTIGF0IGdyb3VuZCszLCBhdCAoc3Bhd25feCwgc3Bhd25feiArIDIpLiBUaGUgcmVmZXJlbmNlJ3MgYGlmICh5IDwgd3kpYAogICAgaXMgdGhlIGJvdW5kcyB0ZXN0LCBzbyBhdCBhIHNtYWxsIHdvcmxkIHRoZSB0b3Agb2YgdGhlIHRyZWUgY2FuIGJlIGRyb3BwZWQuCiAgICAiIiIKICAgIHR4LCB0eiA9IHdvcmxkIC8vIDIsIHdvcmxkIC8vIDIgKyAyCiAgICBmb3IgZHkgaW4gcmFuZ2UoMyk6CiAgICAgICAgdm94ID0gX2F0KHZveCwgZ3JvdW5kICsgZHksIHR4LCB0eiwgT0FLTE9HKQogICAgdm94ID0gX2F0KHZveCwgZ3JvdW5kICsgMywgdHgsIHR6LCBMRUFWRVMpCiAgICByZXR1cm4gdm94CgoKZGVmIF9wbGFjZV9ob3VzZSh2b3g6IFRlbnNvciwgd29ybGQ6IGludCwgZ3JvdW5kOiBpbnQpIC0+IFRlbnNvcjoKICAgICIiInBsYWNlX2hvdXNlX2tlcm5lbCAoa2VybmVscy5jdToxODY4KTogNSB3aWRlIHggNCBkZWVwIHggNCB0YWxsIGF0CiAgICAoc3Bhd25feCArIDYsIHNwYXduX3ogLSAyKSwgYmFzZSB5ID0gZ3JvdW5kLiBGbG9vciBjb2JibGVzdG9uZSwgcGxhbmsgd2FsbHMKICAgIHdpdGggYSBkb29yIG9wZW5pbmcsIG9hayBjb3JuZXIgcGlsbGFycywgYnJpY2sgcm9vZi4iIiIKICAgIGh4LCBoeiwgVywgRCwgSCA9IHdvcmxkIC8vIDIgKyA2LCB3b3JsZCAvLyAyIC0gMiwgNSwgNCwgNAogICAgYmFzZSA9IGdyb3VuZAogICAgZm9yIGR4IGluIHJhbmdlKFcpOgogICAgICAgIGZvciBkeiBpbiByYW5nZShEKToKICAgICAgICAgICAgdm94ID0gX2F0KHZveCwgYmFzZSArIDAsIGh4ICsgZHgsIGh6ICsgZHosIENPQkJMRVNUT05FKQogICAgZm9yIGR5IGluIHJhbmdlKDEsIDQpOgogICAgICAgIGZvciBkeCBpbiByYW5nZShXKToKICAgICAgICAgICAgaWYgZHggPT0gMiBhbmQgZHkgPD0gMjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlICAgICAgICAgICAgICAgICAgICAgICAgIyB0aGUgZG9vciBvcGVuaW5nCiAgICAgICAgICAgIHZveCA9IF9hdCh2b3gsIGJhc2UgKyBkeSwgaHggKyBkeCwgaHogKyAwLCBQTEFOS1MpCiAgICAgICAgICAgIHZveCA9IF9hdCh2b3gsIGJhc2UgKyBkeSwgaHggKyBkeCwgaHogKyBEIC0gMSwgUExBTktTKQogICAgICAgIGZvciBkeiBpbiByYW5nZSgxLCBEIC0gMSk6CiAgICAgICAgICAgIHZveCA9IF9hdCh2b3gsIGJhc2UgKyBkeSwgaHggKyAwLCBoeiArIGR6LCBQTEFOS1MpCiAgICAgICAgICAgIHZveCA9IF9hdCh2b3gsIGJhc2UgKyBkeSwgaHggKyBXIC0gMSwgaHogKyBkeiwgUExBTktTKQogICAgZm9yIGR5IGluIHJhbmdlKDEsIDQpOgogICAgICAgIGZvciBkeCwgZHogaW4gKCgwLCAwKSwgKFcgLSAxLCAwKSwgKDAsIEQgLSAxKSwgKFcgLSAxLCBEIC0gMSkpOgogICAgICAgICAgICB2b3ggPSBfYXQodm94LCBiYXNlICsgZHksIGh4ICsgZHgsIGh6ICsgZHosIE9BS0xPRykKICAgIGZvciBkeCBpbiByYW5nZShXKToKICAgICAgICBmb3IgZHogaW4gcmFuZ2UoRCk6CiAgICAgICAgICAgIHZveCA9IF9hdCh2b3gsIGJhc2UgKyBILCBoeCArIGR4LCBoeiArIGR6LCBCUklDSykKICAgIHJldHVybiB2b3gKCgpkZWYgZXhwYW5kX3dvcmxkKHZveDogVGVuc29yLCBiYXRjaDogaW50KSAtPiBUZW5zb3I6CiAgICAiIiJbWSwgWCwgWl0gLT4gW0IsIFksIFgsIFpdLCB0aGUgbGF5b3V0IGByZW5kZXJgIGdhdGhlcnMgZnJvbS4KCiAgICBgcmVwZWF0YCByYXRoZXIgdGhhbiBgZXhwYW5kYCArIGBjb250aWd1b3VzYCwgYmVjYXVzZSB0aGUgcmVuZGVyIGtlcm5lbAogICAgaW5kZXhlcyBgdm94ZWxzW2Iqd3h5eiArIC4uLl1gIGFuZCBhIG1hdGVyaWFsaXNlZCBjb3B5IGlzIHdoYXQgdGhlCiAgICByZWZlcmVuY2UgaGFzIGluIG1lbW9yeTsgYSBicm9hZGNhc3QgdmlldyB3b3VsZCBoYW5kIHRoZSBnYXRoZXIgYSBzdHJpZGUtMAogICAgYXhpcyBhbmQgcXVpZXRseSB0dXJuIGEgbWVtb3J5IHF1ZXN0aW9uIGludG8gYSBsYXlvdXQgdHJpY2suIFRoZSBjb3N0IG9mCiAgICB0aGF0IGNob2ljZSBpcyBCKndvcmxkXjMgYnl0ZXMgd3JpdHRlbiBvbmNlLCBhbmQgaXQgaXMgdGhlIHNhbWUgYnl0ZXMgdGhlCiAgICByZWZlcmVuY2Ugd3JpdGVzLgogICAgIiIiCiAgICByZXR1cm4gdm94LnJlc2hhcGUoMSwgKnZveC5zaGFwZSkucmVwZWF0KGJhdGNoLCAxLCAxLCAxKS5jb250aWd1b3VzKCkK", "6c75c0e1e9cfc105e152f06c63c7695eff08781b9b8aa3e49ba098fe299ffbc3"], ["minecu_tg/render.py", "IiIiVGhlIHJlbmRlciwgd3JpdHRlbiBhcyBwbGFpbiB0ZW5zb3IgY29kZS4KClRoaXMgaXMgdGhlIGZpbGUgdGhlIHdob2xlIHN0dWR5IHJlc3RzIG9uLCBzbyB0aGUgY29uc3RyYWludCB0aGF0IHByb2R1Y2VkIGl0CmlzIHdvcnRoIHdyaXRpbmcgZG93biBhdCB0aGUgdG9wOiAqKnRoaXMgaXMgdGhlICJwbGFpbiBDKysiIHNpZGUgb2YgdGhlCmNvbXBhcmlzb24uKiogSXQgaXMgd2hhdCBhIHBlcnNvbiB3cml0ZXMgd2hlbiB0aGV5IGFyZSBoYW5kZWQKYG5vdGVzLzE3LW1pbmVjdS1zcGVjLm1kYCBhbmQgdG9sZCB0byBpbXBsZW1lbnQKYHJlbmRlcl9rZXJuZWxfdWludDhfcHJlYmFzaXNgIHdpdGggYXJyYXkgb3BlcmF0aW9ucyBhbmQgYSBsb29wLiBJdCBpcyBub3QKd2hhdCBhIHBlcnNvbiB3cml0ZXMgYWZ0ZXIgcmVhZGluZyB0aGUgcmVmZXJlbmNlJ3MgU0FTUy4KClRoZXJlZm9yZSwgYW5kIHNwZWNpZmljYWxseSwgdGhpcyBmaWxlIGRvZXMgTk9UOgoKICAqIGNhbGwgYFRlbnNvci5jdXN0b21fa2VybmVsYCBvciBhbnkgb3RoZXIgZXNjYXBlIGhhdGNoIHRoYXQgZW1pdHMgQ1VEQQogICogc2V0IGEgdGlsZSBzaXplLCBgVENfU0VMRUNUYCwgYEFMTE9XX1RGMzJgLCBhIGJsb2NrIHNpemUsIG9yIGFueSBvdGhlcgogICAga25vYiB0aGUgY29kZSBnZW5lcmF0b3IgcmVhZHMKICAqIHVucm9sbCB0aGUgRERBIGxvb3AgYnkgaGFuZCwgZnVzZSBzdGFnZXMgYnkgaGFuZCwgb3Igc3RhZ2UgZGF0YSB0aHJvdWdoCiAgICBnbG9iYWwgbWVtb3J5IHRvIHNoYXBlIHRoZSBzY2hlZHVsZQogICogcGFzcyBhIGBDb250ZXh0KC4uLilgIHRvIHN0ZWVyIHRoZSBzY2hlZHVsZXIKICAqIHNwZWNpYWwtY2FzZSBgbWF4X3N0ZXBzYCAoZS5nLiBwZWVsIGEgMTYtd2lkZSBib2R5KSB0byBpbWl0YXRlIHRoZQogICAgcmVmZXJlbmNlJ3MgYCNwcmFnbWEgdW5yb2xsIDRgCgpUaGUgRERBIGxvb3AgaXMgYSBQeXRob24gYGZvcmAgb3ZlciBgbWF4X3N0ZXBzYCB3aXRoIG9uZSBib2R5LCBleGFjdGx5IGFzIG9uZQp3b3VsZCB3cml0ZSBhIHZlY3RvcmlzZWQgcmF5bWFyY2hlci4gV2hhdCBUaW55R3JhZCBkb2VzIHdpdGggdGhhdCB1bnJvbGxlZCBvcApjaGFpbiAtLSBmdXNlcyBpdCBpbnRvIG9uZSBrZXJuZWwsIHNwbGl0cyBpdCBpbnRvIGBtYXhfc3RlcHNgIGtlcm5lbHMsIG9yCmFueXRoaW5nIGluIGJldHdlZW4gLS0gaXMgdGhlIG1lYXN1cmVtZW50LCBub3QgYSBjaG9pY2UgbWFkZSBoZXJlLgoKVEhFIE9ORSBQTEFDRSBUSEUgU0hBUEUgSVMgQ0hPU0VOLCBBTkQgV0hZIElUIElTIE5PVCBBIFNDSEVEVUxFIENIT0lDRQotLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KVGhlIHJlZmVyZW5jZSdzIGlubmVyIGxvb3AgaXMgZGF0YS1kZXBlbmRlbnQgcGVyIHJheTogYGlmIChibG9jayA+PSAwKSB7IC4uLgpnb3RvIGRvbmU7IH1gLiBBIHRlbnNvciBwcm9ncmFtIGhhcyBubyBgZ290b2AsIGFuZCB0aGUgaG9uZXN0IHdheSB0byBleHByZXNzCiJ0aGlzIHJheSBpcyBmaW5pc2hlZCIgaW4gYXJyYXkgY29kZSBpcyB0byBrZWVwIG1hcmNoaW5nIGFuZCBzdG9wICphY2NlcHRpbmcKcmVzdWx0cyogLS0gY2FycnkgYW4gYGFjdGl2ZWAgbWFzaywgYW5kIG1ha2UgZXZlcnkgdXBkYXRlIGEgYHdoZXJlKGFjdGl2ZSAmIGMsCm5ldywgb2xkKWAuIFRoYXQgaXMgdGhlIHN0YW5kYXJkIHZlY3RvcmlzZWQtcmF5bWFyY2hlciBpZGlvbSBhbmQgaXQgaXMgd2hhdAp0aGlzIGZpbGUgZG9lcy4gVGhlIGFsdGVybmF0aXZlLCBhIGB3aGlsZWAvY2FycnktdW50aWwtY29udmVyZ2VkIGZvcm0sIGlzCmV4cHJlc3NpYmxlIGluIFRpbnlHcmFkIG9ubHkgdGhyb3VnaCBsb3ctbGV2ZWwgbG9vcCBjb25zdHJ1Y3RzIHRoYXQgYXJlCmhhbmQtc3RhZ2luZywgc28gaXQgaXMgbm90IHVzZWQuCgpUaGUgY29uc2VxdWVuY2UgaXMgaG9uZXN0IGFuZCBpcyAqdGhlIGZpbmRpbmcqLCBub3QgYSBkZWZlY3Qgb2YgdGhlIHdyaXRpbmc6CmV2ZXJ5IHJheSBydW5zIHRoZSBmdWxsIGBtYXhfc3RlcHNgIGl0ZXJhdGlvbnMgd2hldGhlciBpdCBoaXRzIG9uIHN0ZXAgMSBvcgpzdGVwIDMyLCBzbyB0aGUgd29yayBpcyBgbWF4X3N0ZXBzYCBwZXIgcGl4ZWwgcmF0aGVyIHRoYW4gdGhlIG1lYXN1cmVkIG1lYW4gb2YKfjI2LiBUaGUgaGFuZC13cml0dGVuIGtlcm5lbCdzIGAjcHJhZ21hIHVucm9sbCA0YCAoMTYgc3RhdGljIHN0ZXBzLCAzOCBTQVNTCmluc3RydWN0aW9ucyBhcGFydCkgZXhpc3RzIHRvIGtlZXAgMTYgKmxvYWQgY2hhaW5zKiBpbiBmbGlnaHQsIHdoaWNoIGlzIGEKbWVtb3J5LWxldmVsLXBhcmFsbGVsaXNtIGFyZ3VtZW50IGFib3V0IG9uZSByYXkncyBkZXBlbmRlbnQgY2hhaW47IGEgbWFza2VkCnRlbnNvciBsb29wIGhhcyBubyBkZXBlbmRlbnQgY2hhaW4gdG8gb3ZlcmxhcCBiZWNhdXNlIGFsbCByYXlzIGFyZSBhbHJlYWR5IGluCmZsaWdodCB0b2dldGhlci4gVGhlc2UgYXJlIGRpZmZlcmVudCBtYWNoaW5lcyBkb2luZyB0aGUgc2FtZSBhcml0aG1ldGljLCBhbmQKdGhlIG5vdGVzIGZpbGUgbWVhc3VyZXMgYm90aCBhbmQgZG9lcyBub3QgcHJldGVuZCB0aGV5IGFyZSB0aGUgc2FtZSB0aGluZy4KCk5PIEVBUkxZIEVYSVQgUEVSIFJBWSwgQU5EIFdIWSBUSEFUIElTIFRIRSBQT0lOVAotLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KYGZvciBpIGluIHJhbmdlKG1heF9zdGVwcylgIC0tIHRoZSBsb29wIGJvdW5kIGlzIHRoZSByZWZlcmVuY2UncyBgbWF4X3N0ZXBzYCwKaXRzIHBlci1yYXkgZWFybHkgZXhpdCBiZWNvbWVzIGFuIGBhY3RpdmVgIG1hc2ssIGFuZCB0aGUgd2hvbGUgYm9keSBpcyBvbmUKdW5yb2xsZWQgY2hhaW4uIElmIFRpbnlHcmFkIGZ1c2VzIGFjcm9zcyBpdGVyYXRpb25zIHdlIGdldCBvbmUga2VybmVsIHdpdGgKMzIgaW5saW5lZCBzdGVwczsgaWYgaXQgZG9lcyBub3QsIHdlIGdldCBvbmUga2VybmVsIHBlciBzdGVwIHdpdGggdGhlIHJheQpwb3NpdGlvbiBtYXRlcmlhbGlzZWQgaW4gYmV0d2Vlbi4gQm90aCBhcmUgb3V0Y29tZXMuIFRoZSBjb2RlIGRvZXMgbm90IGNob29zZQpiZXR3ZWVuIHRoZW0uCiIiIgoKZnJvbSB0aW55Z3JhZCBpbXBvcnQgVGVuc29yLCBkdHlwZXMKCmZyb20gLmJsb2NrcyBpbXBvcnQgY29sb3VyX2x1dCwgQUlSCgojIFRoZSByZWZlcmVuY2UncyBgMWUzMGZgIGd1YXJkIGZvciBhIHplcm8gcmF5IGNvbXBvbmVudCAoa2VybmVscy5jdTo2NDYtNjQ4KS4KX0JJRyA9IDFlMzAKCgpkZWYgY2FtZXJhX2Jhc2lzKGNhbWVyYXM6IFRlbnNvciwgZm92X2RlZ3JlZXM6IGZsb2F0LCB3aWR0aDogaW50LCBoZWlnaHQ6IGludCkgLT4gVGVuc29yOgogICAgIiIicHJlY29tcHV0ZV9jYW1lcmFfYmFzaXNfa2VybmVsIChrZXJuZWxzLmN1OjUxMSk6IFtCLCA1XSAtPiBbQiwgMTRdLgoKICAgIGNhbWVyYXMgaXMgKHgsIHksIHosIHlhdywgcGl0Y2gpLiBUaGUgb3V0cHV0IGxheW91dCBpcyB0aGUgcmVmZXJlbmNlJ3M6CgogICAgICAgIFswLi4yXSAgZXllLCB3aXRoIHkgcmFpc2VkIGJ5IHRoZSBleWUgaGVpZ2h0IDEuNgogICAgICAgIFszLi41XSAgZm9yd2FyZCA9IChzaW5feWF3KmNvc19waXRjaCwgc2luX3BpdGNoLCBjb3NfeWF3KmNvc19waXRjaCkKICAgICAgICBbNi4uN10gIHJpZ2h0LngsIHJpZ2h0Lnogb25seSAtLSByaWdodC55IGlzIGlkZW50aWNhbGx5IDAKICAgICAgICBbOC4uMTBdIHVwCiAgICAgICAgWzExXSAgICBzY2FsZV91ID0gaGFsZl9mb3YgKiBhc3BlY3QKICAgICAgICBbMTJdICAgIHNjYWxlX3YgPSBoYWxmX2ZvdgoKICAgIFRyYW5zY3JpYmVkIHN0YXRlbWVudCBmb3Igc3RhdGVtZW50LCBpbmNsdWRpbmcgdGhlIGRyb3BwZWQgcmlnaHQueSAoYSAzCiAgICB2ZWN0b3IgaW4gMiBzbG90cywgd2hpY2ggYSBjb21waWxlciB3b3VsZCBoYXZlIHRvIGJlIHRvbGQpIGFuZCB0aGUKICAgIGBmb3YgKiAwLjAxNzQ1MzI5ZmAgZGVncmVlLXRvLXJhZGlhbiBjb25zdGFudC4gYC0tdXNlX2Zhc3RfbWF0aGAgdHVybnMKICAgIHNpbmYvY29zZi90YW5mIGludG8gYXBwcm94aW1hdGUgaW50cmluc2ljcyBpbiB0aGUgcmVmZXJlbmNlOyBoZXJlIHRoZXkgYXJlCiAgICBUaW55R3JhZCdzIHNpbi9jb3MvdGFuLCB3aGljaCB0aGUgQ1VEQSBiYWNrZW5kIGxvd2VycyB0byB0aGUgc2FtZSBNVUZVCiAgICBzZXF1ZW5jZSB1cCB0byBhIGxhc3QtdWxwIGRpZmZlcmVuY2UuIFRoZSBEREEgY29tcGFyaXNvbiBpcyBhIGZsb2F0CiAgICBjb21wYXJlLCBzbyBhIGxhc3QtdWxwIGRpZmZlcmVuY2UgY2FuIG1vdmUgYSByYXkgb25lIHZveGVsIGF0IGFuIGV4YWN0IHRpZTsKICAgIHRoYXQgaXMgd2h5IGNvcnJlY3RuZXNzIGlzIHBpbm5lZCB0byBhIENQVSByZWZlcmVuY2UgYW5kIHRvIHRoZSBEREEgc3RlcAogICAgY291bnQgcmF0aGVyIHRoYW4gdG8gYSBiaXQtZXhhY3QgaW1hZ2UgYWdhaW5zdCB0aGUgR1BVLgogICAgIiIiCiAgICBmMzIgPSBkdHlwZXMuZmxvYXQzMgogICAgY2FtX3gsIGNhbV95LCBjYW1feiA9IGNhbWVyYXNbOiwgMF0sIGNhbWVyYXNbOiwgMV0sIGNhbWVyYXNbOiwgMl0KICAgIHlhdywgcGl0Y2ggPSBjYW1lcmFzWzosIDNdLCBjYW1lcmFzWzosIDRdCiAgICB6ZXJvID0geWF3ICogMCAgICAgICAgICAgICAgICAgICAgICAgIyBhIFtCXSB6ZXJvLCBmb3IgYnJvYWRjYXN0aW5nIGEgc2NhbGFyCgogICAgc2luX3lhdywgY29zX3lhdyA9IHlhdy5zaW4oKSwgeWF3LmNvcygpCiAgICBzaW5fcGl0Y2gsIGNvc19waXRjaCA9IHBpdGNoLnNpbigpLCBwaXRjaC5jb3MoKQoKICAgICMgdGFuZiBpbiBmbG9hdDMyIG9uIHRoZSBkZXZpY2UsIGFzIHRoZSByZWZlcmVuY2UgZG9lcyAtLSBub3QgbWF0aC50YW4gaW4KICAgICMgZG91YmxlLCB3aGljaCBpcyBhIGRpZmZlcmVudCBsYXN0LXVscCBiZWhhdmlvdXIgZnJvbSB0aGUgb25lIGJlaW5nCiAgICAjIGNvbXBhcmVkIGFnYWluc3QuCiAgICBoYWxmX2ZvdiA9IFRlbnNvcihbZm92X2RlZ3JlZXMgKiAwLjAxNzQ1MzI5ICogMC41XSwgZHR5cGU9ZjMyKS50YW4oKVswXSArIHplcm8KICAgIGFzcGVjdCA9IGZsb2F0KHdpZHRoKSAvIGZsb2F0KGhlaWdodCkKCiAgICBleWUgPSBjYW1feC5zdGFjayhjYW1feSArIDEuNiwgY2FtX3osIGRpbT0xKS5jYXN0KGYzMikKICAgIGZvcndhcmQgPSAoc2luX3lhdyAqIGNvc19waXRjaCkuc3RhY2soc2luX3BpdGNoLCBjb3NfeWF3ICogY29zX3BpdGNoLCBkaW09MSkKICAgIHJpZ2h0ID0gY29zX3lhdy5zdGFjaygtc2luX3lhdywgZGltPTEpICAgICAgICAgICAgICAgICAgICAjIHJpZ2h0X3kgPT0gMAogICAgdXAgPSAoLXNpbl95YXcgKiBzaW5fcGl0Y2gpLnN0YWNrKGNvc19waXRjaCwgLWNvc195YXcgKiBzaW5fcGl0Y2gsIGRpbT0xKQogICAgc2NhbGVzID0gKGhhbGZfZm92ICogYXNwZWN0KS5zdGFjayhoYWxmX2ZvdiwgZGltPTEpCiAgICByZXR1cm4gZXllLmNhdChmb3J3YXJkLCByaWdodCwgdXAsIHNjYWxlcywgZGltPTEpLmNvbnRpZ3VvdXMoKQoKCmRlZiByZW5kZXIodm94ZWxzOiBUZW5zb3IsIGJhc2lzOiBUZW5zb3IsIHdpZHRoOiBpbnQsIGhlaWdodDogaW50LAogICAgICAgICAgIG1heF9zdGVwczogaW50LCBjb3VudF9zdGVwczogYm9vbCA9IEZhbHNlKToKICAgICIiIlRoZSB1aW50OCBvYnNlcnZhdGlvbiwgW0IsIEgsIFcsIDNdLgoKICAgIHZveGVsczogW0IsIFdZLCBXWCwgV1pdIGludDguIGJhc2lzOiBbQiwgMTRdIGZsb2F0MzIsIGZyb20gYGNhbWVyYV9iYXNpc2AuCgogICAgVGhlIGJvZHkgaXMgdGhlIHJlZmVyZW5jZSdzIGByZW5kZXJfa2VybmVsX3VpbnQ4X3ByZWJhc2lzYAogICAgKGtlcm5lbHMuY3U6NTY4KSB3cml0dGVuIG92ZXIgYXJyYXlzLCBpbiB0aGUgb3JkZXIgdGhlIHJlZmVyZW5jZSBkb2VzIGl0OgogICAgYnVpbGQgdGhlIHJheSwgbm9ybWFsaXNlIGl0LCBzZXQgdXAgdGhlIGdyaWQgdHJhdmVyc2FsLCBsb29wLCB0aGVuIGxvb2sgdGhlCiAgICBibG9jayB1cCBpbiB0aGUgY29sb3VyIHRhYmxlLgoKICAgIGBjb3VudF9zdGVwc2AgcmV0dXJucyBgKG9icywgc3RlcHMpYCB3aGVyZSBgc3RlcHNgIGlzIFtCLCAxLCBILCBXXSBpbnQzMiwgdGhlCiAgICBudW1iZXIgb2YgRERBIGJvZGllcyBhIHJheSBleGVjdXRlZC4gSXQgaXMgKippbnN0cnVtZW50YXRpb24qKiwgbm90IHBhcnQgb2YKICAgIHRoZSBjb21wdXRhdGlvbjogaXQgaXMgdGhlIHNhbWUgbG9vcCB3aXRoIGEgY291bnRlciwgaXQgY2Fubm90IGNoYW5nZSBgb2JzYAogICAgKGB2ZXJpZnkucHlgIGFzc2VydHMgdGhlIHR3byBhcmUgYml0LWlkZW50aWNhbCksIGFuZCBldmVyeSB0aW1pbmcgaW4gdGhpcwogICAgc3R1ZHkgaXMgdGFrZW4gd2l0aCBpdCBvZmYuIEl0IGV4aXN0cyBiZWNhdXNlIHRoZSBzdGVwIGNvdW50IGlzIGEgZGlzY3JldGUsCiAgICB0aWUtc2Vuc2l0aXZlIHF1YW50aXR5OiB0d28gaW1wbGVtZW50YXRpb25zIHRoYXQgdGFrZSB0aGUgc2FtZSB0cmF2ZXJzYWwKICAgIGRlY2lzaW9ucyBoYXZlIGlkZW50aWNhbCBzdGVwIGNvdW50cywgYW5kIHR3byB0aGF0IGRvIG5vdCwgZG8gbm90LiBUaGUKICAgIGltYWdlIGlzIGEgbG9zc3kgcmVhZG91dCBvZiB0aGF0IC0tIG1hbnkgc3RlcCBjb3VudHMgY29sbGFwc2UgdG8gdGhlIHNhbWUKICAgIGNvbG91ciAtLSBzbyB0aGUgc3RlcCBjb3VudCBpcyB0aGUgc2hhcnBlciBvZiB0aGUgdHdvIGNoZWNrcy4KICAgICIiIgogICAgQiA9IHZveGVscy5zaGFwZVswXQogICAgd3ksIHd4LCB3eiA9IHZveGVscy5zaGFwZVsxXSwgdm94ZWxzLnNoYXBlWzJdLCB2b3hlbHMuc2hhcGVbM10KICAgIGYzMiwgaTMyID0gZHR5cGVzLmZsb2F0MzIsIGR0eXBlcy5pbnQzMgoKICAgICMgLS0tLSB0aGUgZXllIGFuZCB0aGUgYmFzaXMsIGxpZnRlZCB0byBbQiwgMSwgMSwgMV0gc28gdGhleSBicm9hZGNhc3QKICAgICMgb3ZlciB0aGUgcGl4ZWwgZ3JpZCB0aGUgd2F5IG9uZSBzY2FsYXIgZG9lcyBwZXIgdGhyZWFkIGluIHRoZSByZWZlcmVuY2UuCiAgICAjIFRoZSBzbGljZSBpcyBgaTppKzFgIGFuZCBub3QgYGlgOiBgYmFzaXNbOiwgaV1gIGRyb3BzIHRvIHJhbmsgMSwgYW5kCiAgICAjIGBbTm9uZSwgTm9uZV1gIG9uIGEgcmFuay0xIHRlbnNvciBnaXZlcyAoMSwgMSwgQikgLS0gdGhlIGJhdGNoIGxhbmRzIGluCiAgICAjIHRoZSB3cm9uZyBheGlzLCB3aGljaCBpcyBpbnZpc2libGUgYXQgQj0xIGFuZCB3cm9uZyBhdCBldmVyeSBvdGhlciBCLgogICAgZGVmIGNvbXAoaSk6CiAgICAgICAgcmV0dXJuIGJhc2lzWzosIGk6aSArIDEsIE5vbmUsIE5vbmVdCgogICAgZXllX3gsIGV5ZV95LCBleWVfeiA9IGNvbXAoMCksIGNvbXAoMSksIGNvbXAoMikKICAgIGZ3ZF94LCBmd2RfeSwgZndkX3ogPSBjb21wKDMpLCBjb21wKDQpLCBjb21wKDUpCiAgICByaWdodF94LCByaWdodF96ID0gY29tcCg2KSwgY29tcCg3KQogICAgdXBfeCwgdXBfeSwgdXBfeiA9IGNvbXAoOCksIGNvbXAoOSksIGNvbXAoMTApCiAgICBzY2FsZV91LCBzY2FsZV92ID0gY29tcCgxMSksIGNvbXAoMTIpCgogICAgIyAtLS0tIHRoZSBwaXhlbCBncmlkLiBweCBydW5zIGFsb25nIHRoZSBsYXN0IGF4aXMsIHB5IGFsb25nIHRoZSB0aGlyZCwgc28KICAgICMgdGhlIG91dHB1dCBpcyBhbHJlYWR5IFtCLCAxLCBILCBXXSBoZXJlIGFuZCBuZWVkcyBubyB0cmFuc3Bvc2UgbGF0ZXIuCiAgICBweCA9IFRlbnNvcihsaXN0KHJhbmdlKHdpZHRoKSksIGR0eXBlPWYzMikucmVzaGFwZSgxLCAxLCAxLCB3aWR0aCkKICAgIHB5ID0gVGVuc29yKGxpc3QocmFuZ2UoaGVpZ2h0KSksIGR0eXBlPWYzMikucmVzaGFwZSgxLCAxLCBoZWlnaHQsIDEpCgogICAgIyBrZXJuZWxzLmN1OjYxOS02MjAKICAgIHUgPSAoKHB4ICsgMC41KSAqICgyLjAgLyBmbG9hdCh3aWR0aCkpIC0gMS4wKSAqIHNjYWxlX3UKICAgIHYgPSAoMS4wIC0gKHB5ICsgMC41KSAqICgyLjAgLyBmbG9hdChoZWlnaHQpKSkgKiBzY2FsZV92CgogICAgIyBrZXJuZWxzLmN1OjYyMi02MjQuIHJpZ2h0X3kgaXMgMCwgc28gcmF5X3kgaGFzIG5vIHUgdGVybSAtLSB0aGUgc2FtZQogICAgIyBmYWN0IHRoZSByZWZlcmVuY2UgaGFyZC1jb2RlcyBpbiBpdHMgYmFzaXMgbGF5b3V0LgogICAgcmF5X3ggPSBmd2RfeCArIHUgKiByaWdodF94ICsgdiAqIHVwX3gKICAgIHJheV95ID0gZndkX3kgKyB2ICogdXBfeQogICAgcmF5X3ogPSBmd2RfeiArIHUgKiByaWdodF96ICsgdiAqIHVwX3oKCiAgICAjIGtlcm5lbHMuY3U6NjI2LTYyOSwgcnNxcnRmLiB0aW55Z3JhZCdzIGByc3FydGAgbG93ZXJzIHRvIHRoZSBzYW1lCiAgICAjIE1VRlUuUlNRIG9uIHNtXzc1LgogICAgcmF5X2xlbiA9IChyYXlfeCAqIHJheV94ICsgcmF5X3kgKiByYXlfeSArIHJheV96ICogcmF5X3opLnJzcXJ0KCkKICAgIHJheV94LCByYXlfeSwgcmF5X3ogPSByYXlfeCAqIHJheV9sZW4sIHJheV95ICogcmF5X2xlbiwgcmF5X3ogKiByYXlfbGVuCgogICAgIyAtLS0tIGdyaWQgdHJhdmVyc2FsIHNldHVwLCBrZXJuZWxzLmN1OjYzOC02NTkKICAgIHZveGVsX3ggPSBleWVfeC5mbG9vcigpCiAgICB2b3hlbF95ID0gZXllX3kuZmxvb3IoKQogICAgdm94ZWxfeiA9IGV5ZV96LmZsb29yKCkKCiAgICBzdGVwX3ggPSAocmF5X3ggPj0gMCkud2hlcmUoMS4wLCAtMS4wKQogICAgc3RlcF95ID0gKHJheV95ID49IDApLndoZXJlKDEuMCwgLTEuMCkKICAgIHN0ZXBfeiA9IChyYXlfeiA+PSAwKS53aGVyZSgxLjAsIC0xLjApCgogICAgIyAocmF5ICE9IDApID8gMS9yYXkgOiAxZTMwLiBUaGUgYHdoZXJlYCBldmFsdWF0ZXMgYm90aCBicmFuY2hlcyBhbmQgcGlja3MKICAgICMgb25lLCB3aGljaCBpcyB0aGUgc2FtZSB0cmFkZSB0aGUgcmVmZXJlbmNlJ3MgdGVybmFyeSBtYWtlczsgdGhlIGRpc2NhcmRlZAogICAgIyBicmFuY2ggaXMgYW4gaW5maW5pdHksIG5ldmVyIGEgTmFOLCBzbyBub3RoaW5nIGVzY2FwZXMuCiAgICBpbnZfeCA9IChyYXlfeCAhPSAwKS53aGVyZShyYXlfeC5yZWNpcHJvY2FsKCksIF9CSUcpCiAgICBpbnZfeSA9IChyYXlfeSAhPSAwKS53aGVyZShyYXlfeS5yZWNpcHJvY2FsKCksIF9CSUcpCiAgICBpbnZfeiA9IChyYXlfeiAhPSAwKS53aGVyZShyYXlfei5yZWNpcHJvY2FsKCksIF9CSUcpCgogICAgdF9kZWx0YV94LCB0X2RlbHRhX3ksIHRfZGVsdGFfeiA9IGludl94LmFicygpLCBpbnZfeS5hYnMoKSwgaW52X3ouYWJzKCkKCiAgICB0X21heF94ID0gKHJheV94ID49IDApLndoZXJlKCh2b3hlbF94ICsgMS4wIC0gZXllX3gpICogaW52X3gsICh2b3hlbF94IC0gZXllX3gpICogaW52X3gpCiAgICB0X21heF95ID0gKHJheV95ID49IDApLndoZXJlKCh2b3hlbF95ICsgMS4wIC0gZXllX3kpICogaW52X3ksICh2b3hlbF95IC0gZXllX3kpICogaW52X3kpCiAgICB0X21heF96ID0gKHJheV96ID49IDApLndoZXJlKCh2b3hlbF96ICsgMS4wIC0gZXllX3opICogaW52X3osICh2b3hlbF96IC0gZXllX3opICogaW52X3opCgogICAgIyAtLS0tIHRoZSBEREEsIGtlcm5lbHMuY3U6NjY0LTY3OQogICAgIwogICAgIyBgaGl0YCBzdGFydHMgYXQgQUlSIGFuZCBgZm91bmRgIHN0YXJ0cyBmYWxzZS4gRWFjaCBpdGVyYXRpb24gcmVhZHMgdGhlCiAgICAjIHZveGVsIHVuZGVyIHRoZSBjdXJzb3IsIHJlY29yZHMgdGhlIGhpdCBmb3IgYSByYXkgdGhhdCBpcyBzdGlsbCBsb29raW5nCiAgICAjIGFuZCBzdGlsbCBpbnNpZGUgdGhlIHdvcmxkLCB0aGVuIHN0ZXBzLiBBIHJheSB0aGF0IGhhcyBoaXQgc3RvcHMgY2hhbmdpbmcKICAgICMgYGhpdGA7IGl0IGtlZXBzIHN0ZXBwaW5nLCBoYXJtbGVzc2x5LCBiZWNhdXNlIGV2ZXJ5IHVwZGF0ZSBpcyBtYXNrZWQuCiAgICAjCiAgICAjIFRocmVlIHRoaW5ncyBhYm91dCB0aGlzIGxvb3AgYXJlIGRlbGliZXJhdGUgYW5kIG5vbmUgb2YgdGhlbSBpcyBhCiAgICAjIHNjaGVkdWxpbmcgZGVjaXNpb246CiAgICAjCiAgICAjICAqIGBmb3VuZGAgYW5kIGBpbnNpZGVgIGFyZSBzZXBhcmF0ZSBtYXNrcy4gVGhlIHJlZmVyZW5jZSdzIGBnb3RvIGRvbmVgCiAgICAjICAgIGZpcmVzIG9uIGEgaGl0OyBpdHMgYm91bmRzIHRlc3QgZ2F0ZXMgb25seSB0aGUgKmxvYWQqLiBBIHJheSB0aGF0CiAgICAjICAgIGxlYXZlcyB0aGUgd29ybGQgaGFzIG5vdCBoaXQgYW55dGhpbmcsIGFuZCBjb25mbGF0aW5nIHRoZSB0d28gd291bGQKICAgICMgICAgY29ycnVwdCB0aGUgc3RlcCBjb3VudCBmb3IgZXhhY3RseSB0aGUgcmF5cyB0aGF0IGRvbWluYXRlIHRoaXMKICAgICMgICAgd29ya2xvYWQgLS0gbW9zdCBvZiB0aGVtIHJ1biBvZmYgdGhlIGVuZCBvZiB0aGUgd29ybGQsIG5vdCBpbnRvIGEKICAgICMgICAgYmxvY2suCiAgICAjCiAgICAjICAqIGBpbnNpZGVgIGlzIHJlLWV2YWx1YXRlZCBldmVyeSBpdGVyYXRpb24sIGZyb20gdGhlIGxpdmUgY3Vyc29yLiBUaGUKICAgICMgICAgcmVmZXJlbmNlJ3MgYCh1bnNpZ25lZCl2b3hlbF94IDwgKHVuc2lnbmVkKXd4YCBpcyBpbnNpZGUgaXRzIGxvb3AgZm9yCiAgICAjICAgIHRoZSBzYW1lIHJlYXNvbjogdGhlIGN1cnNvciBtb3Zlcy4KICAgICMKICAgICMgICogdGhlIGdhdGhlciBpbmRleCBpcyBjbGFtcGVkLiBUaGF0IGlzIG1lbW9yeSBzYWZldHksIGJlY2F1c2UgYSB0ZW5zb3IKICAgICMgICAgZ2F0aGVyIGhhcyBubyBib3VuZHMgY2hlY2sgYW5kIHRoZSByZWZlcmVuY2UncyBib3VuZHMgdGVzdCBnYXRlcyBhCiAgICAjICAgIHBvaW50ZXIgaXQgc2ltcGx5IG5ldmVyIGZvcm1zLiBgaW5zaWRlYCBpcyB3aGF0IGRlY2lkZXMgd2hldGhlciB0aGUKICAgICMgICAgdmFsdWUgcmVhZCBjb3VudHMuCiAgICAjCiAgICAjICAqIHRoZSBnYXRoZXIgaXMgd3JpdHRlbiBhcyBhIDQtaW5kZXggYHZveGVsc1tiLCB5LCB4LCB6XWAsIHdoaWNoIGlzIHRoZQogICAgIyAgICBvYnZpb3VzIHdheSB0byBzYXkgaXQgYW5kIHdoaWNoIHRpbnlncmFkIGxvd2VycyB0byBmb3VyIHByZWRpY2F0ZWQKICAgICMgICAgMS1ieXRlIGxvYWRzIG9mIHRoZSAqc2FtZSogYWRkcmVzcyAtLSBvbmUgcGVyIGBib29sYCBjb25zdW1lciBvZiB0aGUKICAgICMgICAgc3RlcCwgYmVjYXVzZSB0aGUgY29tcGlsZXIgY2Fubm90IHByb3ZlIHRoZSBmb3VyIHByZWRpY2F0ZXMgYXJlCiAgICAjICAgIHJlbGF0ZWQuIFRoZSByZWZlcmVuY2UncyBgX19sZGdgIGlzIG9uZSBgTERHLkUuUzguQ09OU1RBTlQuU1lTYC4gVGhhdAogICAgIyAgICBpcyA0eCB0aGUgcmVhZCBpbnN0cnVjdGlvbnMgb24gdGhlIGhvdCBwYXRoIGFuZCBpdCBpcyBhIGNvc3Qgb2YgdGhlCiAgICAjICAgIG1hc2tlZCBmb3JtdWxhdGlvbiwgbm90IGEgY29kZWdlbiBmYWlsdXJlLiBTZWUgbm90ZXMvMTkgZ2FwIDcuCiAgICAjCiAgICAjIFRoZSBib3VuZHMgcHJlZGljYXRlIGlzIHdyaXR0ZW4gdGhlIG9idmlvdXMgc2lnbmVkIHdheS4gVGhlIHJlZmVyZW5jZQogICAgIyBmb2xkcyBlYWNoIGF4aXMgaW50byBvbmUgYCh1bnNpZ25lZCl2IDwgKHVuc2lnbmVkKXdgLCB3aGljaCBwdHhhcyB0dXJucwogICAgIyBpbnRvIGEgc2luZ2xlIGBJU0VUUC5HRS5VMzIuT1JgOyB0aGF0IGZvbGQgaXMgYSBzb3VyY2UtbGV2ZWwgZW5jb2RpbmcKICAgICMgdHJpY2ssIG5vdCBhIGRpZmZlcmVudCBwcmVkaWNhdGUsIGFuZCBpdCBpcyBsZWZ0IG91dCBoZXJlIG9uIHB1cnBvc2UgLS0KICAgICMgc2VlIG5vdGVzLzE5LCB3aGljaCBwcmljZXMgaXQgcmF0aGVyIHRoYW4gcXVpZXRseSB0YWtpbmcgaXQuCiAgICBiX2luZGV4ID0gVGVuc29yLmFyYW5nZShCLCBkdHlwZT1pMzIpLnJlc2hhcGUoQiwgMSwgMSwgMSkKCiAgICBoaXQgPSBUZW5zb3IuZnVsbCgoQiwgMSwgaGVpZ2h0LCB3aWR0aCksIEFJUiwgZHR5cGU9aTMyKQogICAgZm91bmQgPSBUZW5zb3IuZnVsbCgoQiwgMSwgMSwgMSksIEZhbHNlLCBkdHlwZT1kdHlwZXMuYm9vbCkKICAgIHN0ZXBzID0gVGVuc29yLnplcm9zKChCLCAxLCBoZWlnaHQsIHdpZHRoKSwgZHR5cGU9aTMyKQoKICAgIGZvciBzdGVwIGluIHJhbmdlKG1heF9zdGVwcyk6CiAgICAgICAgaWYgY291bnRfc3RlcHM6CiAgICAgICAgICAgICMgdGhlIGJvZHkgcnVucyBvbmNlIHBlciByYXkgcGVyIHN0ZXAgdW50aWwgaXQgaGl0cywgc28gdGhlIGNvdW50IGlzCiAgICAgICAgICAgICMgdGhlIDEtYmFzZWQgaW5kZXggb2YgdGhlIHN0ZXAgdGhhdCBoaXQsIG9yIG1heF9zdGVwcwogICAgICAgICAgICBzdGVwcyA9ICh+Zm91bmQpLndoZXJlKHN0ZXAgKyAxLCBzdGVwcykKCiAgICAgICAgaW5zaWRlID0gKCh2b3hlbF94ID49IDApICYgKHZveGVsX3ggPCBmbG9hdCh3eCkpICYKICAgICAgICAgICAgICAgICAgKHZveGVsX3kgPj0gMCkgJiAodm94ZWxfeSA8IGZsb2F0KHd5KSkgJgogICAgICAgICAgICAgICAgICAodm94ZWxfeiA+PSAwKSAmICh2b3hlbF96IDwgZmxvYXQod3opKSkKICAgICAgICBpeCA9IHZveGVsX3guY2FzdChpMzIpLmNsaXAoMCwgd3ggLSAxKQogICAgICAgIGl5ID0gdm94ZWxfeS5jYXN0KGkzMikuY2xpcCgwLCB3eSAtIDEpCiAgICAgICAgaXogPSB2b3hlbF96LmNhc3QoaTMyKS5jbGlwKDAsIHd6IC0gMSkKICAgICAgICBibG9jayA9IHZveGVsc1tiX2luZGV4LCBpeSwgaXgsIGl6XS5jYXN0KGkzMikKCiAgICAgICAgc29saWQgPSAoYmxvY2sgPj0gMCkgJiBpbnNpZGUgJiB+Zm91bmQKICAgICAgICBoaXQgPSBzb2xpZC53aGVyZShibG9jaywgaGl0KQogICAgICAgIGZvdW5kID0gZm91bmQgfCBzb2xpZAoKICAgICAgICAjIHRoZSBzdGVwIGl0c2VsZiwga2VybmVscy5jdTo2NzItNjc4LCBhcyB0aHJlZSBzZWxlY3RzCiAgICAgICAgc3RlcF90YWtlbl94ID0gKHRfbWF4X3ggPCB0X21heF95KSAmICh0X21heF94IDwgdF9tYXhfeikKICAgICAgICBzdGVwX3Rha2VuX3kgPSAodF9tYXhfeSA8IHRfbWF4X3opICYgfnN0ZXBfdGFrZW5feAogICAgICAgIHRfbWF4X3ggPSBzdGVwX3Rha2VuX3gud2hlcmUodF9tYXhfeCArIHRfZGVsdGFfeCwgdF9tYXhfeCkKICAgICAgICB0X21heF95ID0gc3RlcF90YWtlbl95LndoZXJlKHRfbWF4X3kgKyB0X2RlbHRhX3ksIHRfbWF4X3kpCiAgICAgICAgdF9tYXhfeiA9ICh+c3RlcF90YWtlbl94ICYgfnN0ZXBfdGFrZW5feSkud2hlcmUodF9tYXhfeiArIHRfZGVsdGFfeiwgdF9tYXhfeikKICAgICAgICB2b3hlbF94ID0gc3RlcF90YWtlbl94LndoZXJlKHZveGVsX3ggKyBzdGVwX3gsIHZveGVsX3gpCiAgICAgICAgdm94ZWxfeSA9IHN0ZXBfdGFrZW5feS53aGVyZSh2b3hlbF95ICsgc3RlcF95LCB2b3hlbF95KQogICAgICAgIHZveGVsX3ogPSAofnN0ZXBfdGFrZW5feCAmIH5zdGVwX3Rha2VuX3kpLndoZXJlKHZveGVsX3ogKyBzdGVwX3osIHZveGVsX3opCgogICAgIyAtLS0tIHRoZSBzdG9yZSwga2VybmVscy5jdTo2OTAtNjk5LCBhcyBhIHRhYmxlIGxvb2t1cAogICAgIwogICAgIyBgY29sb3VyX2x1dGAgaGFzIDE0IHJvd3M6IDAuLjEyIGFyZSB0aGUgYmxvY2tzLCByb3cgMTMgaXMgdGhlIHNreS4gQSByYXkKICAgICMgdGhhdCBoaXQgaG9sZHMgYSBibG9jayBpZCBpbiBbMCwgMTJdOyBhIHJheSB0aGF0IGRpZCBub3Qgc3RpbGwgaG9sZHMgQUlSCiAgICAjIGFuZCBpcyBtYXBwZWQgdG8gcm93IDEzLiBUaGUgcmVmZXJlbmNlIHJlYWNoZXMgdGhlIHNreSBieSBhICpicmFuY2gqCiAgICAjIChgaWYgKGhpdF9ibG9jayA+PSAwKSAuLi4gZWxzZSAuLi5gKTsgaGVyZSBib3RoIGFyZSB0aGUgc2FtZSBnYXRoZXIuCiAgICAjIFRoYXQgaXMgYSBjaGFuZ2Ugb2YgZXhwcmVzc2lvbiwgbm90IG9mIHNjaGVkdWxlIC0tIG5vIGtub2IsIG5vIGZ1c2lvbgogICAgIyBoaW50LCBubyB0aWxlIHNpemUuCiAgICBsdXQgPSBjb2xvdXJfbHV0KCkgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIFsxNCwgM10gdWludDgKICAgIGNvbG91cl9yb3cgPSAoaGl0ID49IDApLndoZXJlKGhpdCwgMTMpICAgICAgICAgICAgICAgICAgICMgW0IsIDEsIEgsIFddIGludDMyCiAgICBvYnMgPSBsdXRbY29sb3VyX3Jvd10ucmVzaGFwZShCLCBoZWlnaHQsIHdpZHRoLCAzKSAgICAgICAjIFtCLCBILCBXLCAzXSB1aW50OAogICAgcmV0dXJuIChvYnMsIHN0ZXBzKSBpZiBjb3VudF9zdGVwcyBlbHNlIG9icy5jb250aWd1b3VzKCkK", "fe85581f80ed7fdba57398a6c04fe744a7fd59917d99b132ea0eb2dcfa457952"], ["minecu_tg/cpu_reference.py", "IiIiQSBmbG9hdDY0IENQVSByZWZlcmVuY2Ugb2YgdGhlIHNhbWUgYWxnb3JpdGhtLgoKV0hZIEEgU0VQQVJBVEUgUkVGRVJFTkNFLCBJTlNURUFEIE9GIENPTVBBUklORyBUTyBUSEUgR1BVLiBUaGUgdHdvIHJlbmRlcmVycwphcmUgbm90IGJpdC1jb21wYXJhYmxlIGFuZCBpdCBpcyBub3Qgd29ydGggcHJldGVuZGluZyBvdGhlcndpc2U6CgogICogdGhlIHJlZmVyZW5jZSBydW5zIGAtLXVzZV9mYXN0X21hdGhgLCBzbyBgc2luZmAvYGNvc2ZgL2B0YW5mYC9gcnNxcnRmYCBhbmQKICAgIHRoZSBkaXZpc2lvbnMgYXJlIGFwcHJveGltYXRlLCBhbmQgYF9fc2luY29zZmAgaXMgdGhlIGZhc3QgaW50cmluc2ljOwogICogVGlueUdyYWQncyBDVURBIGJhY2tlbmQgbG93ZXJzIGBzaW5gL2Bjb3NgL2B0YW5gL2Byc3FydGAgdG8gaXRzIG93bgogICAgc2VxdWVuY2UsIGFuZCBpdHMgYC9gIGlzIGEgcGxhaW4gZGl2aWRlOwogICogYSBEREEgc3RlcCBpcyBjaG9zZW4gYnkgYHRfbWF4X3ggPCB0X21heF95YCwgc28gYSBvbmUtdWxwIGRpZmZlcmVuY2UgaW4gYQogICAgYHRfbWF4YCBjYW4gbW92ZSBhIHJheSB0byBhIGRpZmZlcmVudCB2b3hlbCwgYW5kIHRoZW4gdGhlIGltYWdlIGRpZmZlcnMgYnkKICAgIGEgd2hvbGUgYmxvY2sgY29sb3VyLgoKU28gdGhlIGNvbnRyYWN0IHRoaXMgbW9kdWxlIGVuZm9yY2VzIGlzOiAqKnRoZSB0ZW5zb3IgcmVuZGVyIGFncmVlcyB3aXRoIGFuCmluZGVwZW5kZW50IGZsb2F0NjQgdHJhbnNjcmlwdGlvbiBvZiB0aGUgc2FtZSBhbGdvcml0aG0uKiogVGhlIGZsb2F0NjQKdHJhbnNjcmlwdGlvbiBpcyB3cml0dGVuIGZyb20gYG5vdGVzLzE3LW1pbmVjdS1zcGVjLm1kYCBhbmQgZnJvbSB0aGUgcmVmZXJlbmNlJ3MKc291cmNlLCBhbmQgaXQgaXMgY2hlY2tlZCBhZ2FpbnN0IHRoZSByZWZlcmVuY2UncyBvd24gQ1BVIG1vZGVsCihgbWluZWN1X3JlZi9iZW5jaC9kZGFfbW9kZWwucHlgLCBhbiBpbmRlcGVuZGVudCB0aGlyZCB0cmFuc2NyaXB0aW9uKSBzbyB0aGF0IGEKYnVnIGluIHRoaXMgZmlsZSBjYW5ub3Qgc2lsZW50bHkgYmVjb21lIHRoZSBkZWZpbml0aW9uIG9mIGNvcnJlY3QuCgpUaGUgdHdvIG91dHB1dC1sZXZlbCBjbGFpbXMgYHZlcmlmeS5weWAgbWFrZXMgYXJlIHRoZXJlZm9yZToKCiAgMS4gdGhlIFRpbnlHcmFkIGltYWdlIGVxdWFscyB0aGlzIGZsb2F0NjQgaW1hZ2UsIGV4YWN0bHksIG9uIHVpbnQ4IGNvbG91cnM7CiAgMi4gdGhlIG1lYW4gRERBIHN0ZXBzIHBlciBwaXhlbCBhZ3JlZSB0byAzIGRlY2ltYWwgcGxhY2VzIHdpdGggdGhlCiAgICAgcmVmZXJlbmNlJ3Mgb3duIG1lYXN1cmVkIDI2LjQxNiBhdCB0aGUgb24tc3VyZmFjZSBjb25maWd1cmF0aW9uLgoKQ2xhaW0gMiBpcyB0aGUgb25lIHRoYXQgYWN0dWFsbHkgcGlucyB0aGUgKndvcmtsb2FkKiwgYmVjYXVzZSBpdCBpcyBhCmRpc2NyZXRlLCB0aWUtc2Vuc2l0aXZlIHF1YW50aXR5OiBpZiB0aGUgdHdvIGltcGxlbWVudGF0aW9ucyBtYWRlIGRpZmZlcmVudAp0cmF2ZXJzYWwgZGVjaXNpb25zLCB0aGUgc3RlcCBjb3VudCB3b3VsZCBtb3ZlLgoiIiIKCmltcG9ydCBtYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmJsb2NrcyBpbXBvcnQgQkxPQ0tfQ09MT1JTX1U4LCBTS1lfQ09MT1JfVTgKCkFJUiA9IC0xCiMgVGhlIDE0LXJvdyB0YWJsZTogMC4uMTIgYXJlIHRoZSBibG9ja3MsIHJvdyAxMyBpcyB0aGUgc2t5LiBTYW1lIHRhYmxlIHRoZQojIHRlbnNvciB2ZXJzaW9uIGdhdGhlcnMgZnJvbSwgc28gdGhlIGNvbXBhcmlzb24gaXNvbGF0ZXMgdGhlIEREQSBhbmQgbm90IHRoZQojIHBhbGV0dGUuCkxVVCA9IG5wLmFycmF5KEJMT0NLX0NPTE9SU19VOCArIFtTS1lfQ09MT1JfVThdLCBkdHlwZT1ucC51aW50OCkKCgpkZWYgY2FtZXJhX2Jhc2lzKGNhbWVyYXM6IG5wLm5kYXJyYXksIGZvdl9kZWdyZWVzOiBmbG9hdCwgd2lkdGg6IGludCwgaGVpZ2h0OiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJgcmVuZGVyLmNhbWVyYV9iYXNpc2AsIGluIGZsb2F0NjQsIGZvciBvbmUgY2FtZXJhLgoKICAgIFJldHVybnMgdGhlIHNhbWUgMTQgbnVtYmVycyB0aGUga2VybmVsIHN0b3JlcywgaW4gdGhlIHNhbWUgb3JkZXIsIHNvIHRoZQogICAgY29tcGFyaXNvbiBpcyBub3Qgc2Vuc2l0aXZlIHRvIHRoZSBwYWNraW5nLgogICAgIiIiCiAgICBjYW1feCwgY2FtX3ksIGNhbV96LCB5YXcsIHBpdGNoID0gKGZsb2F0KG5wLnJlc2hhcGUoY2FtZXJhcywgLTEpW2ldKSBmb3IgaSBpbiByYW5nZSg1KSkKICAgIHNpbl95YXcsIGNvc195YXcgPSBtYXRoLnNpbih5YXcpLCBtYXRoLmNvcyh5YXcpCiAgICBzaW5fcGl0Y2gsIGNvc19waXRjaCA9IG1hdGguc2luKHBpdGNoKSwgbWF0aC5jb3MocGl0Y2gpCiAgICBoYWxmX2ZvdiA9IG1hdGgudGFuKGZvdl9kZWdyZWVzICogMC4wMTc0NTMyOSAqIDAuNSkKICAgIGFzcGVjdCA9IHdpZHRoIC8gaGVpZ2h0CiAgICByZXR1cm4gbnAuYXJyYXkoWwogICAgICAgIGNhbV94LCBjYW1feSArIDEuNiwgY2FtX3osCiAgICAgICAgc2luX3lhdyAqIGNvc19waXRjaCwgc2luX3BpdGNoLCBjb3NfeWF3ICogY29zX3BpdGNoLAogICAgICAgIGNvc195YXcsIC1zaW5feWF3LAogICAgICAgIC1zaW5feWF3ICogc2luX3BpdGNoLCBjb3NfcGl0Y2gsIC1jb3NfeWF3ICogc2luX3BpdGNoLAogICAgICAgIGhhbGZfZm92ICogYXNwZWN0LCBoYWxmX2ZvdiwKICAgIF0sIGR0eXBlPW5wLmZsb2F0NjQpCgoKZGVmIHJlbmRlcl9vbmUodm94ZWxzOiBucC5uZGFycmF5LCBiYXNpczogbnAubmRhcnJheSwgd2lkdGg6IGludCwgaGVpZ2h0OiBpbnQsCiAgICAgICAgICAgICAgIG1heF9zdGVwczogaW50KToKICAgICIiIk9uZSBlbnYuIHZveGVscyBpcyBbV1ksIFdYLCBXWl0gaW50ODsgYmFzaXMgaXMgdGhlIDE0LXZlY3Rvci4KCiAgICBSZXR1cm5zIChpbWFnZSBbSCwgVywgM10gdWludDgsIHN0ZXBzIFtILCBXXSBpbnQzMiksIHRoZSBzdGVwIGNvdW50IGJlaW5nCiAgICB0aGUgbnVtYmVyIG9mIHRpbWVzIHRoZSBEREEgYm9keSByYW4sIHdoaWNoIGlzIHRoZSBxdWFudGl0eSB0aGUgcmVmZXJlbmNlJ3MKICAgIGRldmljZS1zaWRlIHByb2JlIHJlcG9ydHMuCiAgICAiIiIKICAgIHd5LCB3eCwgd3ogPSB2b3hlbHMuc2hhcGUKICAgIG91dCA9IG5wLmVtcHR5KChoZWlnaHQsIHdpZHRoLCAzKSwgZHR5cGU9bnAudWludDgpCiAgICBzdGVwcyA9IG5wLnplcm9zKChoZWlnaHQsIHdpZHRoKSwgZHR5cGU9bnAuaW50MzIpCgogICAgZXllX3gsIGV5ZV95LCBleWVfeiA9IGJhc2lzWzBdLCBiYXNpc1sxXSwgYmFzaXNbMl0KICAgIGZ3ZF94LCBmd2RfeSwgZndkX3ogPSBiYXNpc1szXSwgYmFzaXNbNF0sIGJhc2lzWzVdCiAgICByaWdodF94LCByaWdodF96ID0gYmFzaXNbNl0sIGJhc2lzWzddCiAgICB1cF94LCB1cF95LCB1cF96ID0gYmFzaXNbOF0sIGJhc2lzWzldLCBiYXNpc1sxMF0KICAgIHNjYWxlX3UsIHNjYWxlX3YgPSBiYXNpc1sxMV0sIGJhc2lzWzEyXQoKICAgIGZvciBweSBpbiByYW5nZShoZWlnaHQpOgogICAgICAgIGZvciBweCBpbiByYW5nZSh3aWR0aCk6CiAgICAgICAgICAgICMga2VybmVscy5jdTo2MTktNjI5CiAgICAgICAgICAgIHUgPSAoMi4wICogKHB4ICsgMC41KSAvIHdpZHRoIC0gMS4wKSAqIHNjYWxlX3UKICAgICAgICAgICAgdiA9ICgxLjAgLSAyLjAgKiAocHkgKyAwLjUpIC8gaGVpZ2h0KSAqIHNjYWxlX3YKICAgICAgICAgICAgcmF5X3ggPSBmd2RfeCArIHUgKiByaWdodF94ICsgdiAqIHVwX3gKICAgICAgICAgICAgcmF5X3kgPSBmd2RfeSArIHYgKiB1cF95CiAgICAgICAgICAgIHJheV96ID0gZndkX3ogKyB1ICogcmlnaHRfeiArIHYgKiB1cF96CiAgICAgICAgICAgIHJsID0gMS4wIC8gbWF0aC5zcXJ0KHJheV94ICogcmF5X3ggKyByYXlfeSAqIHJheV95ICsgcmF5X3ogKiByYXlfeikKICAgICAgICAgICAgcmF5X3gsIHJheV95LCByYXlfeiA9IHJheV94ICogcmwsIHJheV95ICogcmwsIHJheV96ICogcmwKCiAgICAgICAgICAgICMga2VybmVscy5jdTo2MzgtNjU5CiAgICAgICAgICAgIHZ4LCB2eSwgdnogPSBpbnQobWF0aC5mbG9vcihleWVfeCkpLCBpbnQobWF0aC5mbG9vcihleWVfeSkpLCBpbnQobWF0aC5mbG9vcihleWVfeikpCiAgICAgICAgICAgIHN4ID0gMSBpZiByYXlfeCA+PSAwIGVsc2UgLTEKICAgICAgICAgICAgc3kgPSAxIGlmIHJheV95ID49IDAgZWxzZSAtMQogICAgICAgICAgICBzeiA9IDEgaWYgcmF5X3ogPj0gMCBlbHNlIC0xCiAgICAgICAgICAgIGl4ID0gMS4wIC8gcmF5X3ggaWYgcmF5X3ggIT0gMCBlbHNlIDFlMzAKICAgICAgICAgICAgaXkgPSAxLjAgLyByYXlfeSBpZiByYXlfeSAhPSAwIGVsc2UgMWUzMAogICAgICAgICAgICBpeiA9IDEuMCAvIHJheV96IGlmIHJheV96ICE9IDAgZWxzZSAxZTMwCiAgICAgICAgICAgIGR0eCwgZHR5LCBkdHogPSBhYnMoaXgpLCBhYnMoaXkpLCBhYnMoaXopCiAgICAgICAgICAgIHRteCA9ICh2eCArIDEgLSBleWVfeCkgKiBpeCBpZiByYXlfeCA+PSAwIGVsc2UgKHZ4IC0gZXllX3gpICogaXgKICAgICAgICAgICAgdG15ID0gKHZ5ICsgMSAtIGV5ZV95KSAqIGl5IGlmIHJheV95ID49IDAgZWxzZSAodnkgLSBleWVfeSkgKiBpeQogICAgICAgICAgICB0bXogPSAodnogKyAxIC0gZXllX3opICogaXogaWYgcmF5X3ogPj0gMCBlbHNlICh2eiAtIGV5ZV96KSAqIGl6CgogICAgICAgICAgICAjIGtlcm5lbHMuY3U6NjY0LTY4Ny4gYG5gIGNvdW50cyBib2R5IGV4ZWN1dGlvbnMsIHdoaWNoIGlzIHdoYXQgdGhlCiAgICAgICAgICAgICMgcmVmZXJlbmNlJ3MgcHJvYmUgcmVwb3J0cyBhcyAiRERBIHN0ZXBzL3BpeGVsIi4KICAgICAgICAgICAgaGl0ID0gQUlSCiAgICAgICAgICAgIG4gPSAwCiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKG1heF9zdGVwcyk6CiAgICAgICAgICAgICAgICBuICs9IDEKICAgICAgICAgICAgICAgIGlmIDAgPD0gdnggPCB3eCBhbmQgMCA8PSB2eSA8IHd5IGFuZCAwIDw9IHZ6IDwgd3o6CiAgICAgICAgICAgICAgICAgICAgYmxrID0gaW50KHZveGVsc1t2eSwgdngsIHZ6XSkKICAgICAgICAgICAgICAgICAgICBpZiBibGsgPj0gMDoKICAgICAgICAgICAgICAgICAgICAgICAgaGl0ID0gYmxrCiAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0bXggPCB0bXkgYW5kIHRteCA8IHRtejoKICAgICAgICAgICAgICAgICAgICB0bXggKz0gZHR4OyB2eCArPSBzeAogICAgICAgICAgICAgICAgZWxpZiB0bXkgPCB0bXo6CiAgICAgICAgICAgICAgICAgICAgdG15ICs9IGR0eTsgdnkgKz0gc3kKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgdG16ICs9IGR0ejsgdnogKz0gc3oKICAgICAgICAgICAgc3RlcHNbcHksIHB4XSA9IG4KICAgICAgICAgICAgb3V0W3B5LCBweF0gPSBMVVRbMTMgaWYgaGl0IDwgMCBlbHNlIGhpdF0KICAgIHJldHVybiBvdXQsIHN0ZXBzCg==", "d3d6356552db2be483430751924970d288cb1c128a56709aa074d7e18a908108"], ["minecu_tg/verify.py", "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJ2ZXJpZnkucHkgLS0gdGhlIGNvcnJlY3RuZXNzIGdhdGUgZm9yIHRoZSB0ZW5zb3IgcmVuZGVyLgoKUnVucyBvbiB0aGUgQ1BVLCBvbiB0aGlzIGhvc3QsIHdpdGggbm8gR1BVLiBFdmVyeXRoaW5nIGl0IGNsYWltcyBhYm91dCB0aGUKVGlueUdyYWQgaW1wbGVtZW50YXRpb24gaXMgZXN0YWJsaXNoZWQgaGVyZTsgdGhlIFQ0IHJ1biBlc3RhYmxpc2hlcyBvbmx5IHdoYXQKbmVlZHMgYSBjYXJkICh0aW1pbmdzLCB0aGUgcmVhbCBiYWNrZW5kJ3MgY29kZWdlbiwgdGhlIGRldmljZSdzIG93biB2aWV3IG9mIHRoZQpvYnNlcnZhdGlvbikuCgpXSEFUIElTIENIRUNLRUQsIEFORCBXSFkgRUFDSCBDSEVDSyBFWElTVFMKLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCjEuICoqVGhlIHdvcmxkIG1hdGNoZXMgdGhlIHJlZmVyZW5jZSdzIG93biBtb2RlbC4qKgogICBgbWluZWN1X3RnLndvcmxkLmdlbmVyYXRlX3dvcmxkYCBhZ2FpbnN0IGBtaW5lY3VfcmVmL2JlbmNoL2RkYV9tb2RlbC5weWAncwogICBgYnVpbGRfd29ybGRgLCBhdCB0aHJlZSB3b3JsZCBzaXplcywgZXhhY3QuIFRoaXMgaXMgYSBjaGVjayBhZ2FpbnN0IGEKICAgKnRoaXJkKiB0cmFuc2NyaXB0aW9uIHRoYXQgaXMgbm90IG91cnMsIHdoaWNoIGlzIHRoZSBvbmx5IGtpbmQgdGhhdCBjYXRjaGVzCiAgIGEgc2hhcmVkIG1pc3JlYWRpbmcgb2YgdGhlIHNwZWMuCgoyLiAqKlRoZSByZW5kZXIgbWF0Y2hlcyBhbiBpbmRlcGVuZGVudCBmbG9hdDY0IHRyYW5zY3JpcHRpb24sIGV4YWN0bHksIG9uIHRoZQogICB1aW50OCBvYnNlcnZhdGlvbi4qKiBgbWluZWN1X3RnL2NwdV9yZWZlcmVuY2UucHlgIGlzIHdyaXR0ZW4gZnJvbSB0aGUKICAgc3BlYyBpbiBmbG9hdDY0OyB0aGUgcmVuZGVyIGlzIGZsb2F0MzIuIEV4YWN0IGVxdWFsaXR5IG9mIHRoZSBmaW5hbCBpbWFnZSBpcwogICBhIHN0cm9uZyBzdGF0ZW1lbnQgYW5kIGl0IGhvbGRzLCBzbyBpdCBpcyB0aGUgZ2F0ZS4KCjMuICoqVGhlIHBlci1waXhlbCBEREEgc3RlcCBjb3VudHMgYWdyZWUuKiogVGhlIGltYWdlIGlzIGEgbG9zc3kgcmVhZG91dCAtLSBtYW55CiAgIHN0ZXAgY291bnRzIGNvbGxhcHNlIHRvIHRoZSBzYW1lIGJsb2NrIGNvbG91ciAtLSBzbyB0aGUgc3RlcCBjb3VudCBpcyB0aGUKICAgc2hhcnBlciBpZGVudGl0eSBjaGVjay4gSXQgaXMgYSBkaXNjcmV0ZSwgdGllLXNlbnNpdGl2ZSBxdWFudGl0eTogdHdvCiAgIGltcGxlbWVudGF0aW9ucyB0aGF0IHRvb2sgdGhlIHNhbWUgdHJhdmVyc2FsIGRlY2lzaW9ucyBoYXZlIGlkZW50aWNhbCBzdGVwCiAgIGNvdW50cywgYW5kIHR3byB0aGF0IGRpZCBub3QsIGRvIG5vdC4gQWdyZWVtZW50IGlzIHJlcXVpcmVkIGF0CiAgIGA+PSA5OSVgIG9mIHBpeGVscywgYW5kIHRoZSByZXNpZHVhbCBpcyBleHBsYWluZWQgYnkgZmxvYXQzMi12cy1mbG9hdDY0CiAgIHJvdW5kaW5nIG9mIGB0X21heGAgYXQgZXhhY3QgdGllcy4KCjQuICoqVGhlIHN0ZXAgY291bnRlciBpcyBpbmVydC4qKiBSZW5kZXJpbmcgd2l0aCBgY291bnRfc3RlcHM9VHJ1ZWAgYW5kIHdpdGgKICAgaXQgb2ZmIG11c3QgcHJvZHVjZSBiaXQtaWRlbnRpY2FsIG9ic2VydmF0aW9ucy4gV2l0aG91dCB0aGlzIGNoZWNrLCBjaGVjayAzCiAgIGNvdWxkIGJlIG1lYXN1cmluZyBhIGRpZmZlcmVudCBwcm9ncmFtIGZyb20gdGhlIG9uZSB0aGF0IGdldHMgdGltZWQuCgo1LiAqKlRoZSByZXN1bHQgaXMgYmF0Y2gtaW52YXJpYW50LioqIFRoZSByZWZlcmVuY2UgZ2l2ZXMgZXZlcnkgZW52IHRoZSBzYW1lCiAgIHdvcmxkLCBzbyBldmVyeSBlbnYncyBpbWFnZSBtdXN0IGJlIGlkZW50aWNhbCwgYW5kIG11c3QgbWF0Y2ggdGhlIHNpbmdsZS1lbnYKICAgaW1hZ2UuIFRoaXMgaXMgdGhlIGNoZWNrIHRoYXQgY2F0Y2hlcyB0aGUgYGJhc2lzWzosIGksIE5vbmUsIE5vbmVdYCBzaGFwZQogICBidWcsIHdoaWNoIHB1dCB0aGUgYmF0Y2ggYXhpcyBpbiB0aGUgd3JvbmcgcGxhY2UgYW5kIHdhcyAqaW52aXNpYmxlIGF0CiAgIEI9MSo6IGBiYXNpc1s6LCBpXWAgZHJvcHMgdG8gcmFuayAxLCBhbmQgYFtOb25lLCBOb25lXWAgb24gYSByYW5rLTEgdGVuc29yCiAgIGlzICgxLCAxLCBCKSwgbm90IChCLCAxLCAxLCAxKS4gSXQgaXMgaW4gdGhpcyBmaWxlIGJlY2F1c2UgYSBzbW9rZSB0ZXN0IGF0CiAgIEI9MSBkaWQgbm90IGNhdGNoIGl0IGFuZCBhIGJhdGNoIG9mIDUgZGlkLgoKNi4gKipUaGUgd29ybGRnZW4gZmxvb3IgYW5kIHRoZSBwaHlzaWNzIGZsb29yIGFncmVlLioqIGBncm91bmQgPT0gd29ybGQgLy8gNGAuCiAgIEEgcnVuIHRoYXQgYnVyaWVzIHRoZSBleWUgaW5zaWRlIGl0cyBvd24gc3VyZmFjZSBpcyBhIHJ1biB3aG9zZSB0aW1pbmcKICAgbWVhc3VyZXMgYW4gb2JzIHN0b3JlOyB0aGUgcmVmZXJlbmNlJ3Mgb3duIGhhcm5lc3MgZ2F0ZXMgb24gdGhpcyBhbmQgc28gZG9lcwogICB0aGlzIG9uZS4KCkVYSVQgQ09ERS4gMCBpZiBldmVyeSBjaGVjayBwYXNzZXMsIDEgb3RoZXJ3aXNlLCB3aXRoIGV2ZXJ5IGZhaWx1cmUgcHJpbnRlZC4KQSBnYXRlIHRoYXQgb25seSBwcmludHMgaXMgYSBjbGFpbSwgbm90IGEgY2hlY2suCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IG9zCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpvcy5lbnZpcm9uLnNldGRlZmF1bHQoIkRFViIsICJDUFUiKQpvcy5lbnZpcm9uLnNldGRlZmF1bHQoIlBBUkFMTEVMIiwgIjAiKQoKSEVSRSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKUkVQTyA9IEhFUkUucGFyZW50CnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUkVQTykpCgppbXBvcnQgbnVtcHkgYXMgbnAgICMgbm9xYTogRTQwMgoKZnJvbSBtaW5lY3VfdGcgaW1wb3J0IGdlbmVyYXRlX3dvcmxkLCBncm91bmRfb2YgICMgbm9xYTogRTQwMgpmcm9tIG1pbmVjdV90Zy5jcHVfcmVmZXJlbmNlIGltcG9ydCBjYW1lcmFfYmFzaXMgYXMgYmFzaXM2NCwgcmVuZGVyX29uZSAgIyBub3FhOiBFNDAyCmZyb20gbWluZWN1X3RnLnJlbmRlciBpbXBvcnQgY2FtZXJhX2Jhc2lzLCByZW5kZXIgICMgbm9xYTogRTQwMgoKRkFJTFM6IGxpc3Rbc3RyXSA9IFtdCkxJTkVTOiBsaXN0W3N0cl0gPSBbXQoKCmRlZiBjaGVjayhvazogYm9vbCwgd2hhdDogc3RyLCBkZXRhaWw6IHN0ciA9ICIiKSAtPiBib29sOgogICAgdGFnID0gIlBBU1MiIGlmIG9rIGVsc2UgIkZBSUwiCiAgICBwcmludChmIiAgW3t0YWd9XSB7d2hhdH0iICsgKGYiICB7ZGV0YWlsfSIgaWYgZGV0YWlsIGVsc2UgIiIpKQogICAgTElORVMuYXBwZW5kKGYiW3t0YWd9XSB7d2hhdH0iICsgKGYiICB7ZGV0YWlsfSIgaWYgZGV0YWlsIGVsc2UgIiIpKQogICAgaWYgbm90IG9rOgogICAgICAgIEZBSUxTLmFwcGVuZCh3aGF0KQogICAgcmV0dXJuIG9rCgoKZGVmIGRkYV9tb2RlbCgpOgogICAgIiIiVGhlIHJlZmVyZW5jZSdzIG93biBDUFUgbW9kZWwsIHJlYWQtb25seSwgZm9yIHRoZSBjcm9zcy1jaGVjay4iIiIKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUkVQTyAvICJtaW5lY3VfcmVmIiAvICJiZW5jaCIpKQogICAgaW1wb3J0IGRkYV9tb2RlbCAgIyBub3FhOiBQTEMwNDE1CiAgICByZXR1cm4gZGRhX21vZGVsCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpkZWYgY2hlY2tfd29ybGQoTSkgLT4gTm9uZToKICAgIHByaW50KCJcbjEuIHRoZSB3b3JsZCwgYWdhaW5zdCBtaW5lY3VfcmVmL2JlbmNoL2RkYV9tb2RlbC5weSIpCiAgICBmb3Igd29ybGQgaW4gKDE2LCAzMiwgNDgpOgogICAgICAgIGcgPSBncm91bmRfb2Yod29ybGQpCiAgICAgICAgbWluZSA9IGdlbmVyYXRlX3dvcmxkKHdvcmxkLCBnKS5udW1weSgpCiAgICAgICAgcmVmID0gTS5idWlsZF93b3JsZCh3b3JsZCwgZykKICAgICAgICBzYW1lID0gbnAuYXJyYXlfZXF1YWwobWluZS5hc3R5cGUobnAuaW50OCksIHJlZi5hc3R5cGUobnAuaW50OCkpCiAgICAgICAgY2hlY2soc2FtZSwgZiJ3b3JsZCB7d29ybGR9XjMsIGdyb3VuZCB7Z30iLAogICAgICAgICAgICAgIGYie2ludCgocmVmID49IDApLnN1bSgpKX0ve3dvcmxkKiozfSBzb2xpZCIKICAgICAgICAgICAgICArICgiIiBpZiBzYW1lIGVsc2UgZiIgIG5kaWZmPXtpbnQoKG1pbmUgIT0gcmVmKS5zdW0oKSl9IikpCgoKZGVmIGNoZWNrX3JlbmRlcihyZXM6IGludCwgbXM6IGludCwgd29ybGQ6IGludCwgZ3JvdW5kOiBpbnQsIHBpdGNoOiBmbG9hdCwKICAgICAgICAgICAgICAgICBmb3Y6IGZsb2F0LCBiYXRjaGVzOiB0dXBsZVtpbnQsIC4uLl0sIG1pbl9zdGVwX2FncmVlOiBmbG9hdCk6CiAgICAjIEEgbm90ZSB3b3J0aCBoYXZpbmcgaGVyZSwgYmVjYXVzZSBpdCBsb29rcyBsaWtlIGEgYnVnIGFuZCBpcyBub3QuIFRoZQogICAgIyBjYW1lcmEgc2l0cyBhdCAod29ybGQvMiwgZ3JvdW5kKzEuNiwgd29ybGQvMikgYW5kIHRoZSB0cmVlIGFuZCB0aGUgaG91c2UKICAgICMgYXJlIHBsYWNlZCBhdCBmaXhlZCBvZmZzZXRzIGZyb20gVEhBVCBwb2ludCAoa2VybmVscy5jdToxODMxLCA6MTg2OCksIHNvCiAgICAjIHRoZSBsb2NhbCBnZW9tZXRyeSBhcm91bmQgdGhlIGNhbWVyYSBpcyBpZGVudGljYWwgYXQgZXZlcnkgd29ybGQgc2l6ZSBhbmQKICAgICMgb25seSB0aGUgZmFyIHdhbGwgcmVjZWRlcy4gVGhlIGZhciB3YWxsIGNvbnRyaWJ1dGVzIG5vIGhpdHMgLS0gYSByYXkgdGhhdAogICAgIyByZWFjaGVzIGl0IGlzIG91dCBvZiBib3VuZHMgYW5kIG1hcmNoZXMgdG8gdGhlIGNhcCBlaXRoZXIgd2F5LiBTbyB3b3JsZHMKICAgICMgMzIgYW5kIDQ4IGhhdmUgdGhlIHNhbWUgbWVhbiBEREEgc3RlcCBjb3VudC4gVmVyaWZpZWQgYWdhaW5zdAogICAgIyBgbWluZWN1X3JlZi9iZW5jaC9kZGFfbW9kZWwucHlgLCB3aGljaCByZXBvcnRzIHRoZSBzYW1lIDE2LjAyOSBmb3IgYm90aC4KICAgIHByaW50KGYiXG4yLTUuIHRoZSByZW5kZXIsIHdvcmxkIHt3b3JsZH1eMyBncm91bmQge2dyb3VuZH0gcmVzIHtyZXN9ICIKICAgICAgICAgIGYibWF4X3N0ZXBzIHttc30gcGl0Y2gge3BpdGNoOisuMWZ9IGZvdiB7Zm92fSIpCiAgICBNID0gZGRhX21vZGVsKCkKICAgIHZveCA9IE0uYnVpbGRfd29ybGQod29ybGQsIGdyb3VuZCkgICAgICAgICAgICAgICMgW1ksIFgsIFpdIGludDgKICAgIGV5ZSA9ICh3b3JsZCAvIDIuMCwgZ3JvdW5kICsgMS42LCB3b3JsZCAvIDIuMCkKICAgIGNhbXMgPSBucC5hcnJheShbW3dvcmxkIC8gMi4wLCBmbG9hdChncm91bmQpLCB3b3JsZCAvIDIuMCwgMC4wLCBwaXRjaF1dLAogICAgICAgICAgICAgICAgICAgIGR0eXBlPW5wLmZsb2F0MzIpCgogICAgYjY0ID0gYmFzaXM2NChjYW1zLCBmb3YsIHJlcywgcmVzKQogICAgcmVmX2ltZywgcmVmX3N0ZXBzID0gcmVuZGVyX29uZSh2b3gsIGI2NCwgcmVzLCByZXMsIG1zKQoKICAgIGZyb20gdGlueWdyYWQgaW1wb3J0IFRlbnNvciwgZHR5cGVzCiAgICB2b3hUID0gVGVuc29yKG5wLmFzY29udGlndW91c2FycmF5KHZveC5yZXNoYXBlKDEsIHdvcmxkLCB3b3JsZCwgd29ybGQpKSwKICAgICAgICAgICAgICAgICAgZHR5cGU9ZHR5cGVzLmludDgpLmNvbnRpZ3VvdXMoKQogICAgYmFzaXNUID0gY2FtZXJhX2Jhc2lzKFRlbnNvcihjYW1zLnRvbGlzdCgpLCBkdHlwZT1kdHlwZXMuZmxvYXQzMiksIGZvdiwgcmVzLCByZXMpCgogICAgb2JzLCBzdGVwcyA9IHJlbmRlcih2b3hULCBiYXNpc1QsIHJlcywgcmVzLCBtcywgY291bnRfc3RlcHM9VHJ1ZSkKICAgIGdvdCA9IG9icy5udW1weSgpWzBdCiAgICBzdCA9IHN0ZXBzLm51bXB5KClbMCwgMF0uYXN0eXBlKG5wLmludDMyKQoKICAgICMgMi4gdGhlIGltYWdlLCBleGFjdCBvbiB1aW50OAogICAgZCA9IChnb3QuYXN0eXBlKG5wLmludDMyKSAhPSByZWZfaW1nLmFzdHlwZShucC5pbnQzMikpCiAgICBiYWRfcHggPSBpbnQoZC5hbnkoYXhpcz0yKS5zdW0oKSkKICAgIGNoZWNrKGJhZF9weCA9PSAwLCAidWludDggb2JzZXJ2YXRpb24gZXF1YWxzIHRoZSBmbG9hdDY0IHRyYW5zY3JpcHRpb24iLAogICAgICAgICAgZiJ7YmFkX3B4fS97cmVzICogcmVzfSBwaXhlbHMgZGlmZmVyLCBtYXggY2hhbm5lbCBkZWx0YSAiCiAgICAgICAgICBmIntpbnQobnAuYWJzKGQpLm1heCgpKX0iKQoKICAgICMgMy4gdGhlIHN0ZXAgY291bnRzCiAgICBhZ3JlZSA9IGludCgoc3QgPT0gcmVmX3N0ZXBzKS5zdW0oKSkKICAgIGZyYWMgPSBhZ3JlZSAvIChyZXMgKiByZXMpCiAgICBjaGVjayhmcmFjID49IG1pbl9zdGVwX2FncmVlLAogICAgICAgICAgInBlci1waXhlbCBEREEgc3RlcCBjb3VudHMgYWdyZWUgd2l0aCB0aGUgZmxvYXQ2NCB0cmFuc2NyaXB0aW9uIiwKICAgICAgICAgIGYie2FncmVlfS97cmVzICogcmVzfSA9IHsxMDAgKiBmcmFjOi4yZn0lICAiCiAgICAgICAgICBmIihtZWFuIHtzdC5tZWFuKCk6LjNmfSB2cyB7cmVmX3N0ZXBzLm1lYW4oKTouM2Z9KSIpCgogICAgIyA0LiB0aGUgY291bnRlciBpcyBpbmVydAogICAgcGxhaW4gPSByZW5kZXIodm94VCwgYmFzaXNULCByZXMsIHJlcywgbXMpLm51bXB5KClbMF0KICAgIGNoZWNrKG5wLmFycmF5X2VxdWFsKHBsYWluLCBnb3QpLAogICAgICAgICAgImNvdW50X3N0ZXBzPVRydWUgZG9lcyBub3QgY2hhbmdlIHRoZSBvYnNlcnZhdGlvbiIsCiAgICAgICAgICAidGhlIHRpbWVkIHBhdGggYW5kIHRoZSB2ZXJpZmllZCBwYXRoIGFyZSB0aGUgc2FtZSBwcm9ncmFtIikKCiAgICAjIDUuIGJhdGNoIGludmFyaWFuY2UuIFRoZSByZWZlcmVuY2UgZ2l2ZXMgZXZlcnkgZW52IHRoZSBzYW1lIHdvcmxkIEFORCB0aGUKICAgICMgc2FtZSBzcGF3biwgc28gd2l0aCBvbmUgc2hhcmVkIGNhbWVyYSBldmVyeSBlbnYncyBpbWFnZSBtdXN0IGJlIGlkZW50aWNhbAogICAgIyBhbmQgbXVzdCBlcXVhbCB0aGUgc2luZ2xlLWVudiBpbWFnZS4gVGhpcyBpcyB0aGUgY2hlY2sgdGhhdCBjYXRjaGVzIHRoZQogICAgIyBgYmFzaXNbOiwgaSwgTm9uZSwgTm9uZV1gIHNoYXBlIGJ1Zywgd2hpY2ggcHV0IHRoZSBiYXRjaCBheGlzIGluIHRoZSB3cm9uZwogICAgIyBwbGFjZSBhbmQgd2FzIGludmlzaWJsZSBhdCBCPTEuCiAgICAjCiAgICAjIEEgc2Vjb25kLCBzZXBhcmF0ZSBzdWItY2hlY2s6IHdpdGggYSAqcGVyLWVudiogY2FtZXJhIGV2ZXJ5IGVudiBtdXN0IG1hdGNoCiAgICAjIGEgZmxvYXQ2NCByZW5kZXIgb2YgSVRTIE9XTiBjYW1lcmEuIENvbXBhcmluZyBlYWNoIGVudiBhZ2FpbnN0IGl0cyBvd24KICAgICMgcmVmZXJlbmNlIGlzIHN0cm9uZ2VyIHRoYW4gY29tcGFyaW5nIGVudiAwIGFnYWluc3QgdGhlIHNoYXJlZC1jYW1lcmEKICAgICMgcmVmZXJlbmNlLCBhbmQgaXQgaXMgdGhlIG9ubHkgdmVyc2lvbiBvZiB0aGlzIGNoZWNrIHRoYXQgaXMgY29ycmVjdCB3aGVuCiAgICAjIHRoZSBjYW1lcmFzIGRpZmZlci4KICAgICMKICAgICMgVGhlIHNwcmVhZCBoZXJlIGlzIFdJREUgb24gcHVycG9zZTogKy0xLjIgcmFkLCBzbyB0aGUgaW1hZ2VzIGdlbnVpbmVseQogICAgIyBkaWZmZXIgYW5kIHRoZSBjaGVjayBoYXMgdGVldGguIFRoZSB0aW1lZCBiYXR0ZXJ5J3Mgb3duIHBlci1lbnYgc3ByZWFkIGlzCiAgICAjIDAuMDAxKihpJTk3LTQ4KSA9ICstMC4wNDggcmFkLCBhbmQgdGhhdCBpcyBkZWxpYmVyYXRlbHkgcmVwb3J0ZWQgYXMgYQogICAgIyBjYXZlYXQgb24gdGhhdCBoYXJuZXNzIHJhdGhlciB0aGFuIHVzZWQgaGVyZSAtLSBhdCArLTAuMDQ4IHJhZCB0aGUgd2hvbGUKICAgICMgYmF0Y2ggcmVuZGVycyB0aGUgU0FNRSAzLWNvbG91ciBpbWFnZSwgYmVjYXVzZSB0aGUgc2NlbmUgaGFzIHRocmVlIGNvbG91cnMKICAgICMgYW5kIGEgMC4wNSByYWQgcm90YXRpb24gbW92ZXMgdGhlIGhvcml6b24gYnkgdW5kZXIgdHdvIHBpeGVscy4gTm90aGluZwogICAgIyBkb3duc3RyZWFtIGNhbiB1c2UgdGhlIG9ic2VydmF0aW9uIHRvIHRlbGwgdGhlIGVudnMgYXBhcnQgYXQgdGhhdCBzcHJlYWQuCiAgICBpZGVudCwgdmFyaWVkX29rLCB2YXJpZWRfZGV0YWlsID0gVHJ1ZSwgVHJ1ZSwgIiIKICAgIGZvciBCIGluIGJhdGNoZXM6CiAgICAgICAgdm94QiA9IFRlbnNvcihucC5hc2NvbnRpZ3VvdXNhcnJheShucC5zdGFjayhbdm94XSAqIEIpKSwKICAgICAgICAgICAgICAgICAgICAgIGR0eXBlPWR0eXBlcy5pbnQ4KS5jb250aWd1b3VzKCkKICAgICAgICBzaGFyZWQgPSBucC5hcnJheShbW3dvcmxkIC8gMi4wLCBmbG9hdChncm91bmQpLCB3b3JsZCAvIDIuMCwgMC4wLCBwaXRjaF1dCiAgICAgICAgICAgICAgICAgICAgICAgICAgKiBCLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIG9CID0gcmVuZGVyKHZveEIsIGNhbWVyYV9iYXNpcyhUZW5zb3Ioc2hhcmVkLnRvbGlzdCgpLCBkdHlwZT1kdHlwZXMuZmxvYXQzMiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvdiwgcmVzLCByZXMpLCByZXMsIHJlcywgbXMpLm51bXB5KCkKICAgICAgICBpZiBvQi5zaGFwZSAhPSAoQiwgcmVzLCByZXMsIDMpIG9yIFwKICAgICAgICAgICBub3QgYWxsKG5wLmFycmF5X2VxdWFsKG9CWzBdLCBvQltpXSkgZm9yIGkgaW4gcmFuZ2UoQikpIG9yIFwKICAgICAgICAgICBub3QgbnAuYXJyYXlfZXF1YWwob0JbMF0sIGdvdCk6CiAgICAgICAgICAgIGlkZW50ID0gRmFsc2UKICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgc3ByZWFkID0gbnAuYXJyYXkoW1t3b3JsZCAvIDIuMCwgZmxvYXQoZ3JvdW5kKSwgd29ybGQgLyAyLjAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAxLjIgKiAoaSAlIDcgLSAzKSwgcGl0Y2hdIGZvciBpIGluIHJhbmdlKEIpXSwKICAgICAgICAgICAgICAgICAgICAgICAgICBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIG9TID0gcmVuZGVyKHZveEIsIGNhbWVyYV9iYXNpcyhUZW5zb3Ioc3ByZWFkLnRvbGlzdCgpLCBkdHlwZT1kdHlwZXMuZmxvYXQzMiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvdiwgcmVzLCByZXMpLCByZXMsIHJlcywgbXMpLm51bXB5KCkKICAgICAgICBiYWQgPSAwCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UoQik6CiAgICAgICAgICAgIHJlZl9pLCBfID0gcmVuZGVyX29uZSh2b3gsIGJhc2lzNjQoc3ByZWFkW2k6aSArIDFdLCBmb3YsIHJlcywgcmVzKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlcywgcmVzLCBtcykKICAgICAgICAgICAgaWYgbm90IG5wLmFycmF5X2VxdWFsKG9TW2ldLCByZWZfaSk6CiAgICAgICAgICAgICAgICBiYWQgKz0gMQogICAgICAgIG5fZGlzdGluY3QgPSBsZW4oe29TW2ldLnRvYnl0ZXMoKSBmb3IgaSBpbiByYW5nZShCKX0pCiAgICAgICAgaWYgYmFkOgogICAgICAgICAgICB2YXJpZWRfb2ssIHZhcmllZF9kZXRhaWwgPSBGYWxzZSwgZiJ7YmFkfS97Qn0gZW52cyBkaXNhZ3JlZSB3aXRoIHRoZWlyIG93biByZWZlcmVuY2UiCiAgICAgICAgZWxpZiBCID4gMSBhbmQgbl9kaXN0aW5jdCA8IDI6CiAgICAgICAgICAgIHZhcmllZF9vaywgdmFyaWVkX2RldGFpbCA9IEZhbHNlLCAiYWxsIGVudnMgaWRlbnRpY2FsIGRlc3BpdGUgYSArLTEuMiByYWQgeWF3IHNwcmVhZCIKICAgICAgICBlbHNlOgogICAgICAgICAgICB2YXJpZWRfZGV0YWlsID0gKGYie25fZGlzdGluY3R9IGRpc3RpbmN0IGltYWdlcyBvdmVyIHtCfSBlbnZzIGF0IGEgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYiKy0xLjIgcmFkIHlhdyBzcHJlYWQsIGVhY2ggbWF0Y2hpbmcgaXRzIG93biBmbG9hdDY0IHJlZmVyZW5jZSIpCgogICAgY2hlY2soaWRlbnQsIGYiYSBzaGFyZWQgY2FtZXJhIGdpdmVzIGlkZW50aWNhbCBlbnZzIGF0IGJhdGNoIHtsaXN0KGJhdGNoZXMpfSIsCiAgICAgICAgICAiY2F0Y2hlcyBhIGJhdGNoIGF4aXMgaW4gdGhlIHdyb25nIHBsYWNlLCB3aGljaCBCPTEgY2Fubm90IikKICAgIGNoZWNrKHZhcmllZF9vaywgZiJwZXItZW52IGNhbWVyYXMgcmVuZGVyIGNvcnJlY3RseSBhdCBiYXRjaCB7bGlzdChiYXRjaGVzKX0iLAogICAgICAgICAgdmFyaWVkX2RldGFpbCkKCgpkZWYgY2hlY2tfZ3JvdW5kX2dhdGUoKSAtPiBOb25lOgogICAgcHJpbnQoIlxuNi4gdGhlIHdvcmxkZ2VuIGZsb29yIGFuZCB0aGUgcGh5c2ljcyBmbG9vciIpCiAgICBmb3Igd29ybGQgaW4gKDE2LCAzMiwgNDgpOgogICAgICAgIGcgPSBncm91bmRfb2Yod29ybGQpCiAgICAgICAgY2hlY2soZyA9PSB3b3JsZCAvLyA0LCBmIndvcmxkIHt3b3JsZH06IGdyb3VuZCB7Z30gPT0gd29ybGQvNCB7d29ybGQgLy8gNH0iLAogICAgICAgICAgICAgICJwaHlzaWNzX2tlcm5lbCBmbG9vcnMgdGhlIGFnZW50IGF0IHdvcmxkX3kvNDsgYW55dGhpbmcgZWxzZSAiCiAgICAgICAgICAgICAgImJ1cmllcyBvciBmbG9hdHMgaXQiKQoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1pbi1zdGVwLWFncmVlIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjk5KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXF1aWNrIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBwcmludCgiPSIgKiA3OCkKICAgIHByaW50KCIgIG1pbmVjdV90ZyB2ZXJpZmljYXRpb24gLS0gdGhlIHRlbnNvciByZW5kZXIgYWdhaW5zdCBhIGZsb2F0NjQgIgogICAgICAgICAgInRyYW5zY3JpcHRpb24iKQogICAgcHJpbnQoZiIgIERFVj17b3MuZW52aXJvbi5nZXQoJ0RFVicpfSAgdGlueWdyYWQgZnJvbSB0aGUgdmVudiwgQ1BVIGJhY2tlbmQiKQogICAgcHJpbnQoIj0iICogNzgpCgogICAgTSA9IGRkYV9tb2RlbCgpCiAgICBjaGVja193b3JsZChNKQogICAgY2hlY2tfZ3JvdW5kX2dhdGUoKQoKICAgIGNhc2VzID0gWygzMiwgMzIsIDE2LCA0LCAwLjApLCAoMzIsIDMyLCAxNiwgNCwgLTAuMildCiAgICBpZiBub3QgYS5xdWljazoKICAgICAgICBjYXNlcyArPSBbKDMyLCAzMiwgMzIsIDgsIDAuMCksICgzMiwgMzIsIDQ4LCAxMiwgMC4wKSwgKDMyLCAxNiwgMTYsIDQsIDAuMCldCiAgICBmb3IgcmVzLCBtcywgd29ybGQsIGdyb3VuZCwgcGl0Y2ggaW4gY2FzZXM6CiAgICAgICAgY2hlY2tfcmVuZGVyKHJlcz1yZXMsIG1zPW1zLCB3b3JsZD13b3JsZCwgZ3JvdW5kPWdyb3VuZCwgcGl0Y2g9cGl0Y2gsCiAgICAgICAgICAgICAgICAgICAgIGZvdj03MC4wLCBiYXRjaGVzPSgxLCA1KSwgbWluX3N0ZXBfYWdyZWU9YS5taW5fc3RlcF9hZ3JlZSkKCiAgICBwcmludCgiXG4iICsgIj0iICogNzgpCiAgICBpZiBGQUlMUzoKICAgICAgICBwcmludChmIiAge2xlbihGQUlMUyl9IEZBSUxVUkUoUykuIFRoZSB0ZW5zb3IgcmVuZGVyIGlzIE5PVCB2ZXJpZmllZC4iKQogICAgICAgIGZvciBmIGluIEZBSUxTOgogICAgICAgICAgICBwcmludChmIiAgICAtIHtmfSIpCiAgICAgICAgcmV0dXJuIDEKICAgIHByaW50KCIgIGFsbCBjaGVja3MgcGFzc2VkLiBUaGUgdGVuc29yIHJlbmRlciBpcyB2ZXJpZmllZCBhZ2FpbnN0IGFuICIKICAgICAgICAgICJpbmRlcGVuZGVudCIpCiAgICBwcmludCgiICBmbG9hdDY0IHRyYW5zY3JpcHRpb24gb2YgdGhlIHNhbWUgYWxnb3JpdGhtLCBhbmQgdGhlIHN0ZXAgY291bnRzIikKICAgIHByaW50KCIgIGFncmVlIHBpeGVsIGZvciBwaXhlbCBvbiB0aGUgdGllLWluc2Vuc2l0aXZlIHBhcnRzLiIpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBjb2RlID0gbWFpbigpCiAgICBQYXRoKCIvdG1wL21pbmVjdV90Z192ZXJpZnkubG9nIikud3JpdGVfdGV4dCgiXG4iLmpvaW4oTElORVMpICsgIlxuIikKICAgIHN5cy5leGl0KGNvZGUpCg==", "a79e7cdcfbbb10c7b4adef880f87a27b3d471489c283c6d5c07d74aa80843caf"], ["minecu_tg/codegen.py", "IiIiQ2Vuc3VzIGFuZCBjYXB0dXJlIG9mIHRoZSBjb2RlIFRpbnlHcmFkIGFjdHVhbGx5IGdlbmVyYXRlcy4KClRocmVlIHF1ZXN0aW9ucywgYW5kIHRoZSBtYWNoaW5lcnkgZm9yIGVhY2g6CgoxLiAqKkhvdyBtYW55IGtlcm5lbHMgZG9lcyBvbmUgcmVuZGVyIGxhdW5jaD8qKiBgR2xvYmFsQ291bnRlcnMua2VybmVsX2NvdW50YCwKICAgd2hpY2ggaXMgdGhlIGNvdW50IGBERUJVRz0yYCByZXBvcnRzIGFuZCB0aGUgc2FtZSBjb3VudGVyIGBwZXJmX2tkYS5weWAgdXNlZAogICBmb3IgS0RBJ3MgMTIsMDgyLgoKMi4gKipIb3cgbWFueSBkaXN0aW5jdCBjb21waWxlZCBwcm9ncmFtcyBpcyB0aGF0PyoqIGBsZW4odG9fcHJvZ3JhbV9jYWNoZSlgCiAgIGRlbHRhcy4gRGlzdGluY3QgcHJvZ3JhbXMgaXMgbm90IGxhdW5jaGVzOiBhIGxvb3AgdGhhdCB1bnJvbGxzIHRvIHRoZSBzYW1lCiAgIHNoYXBlIHR3aWNlIHJldXNlcyBvbmUgcHJvZ3JhbS4gQm90aCBudW1iZXJzIGFyZSByZXBvcnRlZCwgYW5kIHRoZQogICBkaWZmZXJlbmNlIGJldHdlZW4gdGhlbSBpcyBzdGF0ZWQgcmF0aGVyIHRoYW4gbGVmdCBmb3IgdGhlIHJlYWRlci4KCjMuICoqV2hhdCBpcyB0aGUgZ2VuZXJhdGVkIENVREEgQz8qKiBDYXB0dXJlZCBvbiB0aGUgcmVhbCBiYWNrZW5kIGJ5IHdyYXBwaW5nCiAgIGBEZXZpY2VbREVGQVVMVF0uY29tcGlsZXIuY29tcGlsZV9jYWNoZWRgLgoKV0hZIGBQQVJBTExFTD0wYCBJUyBNQU5EQVRPUlkgQU5EIE5PVCBBTiBPUFRJTUlTQVRJT04KLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmB0aW55Z3JhZC9lbmdpbmUvcmVhbGl6ZS5weToyNzNgOgoKICAgIHBvb2wgPSBOb25lIGlmIGxlbih0b2RvKSA9PSAxIG9yIGFueShiZWFtKSBlbHNlIGdldF93b3JrZXJfcG9vbCgpCgpgbG93ZXJfYW5kX2NvbXBpbGVgIGNvbXBpbGVzIGluIGEgKiptdWx0aXByb2Nlc3Npbmcgd29ya2VyIHBvb2wqKiB3aGVuZXZlciBtb3JlCnRoYW4gb25lIGtlcm5lbCBuZWVkcyBjb21waWxpbmcsIGFuZCBvbmx5IHRha2VzIHRoZSBpbi1wcm9jZXNzIGBtYXBgIGJyYW5jaAp3aGVuIGV4YWN0bHkgb25lIGlzIG91dHN0YW5kaW5nLiBgY29tcGlsZV9jYWNoZWRgIHJ1bnMgaW5zaWRlIHdoaWNoZXZlciBwcm9jZXNzCmRvZXMgdGhlIHdvcmssIHNvIGFuIGluLXByb2Nlc3MgaG9vayBvbiB0aGUgcGFyZW50IHNlZXMgb25seSB0aGUgYGxlbih0b2RvKT09MWAKcm91bmRzLiBUaGF0IGlzIHdoeSBga2VybmVscy9idWlsZF9ub3RlYm9va3MucHk6NTc4YCBjYXB0dXJlZCAxIGtlcm5lbCBvdXQgb2YKMTIsMDgyIGZvciBLREEgKGBub3Rlcy8xMy1zY2hlZHVsZS12cy1kZWNheS5tZGAgwqc0KS4gYFBBUkFMTEVMPTBgIG1ha2VzCmBnZXRfd29ya2VyX3Bvb2woKWAgcmV0dXJuIE5vbmUsIHNvIGV2ZXJ5dGhpbmcgY29tcGlsZXMgaW4tcHJvY2VzcyBhbmQgdGhlIGhvb2sKc2VlcyB0aGUgd2hvbGUgc2NoZWR1bGUuIFRoaXMgbW9kdWxlIHNldHMgaXQgYW5kIGFzc2VydHMgaXQgdG9vayBlZmZlY3QuCgpPbiB0aGUgQ1BVIGJhY2tlbmQgdGhlcmUgaXMgbm8gYGNvbXBpbGVfY2FjaGVkYCAodGhlIENQVSBydW50aW1lIGlzIG5vdCBhCmNvbXBpbGVyKSwgc28gdGhlIHNvdXJjZSBjYXB0dXJlIGlzIHVuYXZhaWxhYmxlIHRoZXJlIGFuZCBpcyByZXBvcnRlZCBhcyBzdWNoCnJhdGhlciB0aGFuIHNpbGVudGx5IHNraXBwZWQuIFRoZSBrZXJuZWwgKmNvdW50cyogd29yayBvbiBldmVyeSBiYWNrZW5kLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjb2xsZWN0aW9ucwppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKIyBtb2R1bGUtbGV2ZWwgc28gdW5pbnN0YWxsIGNhbiBmaW5kIHdoYXQgaW5zdGFsbCBjcmVhdGVkCkNBUFRVUkVfU1RBVEU6IGxpc3RbZGljdF0gPSBbXQoKCmRlZiBpbnN0YWxsX2xhdW5jaF9yZWNvcmRlcigpIC0+IGxpc3Rbc3RyXToKICAgICIiIlJlY29yZCB0aGUgbmFtZSBvZiBldmVyeSBrZXJuZWwgbGF1bmNoLCBpbiBvcmRlci4KCiAgICBUaGlzIGlzIHRoZSBzYW1lIGNlbnN1cyBwb2ludCBgREVCVUc9MmAgcHJpbnRzIGZyb20gLS0gdGhlIHBlci1sYXVuY2ggbGluZQogICAgaW4gYHRpbnlncmFkL2VuZ2luZS9yZWFsaXplLnB5OnRyYWNrX3N0YXRzYCAtLSByZWFjaGVkIGJ5IHdyYXBwaW5nIHRoZQogICAgZGV2aWNlJ3MgYHJ1bnRpbWVfdC5fX2NhbGxfX2AgaW5zdGVhZCBvZiBieSBzZXR0aW5nIGBERUJVRz0yYC4gVGhlIHJlYXNvbgogICAgZm9yIHByZWZlcnJpbmcgdGhlIGhvb2s6IGBERUJVRz49MmAgbWFrZXMgYHJ1bl9saW5lYXJgIHBhc3MgYHdhaXQ9VHJ1ZWAKICAgIChgcmVhbGl6ZS5weTozMThgKSwgc28gZXZlcnkgbGF1bmNoIGJlY29tZXMgYSBob3N0LXN5bmNocm9uaXNlZCBldmVudC4gVGhlCiAgICAqY291bnQqIGlzIHRoZSBzYW1lIGVpdGhlciB3YXksIGJ1dCBhIGNlbnN1cyB0aGF0IGFsc28gcmVwb3J0cyB0aW1pbmcgbXVzdAogICAgbm90IGJlIHRoZSBpbnN0cnVtZW50ZWQgb25lLgoKICAgIE5hbWVzIGNvbWUgZnJvbSB0aGUgUHJvZ3JhbUluZm8gYXR0YWNoZWQgdG8gdGhlIGNvbXBpbGVkIEFTVCwgc28gYW4gZW50cnkKICAgIGxpa2UgYEVfMzJfMzJfNjRfOF80YCBpcyB0aW55Z3JhZCdzIG93biBzdW1tYXJ5IG9mIHRoZSBrZXJuZWw6IHRoZSBlbGVtZW50CiAgICBzaXplcyBhbmQgcmFua3Mgb2YgaXRzIGJvdW5kcy4gVHdvIGxhdW5jaGVzIHdpdGggdGhlIHNhbWUgbmFtZSBhcmUgdGhlIHNhbWUKICAgIGNvbXBpbGVkIHByb2dyYW0gcmV1c2VkLCB3aGljaCBpcyBleGFjdGx5IHRoZSByZXBldGl0aW9uIHRoZSBEREEgbG9vcAogICAgcHJlZGljdGlvbiBpcyBhYm91dC4KICAgICIiIgogICAgZnJvbSB0aW55Z3JhZCBpbXBvcnQgRGV2aWNlCgogICAgZGV2ID0gRGV2aWNlW0RldmljZS5ERUZBVUxUXQogICAgYmFzZSA9IGRldi5ydW50aW1lX3QgaWYgaXNpbnN0YW5jZShkZXYucnVudGltZV90LCB0eXBlKSBlbHNlIHR5cGUoZGV2LnJ1bnRpbWVfdCkKICAgIG9yaWcgPSBiYXNlLl9fY2FsbF9fCiAgICBuYW1lczogbGlzdFtzdHJdID0gW10KCiAgICBkZWYgc3B5KHNlbGYsICphLCAqKmspOgogICAgICAgICMgVHdvIHRoaW5ncyBwZXIgbGF1bmNoLCBhbmQgdGhlIHNlY29uZCBpcyB0aGUgb25lIHRoYXQgbWF0dGVyczoKICAgICAgICAjICAgKiB0aGUgbmFtZSAtLSBldmVyeSBiYWNrZW5kJ3MgUHJvZ3JhbSBjYXJyaWVzIHRoZSBjb21waWxlZCBuYW1lCiAgICAgICAgIyAgICAgKENQVVByb2dyYW0ubmFtZSwgSENRMkNvbXBpbGVkLm5hbWUpLiBOT1RFIHRoZSBuYW1lIGlzIGRlcml2ZWQKICAgICAgICAjICAgICBmcm9tIHRoZSBrZXJuZWwncyAqYm91bmRzKiwgbm90IGl0cyBib2R5LCBzbyBhIHJ1biBvZiBsYXVuY2hlcwogICAgICAgICMgICAgIHRoYXQgc2hhcmUgYSBuYW1lIGNhbiBzdGlsbCBiZSBhIHJ1biBvZiBESUZGRVJFTlQgcHJvZ3JhbXMuIFRoYXQKICAgICAgICAjICAgICBpcyBleGFjdGx5IHRoZSB0cmFwIHRoaXMgY2Vuc3VzIGV4aXN0cyB0byBhdm9pZC4KICAgICAgICAjICAgKiB0aGUgcHJvZ3JhbSBvYmplY3QgaWQgLS0gYGdldF9ydW50aW1lYCBjYWNoZXMgb25lIFByb2dyYW0gcGVyCiAgICAgICAgIyAgICAgY29tcGlsZWQgQVNULCBzbyB3aXRoaW4gb25lIG1lYXN1cmVtZW50IHR3byBsYXVuY2hlcyB3aXRoIHRoZQogICAgICAgICMgICAgIHNhbWUgaWQgYXJlIHRoZSBzYW1lIGNvZGUsIGFuZCBkaWZmZXJlbnQgaWRzIGFyZSBkaWZmZXJlbnQgY29kZS4KICAgICAgICAjIFRoZSBuYW1lIGZhbGxzIGJhY2sgdG8gdGhlIHJlcHIgc28gYSBtaXNzIGlzIHZpc2libGUsIG5vdCBkcm9wcGVkLgogICAgICAgIG5hbWVzLmFwcGVuZChmIntnZXRhdHRyKHNlbGYsICduYW1lJywgTm9uZSkgb3IgcmVwcihzZWxmKX0je2lkKHNlbGYpICYgMHhmZmZmZmY6MDZ4fSIpCiAgICAgICAgcmV0dXJuIG9yaWcoc2VsZiwgKmEsICoqaykKCiAgICBiYXNlLl9fY2FsbF9fID0gc3B5CiAgICBDQVBUVVJFX1NUQVRFLmFwcGVuZCh7ImNscyI6IGJhc2UsICJhdHRyIjogIl9fY2FsbF9fIiwgIm9yaWciOiBvcmlnfSkKICAgIHJldHVybiBuYW1lcwoKCmRlZiB1bmluc3RhbGwoc2luaykgLT4gTm9uZToKICAgIHdoaWxlIENBUFRVUkVfU1RBVEU6CiAgICAgICAgaCA9IENBUFRVUkVfU1RBVEUucG9wKCkKICAgICAgICBpZiAiY2xzIiBpbiBoOgogICAgICAgICAgICBoWyJjbHMiXS5fX2NhbGxfXyA9IGhbIm9yaWciXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGRlbCBoWyJjb21wIl0uX19kaWN0X19bImNvbXBpbGVfY2FjaGVkIl0KICAgICAgICAgICAgZXhjZXB0IChLZXlFcnJvciwgQXR0cmlidXRlRXJyb3IpOgogICAgICAgICAgICAgICAgaFsiY29tcCJdLmNvbXBpbGVfY2FjaGVkID0gaFsib3JpZyJdCgoKZGVmIGluc3RhbGxfY2FwdHVyZShjYXB0dXJlX29ubHk6IGJvb2wgPSBGYWxzZSkgLT4gdHVwbGVbbGlzdFtzdHJdLCBzdHIgfCBOb25lXToKICAgICIiIldyYXAgYGNvbXBpbGVfY2FjaGVkYCBvbiB0aGUgZGVmYXVsdCBkZXZpY2Ugc28gZXZlcnkga2VybmVsJ3Mgc291cmNlIGlzCiAgICBjYXB0dXJlZC4gUmV0dXJucyAoc2luaywgbm90ZSkuIGBub3RlYCBleHBsYWlucyBhbiBlbXB0eSBzaW5rLgoKICAgIGBjYXB0dXJlX29ubHk9VHJ1ZWAgcmV0dXJucyBhbiBlbXB0eSAiYmluYXJ5IiBpbnN0ZWFkIG9mIGNvbXBpbGluZywgZm9yIHRoZQogICAgTlVMTC1kZXZpY2UgY2FzZSB3aGVyZSB0aGUgaG9zdCBoYXMgbm8gbnZjYyBhbmQgdGhlIHByb2dyYW0gaXMgbmV2ZXIKICAgIGV4ZWN1dGVkLiBJdCBjaGFuZ2VzIG5vdGhpbmcgYWJvdXQgdGhlICpnZW5lcmF0ZWQgc291cmNlKiwgd2hpY2ggaXMKICAgIHByb2R1Y2VkIGJlZm9yZSB0aGUgY29tcGlsZSBjYWxsLCBidXQgaXQgZG9lcyBtZWFuIG5vIGN1YmluIGV4aXN0cyBmb3IgdGhpcwogICAgcGFydGljdWxhciBjYXB0dXJlOiB0aGUgU0FTUyBtYXRyaXggaW4gdGhlIG5vdGVzIGNvbWVzIGZyb20gdGhlIFQ0IHJ1biBhbmQKICAgIGZyb20gdGhlIG9mZmxpbmUgYGFyY2hsYWJgIHJvdXRlLCBib3RoIG9mIHdoaWNoIGRvIGNvbXBpbGUuCiAgICAiIiIKICAgIGZyb20gdGlueWdyYWQgaW1wb3J0IERldmljZQoKICAgIGRldiA9IERldmljZVtEZXZpY2UuREVGQVVMVF0KICAgIGNvbXAgPSBnZXRhdHRyKGRldiwgImNvbXBpbGVyIiwgTm9uZSkKICAgIG9yaWcgPSBnZXRhdHRyKGNvbXAsICJjb21waWxlX2NhY2hlZCIsIE5vbmUpIGlmIGNvbXAgaXMgbm90IE5vbmUgZWxzZSBOb25lCiAgICBpZiBvcmlnIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIFtdLCAoZiJiYWNrZW5kIHtEZXZpY2UuREVGQVVMVCFyfSBoYXMgbm8gY29tcGlsZXIuY29tcGlsZV9jYWNoZWQ7ICIKICAgICAgICAgICAgICAgICAgICAibGF1bmNoIGFuZCBwcm9ncmFtIGNvdW50cyBhcmUgc3RpbGwgdmFsaWQsIHNvdXJjZSBjYXB0dXJlIGlzIG5vdCIpCiAgICBzaW5rOiBsaXN0W3N0cl0gPSBbXQogICAgaG9sZGVyID0ge30KCiAgICBkZWYgc3B5KHNyYywgKmEsICoqayk6CiAgICAgICAgc2luay5hcHBlbmQoc3RyKHNyYykpCiAgICAgICAgcmV0dXJuIGIiIiBpZiBjYXB0dXJlX29ubHkgZWxzZSBvcmlnKHNyYywgKmEsICoqaykKCiAgICBob2xkZXJbImNvbXAiXSwgaG9sZGVyWyJvcmlnIl0gPSBjb21wLCBvcmlnCiAgICBjb21wLmNvbXBpbGVfY2FjaGVkID0gc3B5CiAgICBDQVBUVVJFX1NUQVRFLmFwcGVuZChob2xkZXIpCiAgICByZXR1cm4gc2luaywgTm9uZQoKCmRlZiBjZW5zdXMocmVuZGVyX2ZuLCBzaGFwZXMsIG91dF9kaXI6IFBhdGggfCBOb25lID0gTm9uZSwKICAgICAgICAgICBrZWVwX3NvdXJjZXM6IGJvb2wgPSBUcnVlLCBjYXB0dXJlX29ubHk6IGJvb2wgPSBGYWxzZSkgLT4gZGljdDoKICAgICIiIkNvdW50IGtlcm5lbHMgcGVyIHJlbmRlciwgZm9yIGVhY2ggbWF4X3N0ZXBzIGluIGBzaGFwZXNgLgoKICAgIGByZW5kZXJfZm4obWF4X3N0ZXBzKWAgbXVzdCBidWlsZCB0aGUgZ3JhcGguIFRoZSB0ZW5zb3IgaXMgKipyZWFsaXplZCoqLCBub3QKICAgIHJlYWQgYmFjazogYSBgLm51bXB5KClgIHdvdWxkIGFkZCBhIGRldmljZS10by1ob3N0IGNvcHkgdG8gdGhlIGxhdW5jaCBjb3VudCwKICAgIGFuZCB0aGF0IGNvcHkgYmVsb25ncyB0byB0aGUgaGFybmVzcywgbm90IHRvIHRoZSByZW5kZXIuIE9uIHRoZSBOVUxMIGRldmljZQogICAgYSByZWFkLWJhY2sgaXMgaW1wb3NzaWJsZSBhdCBhbGwgKGBvcHNfbnVsbC5weToyOGApLgoKICAgIEVhY2ggc2hhcGUgcmVwb3J0cywgaW4gb3JkZXI6IHRoZSBsYXVuY2ggY291bnQgKEdsb2JhbENvdW50ZXJzKSwgdGhlIG51bWJlcgogICAgb2YgZGlzdGluY3QgY29tcGlsZWQgcHJvZ3JhbXMsIGFuZCB0aGUgb3JkZXJlZCBsaXN0IG9mIGxhdW5jaCBuYW1lcyB3aXRoCiAgICBtdWx0aXBsaWNpdGllcyAtLSBzbyAiMzEgbGF1bmNoZXMgb2Ygb25lIGlkZW50aWNhbCBrZXJuZWwiIGlzIHZpc2libGUgYXMgYQogICAgZmFjdCByYXRoZXIgdGhhbiBpbmZlcnJlZC4KICAgICIiIgogICAgZnJvbSB0aW55Z3JhZCBpbXBvcnQgRGV2aWNlLCBHbG9iYWxDb3VudGVycwogICAgZnJvbSB0aW55Z3JhZC5jb2RlZ2VuIGltcG9ydCB0b19wcm9ncmFtX2NhY2hlCgogICAgc2luaywgbm90ZSA9IGluc3RhbGxfY2FwdHVyZShjYXB0dXJlX29ubHkpCiAgICBsYXVuY2hlc19sb2cgPSBpbnN0YWxsX2xhdW5jaF9yZWNvcmRlcigpCiAgICByb3dzID0gW10KICAgIHRyeToKICAgICAgICBmb3IgbXMgaW4gc2hhcGVzOgogICAgICAgICAgICBuMCA9IGxlbih0b19wcm9ncmFtX2NhY2hlKQogICAgICAgICAgICBzaW5rLmNsZWFyKCkKICAgICAgICAgICAgbGF1bmNoZXNfbG9nLmNsZWFyKCkKICAgICAgICAgICAgR2xvYmFsQ291bnRlcnMucmVzZXQoKQogICAgICAgICAgICBvdXQgPSByZW5kZXJfZm4obXMpCiAgICAgICAgICAgIG91dCA9IG91dFswXSBpZiBpc2luc3RhbmNlKG91dCwgdHVwbGUpIGVsc2Ugb3V0CiAgICAgICAgICAgIG5fZWxlbSwgbl9ieXRlcyA9IDEsIG91dC5kdHlwZS5pdGVtc2l6ZQogICAgICAgICAgICBmb3IgZCBpbiBvdXQuc2hhcGU6CiAgICAgICAgICAgICAgICBuX2VsZW0gKj0gZAogICAgICAgICAgICBvdXQucmVhbGl6ZSgpCiAgICAgICAgICAgIGxhdW5jaGVzID0gR2xvYmFsQ291bnRlcnMua2VybmVsX2NvdW50CiAgICAgICAgICAgIG5fbmV3ID0gbGVuKHRvX3Byb2dyYW1fY2FjaGUpIC0gbjAKICAgICAgICAgICAgbmFtZXMgPSBbbiBmb3IgbiBpbiBsYXVuY2hlc19sb2ddCiAgICAgICAgICAgIGNvdW50cyA9IGNvbGxlY3Rpb25zLkNvdW50ZXIobmFtZXMpCiAgICAgICAgICAgIHJvdyA9IHsKICAgICAgICAgICAgICAgICJtYXhfc3RlcHMiOiBtcywKICAgICAgICAgICAgICAgICJrZXJuZWxfbGF1bmNoZXMiOiBsYXVuY2hlcywKICAgICAgICAgICAgICAgICJuZXdfZGlzdGluY3RfcHJvZ3JhbXMiOiBuX25ldywKICAgICAgICAgICAgICAgICJuZXdfc291cmNlc19jYXB0dXJlZCI6IGxlbihzaW5rKSwKICAgICAgICAgICAgICAgICJ1b3BzX3RvdGFsX2dyYXBoIjogR2xvYmFsQ291bnRlcnMuZ2xvYmFsX29wcywKICAgICAgICAgICAgICAgICJieXRlc19lc3RpbWF0ZSI6IEdsb2JhbENvdW50ZXJzLmdsb2JhbF9tZW0sCiAgICAgICAgICAgICAgICAib2JzX3NoYXBlIjogbGlzdChvdXQuc2hhcGUpLAogICAgICAgICAgICAgICAgIm9ic19ieXRlcyI6IG5fZWxlbSAqIG5fYnl0ZXMsCiAgICAgICAgICAgICAgICAibGF1bmNoX2tleXNfb3JkZXJlZCI6IG5hbWVzLAogICAgICAgICAgICAgICAgImxhdW5jaF9jb3VudHNfYnlfcHJvZ3JhbSI6IGRpY3QoY291bnRzKSwKICAgICAgICAgICAgICAgICJkaXN0aW5jdF9wcm9ncmFtc19sYXVuY2hlZCI6IGxlbihjb3VudHMpLAogICAgICAgICAgICAgICAgIm5fcmVwZWF0ZWRfcHJvZ3JhbSI6IG1heChjb3VudHMudmFsdWVzKCksIGRlZmF1bHQ9MCksCiAgICAgICAgICAgICAgICAibl9yZXBlYXRlZF9wcm9ncmFtX2tleSI6IG1heChjb3VudHMsIGtleT1jb3VudHMuZ2V0KSBpZiBjb3VudHMgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgIm5fZGlzdGluY3RfbmFtZXMiOiBsZW4oe2suc3BsaXQoJyMnKVswXSBmb3IgayBpbiBuYW1lc30pLAogICAgICAgICAgICB9CiAgICAgICAgICAgIGlmIG91dF9kaXIgaXMgbm90IE5vbmUgYW5kIGtlZXBfc291cmNlcyBhbmQgc2luazoKICAgICAgICAgICAgICAgIGQgPSBvdXRfZGlyIC8gZiJtYXhfc3RlcHNfe21zOjA0ZH0iCiAgICAgICAgICAgICAgICBkLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgICAgIGZvciBpLCBzcmMgaW4gZW51bWVyYXRlKHNpbmspOgogICAgICAgICAgICAgICAgICAgIChkIC8gZiJrZXJuZWx7aTowM2R9LmN1Iikud3JpdGVfdGV4dChzcmMpCiAgICAgICAgICAgICAgICByb3dbInNvdXJjZXNfZGlyIl0gPSBzdHIoZCkKICAgICAgICAgICAgICAgIHJvd1sic291cmNlX2J5dGVzIl0gPSBbbGVuKHMpIGZvciBzIGluIHNpbmtdCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHJvdykKICAgIGZpbmFsbHk6CiAgICAgICAgdW5pbnN0YWxsKHNpbmspCgogICAgcmV0dXJuIHsiZGV2aWNlIjogRGV2aWNlLkRFRkFVTFQsICJwYXJhbGxlbCI6IG9zLmVudmlyb24uZ2V0KCJQQVJBTExFTCIpLAogICAgICAgICAgICAiY2FwdHVyZV9ub3RlIjogbm90ZSwgInJvd3MiOiByb3dzfQoKCmRlZiBzdW1tYXJpc2UoYzogZGljdCkgLT4gc3RyOgogICAgbGluZXMgPSBbZiJkZXZpY2U9e2NbJ2RldmljZSddfSAgUEFSQUxMRUw9e2NbJ3BhcmFsbGVsJ119Il0KICAgIGlmIGNbImNhcHR1cmVfbm90ZSJdOgogICAgICAgIGxpbmVzLmFwcGVuZChmIiAgY2FwdHVyZToge2NbJ2NhcHR1cmVfbm90ZSddfSIpCiAgICBsaW5lcy5hcHBlbmQoZiJ7J21heF9zdGVwcyc6PjEwfSB7J2xhdW5jaGVzJzo+OX0geydkaXN0aW5jdCc6Pjh9ICIKICAgICAgICAgICAgICAgICBmInsnbmFtZXMnOj42fSB7J3RvcC1yZXAnOj44fSB7J3VvcHMnOj4xMn0iKQogICAgZm9yIHIgaW4gY1sicm93cyJdOgogICAgICAgIGxpbmVzLmFwcGVuZChmIntyWydtYXhfc3RlcHMnXTo+MTB9IHtyWydrZXJuZWxfbGF1bmNoZXMnXTo+OX0gIgogICAgICAgICAgICAgICAgICAgICBmIntyWydkaXN0aW5jdF9wcm9ncmFtc19sYXVuY2hlZCddOj44fSB7clsnbl9kaXN0aW5jdF9uYW1lcyddOj42fSAiCiAgICAgICAgICAgICAgICAgICAgIGYie3JbJ25fcmVwZWF0ZWRfcHJvZ3JhbSddOj44fSB7clsndW9wc190b3RhbF9ncmFwaCddOj4xMn0iKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaW1wb3J0IGFyZ3BhcnNlCiAgICBpbXBvcnQgc3lzCgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dGRpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXN0ZXBzIiwgZGVmYXVsdD0iMSwyLDQsOCwxNiwzMiw2NCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tanNvbiIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICBwcmludChzdW1tYXJpc2UoY2Vuc3VzKE5vbmUsIFtpbnQoeCkgZm9yIHggaW4gYS5zdGVwcy5zcGxpdCgiLCIpXSwgYS5vdXRkaXIpKSkK", "f926389a8c3064d17224da3f6152330468712e74f27e95a0ff628d6e97a05fb3"], ["minecu_tg/bench.py", "IiIiVGltaW5nIGFuZCBtZW1vcnksIG9uIGEgcmVhbCBkZXZpY2UuCgpXSEFUIElTIFRJTUVELCBBTkQgV0hZIElUIElTIE5PVCBUSEUgV0hPTEUgU1RFUAotLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KVGhlIGhhbmQtd3JpdHRlbiBiYXNlbGluZSdzIG51bWJlciBpcyBhIHdob2xlICpzdGVwKjogc2V2ZW4gb3IgbmluZSBsYXVuY2hlcywKZml2ZSBvZiB0aGVtIHN0YXRlIGtlcm5lbHMsIGFuZCB0aGUgbGFzdCBvbmUgdGhlIHJlbmRlci4gVGlueUdyYWQsIGdpdmVuIHRoZQpyZW5kZXIgYWxvbmUsIGVtaXRzIGEgZ3JhcGggb2Yga2VybmVscy4gVGhvc2UgYXJlIGRpZmZlcmVudCBvYmplY3RzLCBhbmQKYG5vdGVzLzE3YCDCpzguMiBzYXlzIHNvIGJlZm9yZSBhbnkgbnVtYmVyIGlzIHRha2VuLiBUaGlzIG1vZHVsZSB0aGVyZWZvcmUgdGltZXMKdGhyZWUgc2VwYXJhdGUgdGhpbmdzIGFuZCBsYWJlbHMgYWxsIHRocmVlOgoKICAqIGByZW5kZXJgICAgICAgLS0gdGhlIFRpbnlHcmFkIGdyYXBoIGZvciB0aGUgb2JzZXJ2YXRpb24sIGFuZCBub3RoaW5nIGVsc2UuCiAgICAgICAgICAgICAgICAgICAgIFRoaXMgaXMgdGhlIHBlci1vcCBudW1iZXIsIGNvbXBhcmFibGUgdG8gdGhlIHJlZmVyZW5jZSdzCiAgICAgICAgICAgICAgICAgICAgIGByZW5kZXJfa2VybmVsX3VpbnQ4X3ByZWJhc2lzYCBwZXIta2VybmVsIG1lZGlhbi4KICAqIGByZW5kZXIrZDJoYCAgLS0gdGhlIHNhbWUgcGx1cyBhIHJlYWQtYmFjaywgc28gdGhlIGNvc3Qgb2YgZ2V0dGluZyB0aGUKICAgICAgICAgICAgICAgICAgICAgYW5zd2VyIG9mZiB0aGUgZGV2aWNlIGlzIHZpc2libGUgcmF0aGVyIHRoYW4gYXNzdW1lZCBhd2F5LgogICogYHdob2xlIGdyYXBoYCAtLSB0aGUgd2hvbGUgYG9icyA9IHJlbmRlciguLi4pYCBpbmNsdWRpbmcgdGhlIGJhc2lzIGtlcm5lbCwKICAgICAgICAgICAgICAgICAgICAgd2hpY2ggaXMgdGhlIG9iamVjdCBtb3N0IGNvbXBhcmFibGUgdG8gdGhlIHJlZmVyZW5jZSdzCiAgICAgICAgICAgICAgICAgICAgIHdob2xlIHN0ZXAuIEl0IGlzIHN0aWxsIG5vdCBhIHdob2xlIHN0ZXAsIGJlY2F1c2UgdGhlCiAgICAgICAgICAgICAgICAgICAgIHJlZmVyZW5jZSdzIHN0ZXAgYWxzbyBkb2VzIGRlY29kZS9waHlzaWNzL3JheWNhc3QvcmVzZXQsCiAgICAgICAgICAgICAgICAgICAgIGFuZCB0aGF0IGRpZmZlcmVuY2UgaXMgbmFtZWQgcmF0aGVyIHRoYW4gaGlkZGVuLgoKV0hZIE1FTU9SWSBJUyBNRUFTVVJFRCBBTkQgTk9UIERFUklWRUQKLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCkF0IGBtYXhfc3RlcHM9MzJgIHRoZSBnZW5lcmF0ZWQgc2NoZWR1bGUgd3JpdGVzIHRoZSBEREEgc3RhdGUgdG8gZ2xvYmFsIG1lbW9yeQpvbmNlIHBlciBzdGVwIGFuZCB0aGUgZmluYWwga2VybmVsIHRha2VzIGFsbCBvZiB0aG9zZSBidWZmZXJzIGFzIGlucHV0cy4gVGhhdAptYWtlcyB0aGUgcGVhayBmb290cHJpbnQgYSBmdW5jdGlvbiBvZiBgbWF4X3N0ZXBzYCBhbmQgb2YgdGhlIHBpeGVsIGNvdW50LCBhbmQKYXQgdGhlIHByb2plY3QncyBoZWFkbGluZSBzaGFwZSAoYmF0Y2ggMzI3NjgsIDMyeDMyKSB0aGUgYXJpdGhtZXRpYyBzYXlzIHRoZQpncmFwaCBjYW5ub3QgYmUgYWxsb2NhdGVkIGF0IGFsbC4gQW4gYXJpdGhtZXRpYyBjb25zZXF1ZW5jZSBpcyBub3QgYQptZWFzdXJlbWVudCwgc28gdGhlIGZvb3RwcmludCBpcyBtZWFzdXJlZCB3aXRoIHRoZSBhbGxvY2F0b3IncyBvd24gY291bnRlcnMgYW5kCnRoZSBydW4gcmVwb3J0cyB3aGF0IGl0IGFjdHVhbGx5IGFsbG9jYXRlZCwgaW5jbHVkaW5nIHRoZSBzaGFwZXMgd2hlcmUgaXQKZmFpbGVkLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBnYwppbXBvcnQgc3RhdGlzdGljcwppbXBvcnQgdGltZQoKCmRlZiBfc3luYyhkZXYpOgogICAgZnJvbSB0aW55Z3JhZCBpbXBvcnQgRGV2aWNlCiAgICBkID0gRGV2aWNlW2Rldl0KICAgIGlmIGhhc2F0dHIoZCwgInN5bmNocm9uaXplIik6CiAgICAgICAgZC5zeW5jaHJvbml6ZSgpCgoKY2xhc3MgTWVtUHJvYmU6CiAgICAiIiJUcmFjayB3aGF0IHRoZSBhbGxvY2F0b3IgYWN0dWFsbHkgaGFuZHMgb3V0LCBieSB3cmFwcGluZyBgYWxsb2NgL2BmcmVlYC4KCiAgICBUaGlzIHZlcnNpb24gb2YgdGlueWdyYWQgaGFzIG5vIGBtYXhfbWVtb3J5X2FsbG9jYXRlZGAsIHNvIHRoZSBmb290cHJpbnQgaXMKICAgIG1lYXN1cmVkIGJ5IGluc3RydW1lbnRpbmcgdGhlIG9uZSBwbGFjZSBieXRlcyBhcmUgcmVxdWVzdGVkOgogICAgYEFsbG9jYXRvci5hbGxvY2AgLyBgQWxsb2NhdG9yLmZyZWVgIChkZXZpY2UucHkpLiBgYWxsb2NgIHJlY29yZHMgdGhlCiAgICByZXF1ZXN0ZWQgc2l6ZSBhbmQgYWRkcyBpdCB0byBhIGxpdmUgdG90YWw7IGBmcmVlYCBzdWJ0cmFjdHMuIGBwZWFrYCBpcyB0aGUKICAgIGhpZ2gtd2F0ZXIgbWFyayBvZiB0aGF0IHRvdGFsLgoKICAgIFRoaXMgaXMgYSBtZWFzdXJlbWVudCBvZiB0aGUgcmVhbCBhbGxvY2F0b3IsIG5vdCBhbiBlc3RpbWF0ZSBmcm9tIHRoZQogICAgZ2VuZXJhdGVkIHNvdXJjZSwgYW5kIHRoZSB0d28gYXJlIHJlcG9ydGVkIHNlcGFyYXRlbHkgaW4gdGhlIG5vdGVzIGJlY2F1c2UKICAgIHRoZXkgYW5zd2VyIGRpZmZlcmVudCBxdWVzdGlvbnM6IHRoZSBhbGxvY2F0b3IncyBwZWFrIGlzIHdoYXQgdGhlIGNhcmQgaGFkCiAgICB0byBoYXZlLCBhbmQgdGhlIHNvdXJjZSdzIGFyZ3VtZW50IGxpc3QgaXMgd2hhdCB0aGUgc2NoZWR1bGUgc3RydWN0dXJhbGx5CiAgICByZXF1aXJlcy4KCiAgICBgb29tYCBpcyB0aGUgZXhjZXB0aW9uIHRleHQgd2hlbiBhbiBhbGxvY2F0aW9uIGZhaWxzLCB3aGljaCBhdCB0aGUgaGVhZGxpbmUKICAgIHNoYXBlIGlzIHRoZSBmaW5kaW5nIHJhdGhlciB0aGFuIGEgZmFpbHVyZSB0byBoaWRlLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRldjogc3RyKToKICAgICAgICBmcm9tIHRpbnlncmFkIGltcG9ydCBEZXZpY2UKICAgICAgICBmcm9tIHRpbnlncmFkLmRldmljZSBpbXBvcnQgQWxsb2NhdG9yCiAgICAgICAgc2VsZi5kZXYgPSBkZXYKICAgICAgICBzZWxmLmxpdmUgPSAwCiAgICAgICAgc2VsZi5wZWFrID0gMAogICAgICAgIHNlbGYudG90YWwgPSAwCiAgICAgICAgc2VsZi5uX2FsbG9jcyA9IDAKICAgICAgICBzZWxmLm9vbTogc3RyIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLmQgPSBEZXZpY2VbZGV2XQogICAgICAgIHNlbGYuX2EsIHNlbGYuX2YgPSBBbGxvY2F0b3IuYWxsb2MsIEFsbG9jYXRvci5mcmVlCiAgICAgICAgcHJvYmUgPSBzZWxmICAgICAgICAgICAgICAgICAgICAjIHRoZSBjbG9zdXJlcyB0YWtlIHRoZSBhbGxvY2F0b3IgYXMgYGFgCgogICAgICAgIGRlZiBhbGxvYyhhLCBzaXplLCBvcHRpb25zPU5vbmUpOgogICAgICAgICAgICByID0gcHJvYmUuX2EoYSwgc2l6ZSwgb3B0aW9ucykKICAgICAgICAgICAgcHJvYmUubl9hbGxvY3MgKz0gMQogICAgICAgICAgICBwcm9iZS50b3RhbCArPSBpbnQoc2l6ZSkKICAgICAgICAgICAgcHJvYmUubGl2ZSArPSBpbnQoc2l6ZSkKICAgICAgICAgICAgcHJvYmUucGVhayA9IG1heChwcm9iZS5wZWFrLCBwcm9iZS5saXZlKQogICAgICAgICAgICByZXR1cm4gcgoKICAgICAgICBkZWYgZnJlZShhLCBvcGFxdWUsIHNpemUsIG9wdGlvbnM9Tm9uZSk6CiAgICAgICAgICAgIHByb2JlLmxpdmUgLT0gaW50KHNpemUpCiAgICAgICAgICAgIHJldHVybiBwcm9iZS5fZihhLCBvcGFxdWUsIHNpemUsIG9wdGlvbnMpCgogICAgICAgIGRlZiBhbGxvY19vb20oYSwgc2l6ZSwgb3B0aW9ucz1Ob25lKToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcmV0dXJuIGFsbG9jKGEsIHNpemUsIG9wdGlvbnMpCiAgICAgICAgICAgIGV4Y2VwdCBNZW1vcnlFcnJvciBhcyBlOgogICAgICAgICAgICAgICAgcHJvYmUub29tID0gZiJ7ZX0iWzozMDBdCiAgICAgICAgICAgICAgICByYWlzZQoKICAgICAgICBBbGxvY2F0b3IuYWxsb2MsIEFsbG9jYXRvci5mcmVlID0gYWxsb2Nfb29tLCBmcmVlCgogICAgZGVmIHJlc2V0KHNlbGYpOgogICAgICAgIHNlbGYubGl2ZSA9IHNlbGYucGVhayA9IHNlbGYudG90YWwgPSBzZWxmLm5fYWxsb2NzID0gMAogICAgICAgIHNlbGYub29tID0gTm9uZQoKICAgIGRlZiByZXN0b3JlKHNlbGYpOgogICAgICAgIGZyb20gdGlueWdyYWQuZGV2aWNlIGltcG9ydCBBbGxvY2F0b3IKICAgICAgICBBbGxvY2F0b3IuYWxsb2MsIEFsbG9jYXRvci5mcmVlID0gc2VsZi5fYSwgc2VsZi5fZgoKICAgIGRlZiBfX2VudGVyX18oc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgX19leGl0X18oc2VsZiwgKmEpOgogICAgICAgIHNlbGYucmVzdG9yZSgpCiAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgZGVmIGFzX2RpY3Qoc2VsZikgLT4gZGljdDoKICAgICAgICByZXR1cm4geyJwZWFrX2xpdmVfYnl0ZXMiOiBzZWxmLnBlYWssICJ0b3RhbF9hbGxvY2F0ZWRfYnl0ZXMiOiBzZWxmLnRvdGFsLAogICAgICAgICAgICAgICAgIm5fYWxsb2NzIjogc2VsZi5uX2FsbG9jcywgImxpdmVfYXRfZW5kX2J5dGVzIjogc2VsZi5saXZlLAogICAgICAgICAgICAgICAgIm9vbSI6IHNlbGYub29tfQoKCmRlZiB0aW1lX2NhbGwoZm4sIGRldiwgaXRlcnM6IGludCwgd2FybXVwOiBpbnQgPSA1KSAtPiBkaWN0OgogICAgIiIiTWVkaWFuIC8gbWluIC8gbWF4IC8gc3RkIG9mIGBmbigpYCwgaW4gbWlsbGlzZWNvbmRzLCB3YWxsIGNsb2NrLgoKICAgIFRoaXMgaXMgdGhlICplbmQgdG8gZW5kKiBudW1iZXI6IHdoYXRldmVyIGBmbmAgZG9lcywgaG9zdCBzaWRlIGFuZCBkZXZpY2UKICAgIHNpZGUsIHBsdXMgb25lIHN5bmNocm9uaXphdGlvbi4gQm90aCBoYWx2ZXMgYXJlIHJlYWwgY29zdHMgYW5kIG5laXRoZXIgaXMKICAgIGRpdmlkZWQgb3V0LCBiZWNhdXNlIHRoZSBwb2ludCBpcyB3aGF0IGEgY2FsbGVyIHBheXMuIGBkZXZpY2VfdGltZV9tc2AgaXMKICAgIHRoZSBkZXZpY2Utb25seSBjb3VudGVycGFydCwgbWVhc3VyZWQgc2VwYXJhdGVseS4KICAgICIiIgogICAgZm9yIF8gaW4gcmFuZ2Uod2FybXVwKToKICAgICAgICBmbigpCiAgICBfc3luYyhkZXYpCiAgICB0cyA9IFtdCiAgICBmb3IgXyBpbiByYW5nZShpdGVycyk6CiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZm4oKQogICAgICAgIF9zeW5jKGRldikKICAgICAgICB0cy5hcHBlbmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgKiAxZTMpCiAgICB0cy5zb3J0KCkKICAgIHJldHVybiB7CiAgICAgICAgIml0ZXJzIjogaXRlcnMsCiAgICAgICAgIm1lZGlhbl9tcyI6IHN0YXRpc3RpY3MubWVkaWFuKHRzKSwKICAgICAgICAibWluX21zIjogdHNbMF0sCiAgICAgICAgIm1heF9tcyI6IHRzWy0xXSwKICAgICAgICAibWVhbl9tcyI6IHN0YXRpc3RpY3MuZm1lYW4odHMpLAogICAgICAgICJzdGRfbXMiOiBzdGF0aXN0aWNzLnN0ZGV2KHRzKSBpZiBsZW4odHMpID4gMSBlbHNlIDAuMCwKICAgIH0KCgpkZWYgZGV2aWNlX3RpbWVfbXModGVuc29yLCBpdGVyczogaW50ID0gNSkgLT4gZGljdCB8IE5vbmU6CiAgICAiIiJEZXZpY2Utb25seSB0aW1lIGZvciBhIHJlYWxpemVkIHRlbnNvciwgdmlhIHRpbnlncmFkJ3Mgb3duIGB0aW1lX2NhbGxgLgoKICAgIGB0aW55Z3JhZC9lbmdpbmUvcmVhbGl6ZS5weTp0aW1lX2NhbGxgIGJ1aWxkcyBhbiBFeGVjQ29udGV4dCB3aXRoCiAgICBgdXBkYXRlX3N0YXRzPUZhbHNlLCB3YWl0PVRydWVgIGFuZCByZS1leGVjdXRlcyBhIHNpbmdsZSBDQUxMLCByZXR1cm5pbmcKICAgIHRoZSBkZXZpY2UtbWVhc3VyZWQgZHVyYXRpb24gb2YgdGhlIGxhcmdlc3Qga2VybmVsIGluIHRoZSBncm91cC4gVGhhdCBpcwogICAgdGhlIHNhbWUgcXVhbnRpdHkgYGJlbmNoLmN1YCBnZXRzIGZyb20gYGN1ZGFFdmVudGAgYXJvdW5kIGEgbGF1bmNoLCBhbmQgaXQKICAgIGlzIHRoZSByaWdodCBpbnN0cnVtZW50IGhlcmU6IHdhbGwgY2xvY2sgYXJvdW5kIGEgUHl0aG9uLWxldmVsIGNhbGwgaXMKICAgIGRvbWluYXRlZCBieSB0cmFjaW5nLCBhbmQgdHJhY2luZyBpcyBub3QgdGhlIHJlbmRlci4KCiAgICBUd28gY2F2ZWF0cywgYm90aCBjYXJyaWVkIGludG8gdGhlIG5vdGVzIHJhdGhlciB0aGFuIGhpZGRlbjoKCiAgICAgICogaXQgcmUtdXNlcyB0aGUgKmNvbXBpbGVkKiBwcm9ncmFtLCBzbyB0aGlzIG1lYXN1cmVzIHJlLWV4ZWN1dGlvbiwgbm90CiAgICAgICAgdGhlIGZpcnN0IGNhbGwncyBjb21waWxlLiBDb21waWxhdGlvbiBpcyByZXBvcnRlZCBzZXBhcmF0ZWx5ICh0aGUKICAgICAgICBjZW5zdXMncyBgbmV3X2Rpc3RpbmN0X3Byb2dyYW1zYCBhbmQgdGhlIHdhbGwtY2xvY2sgYnVpbGQtb25seSBmaWd1cmUpLgogICAgICAqIGl0IHRha2VzIHRoZSBtYXggb3ZlciB0aGUga2VybmVscyBpbiB0aGUgZ3JvdXAsIHNvIGZvciBhIGdyYXBoIGl0IGlzIGEKICAgICAgICBwZXIta2VybmVsIG51bWJlciwgbm90IGEgd2hvbGUtZ3JhcGggbnVtYmVyLiBUaGUgd2hvbGUtZ3JhcGggZGV2aWNlCiAgICAgICAgdGltZSBpcyB0aGVyZWZvcmUgbWVhc3VyZWQgYXMgYHN1bSBvZiBwZXIta2VybmVsIGRldmljZSB0aW1lc2AgYnkKICAgICAgICB3YWxraW5nIHRoZSBMSU5FQVIsIHdoaWNoIGBsaW5lYXJfZGV2aWNlX3RpbWVfbXNgIGRvZXMuCgogICAgUmV0dXJucyBOb25lIHdoZW4gdGhlIGJhY2tlbmQgZG9lcyBub3Qgc3VwcG9ydCBpdCwgcmF0aGVyIHRoYW4gYSB6ZXJvIHRoYXQKICAgIHdvdWxkIHJlYWQgYXMgImluZmluaXRlbHkgZmFzdCIuCiAgICAiIiIKICAgIGZyb20gdGlueWdyYWQuZW5naW5lLnJlYWxpemUgaW1wb3J0IHRpbWVfY2FsbAogICAgdHJ5OgogICAgICAgIGxpbmVhciwgdmFyX3ZhbHMgPSBfbGluZWFyX2FuZF92YXJzKHRlbnNvcikKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICByZXR1cm4geyJlcnJvciI6IGYie3R5cGUoZSkuX19uYW1lX199OiB7ZX0iWzoyMDBdfQogICAgdHJ5OgogICAgICAgIGJlc3QgPSBOb25lCiAgICAgICAgZm9yIGNhbGwgaW4gbGluZWFyLnNyYzoKICAgICAgICAgICAgdCA9IG5leHQodGltZV9jYWxsKGNhbGwsIHZhcl92YWxzPXZhcl92YWxzKSkKICAgICAgICAgICAgYmVzdCA9IHQgaWYgYmVzdCBpcyBOb25lIGVsc2UgbWF4KGJlc3QsIHQpCiAgICAgICAgcmV0dXJuIHsibWF4X2tlcm5lbF9tcyI6IGJlc3QgKiAxZTMgaWYgYmVzdCBpcyBub3QgTm9uZSBlbHNlIE5vbmV9CiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcmV0dXJuIHsiZXJyb3IiOiBmInt0eXBlKGUpLl9fbmFtZV9ffToge2V9Ils6MjAwXX0KCgpkZWYgX2xpbmVhcl9hbmRfdmFycyh0ZW5zb3IpOgogICAgZnJvbSB0aW55Z3JhZCBpbXBvcnQgVGVuc29yIGFzIF9UCiAgICBsaW4sIHZ2ID0gX1QubGluZWFyX3dpdGhfdmFycyh0ZW5zb3IpCiAgICByZXR1cm4gbGluLCB2dgoKCmRlZiBsaW5lYXJfZGV2aWNlX3RpbWVfbXModGVuc29yLCBpdGVyczogaW50ID0gMykgLT4gZGljdCB8IE5vbmU6CiAgICAiIiJTdW0gb2YgcGVyLWtlcm5lbCBkZXZpY2UgdGltZXMgb3ZlciB0aGUgd2hvbGUgZ3JhcGgsIGluIG1zLgoKICAgIFdhbGtzIHRoZSBjb21waWxlZCBMSU5FQVIgYW5kIHRpbWVzIGVhY2ggQ0FMTCB3aXRoIGB0aW1lX2NhbGxgLCB0aGVuIHN1bXMuCiAgICBUaGlzIGlzIHRoZSB3aG9sZS1ncmFwaCBkZXZpY2UgdGltZTogaXQgaXMgd2hhdCB0aGUgcmVmZXJlbmNlJ3MKICAgIHdob2xlLXN0ZXAgbnVtYmVyIGNvbnRhaW5zLCBvbiB0aGUgc2FtZSBraW5kIG9mIGNsb2NrLCBhbmQgaXQgZXhjbHVkZXMgaG9zdAogICAgdHJhY2luZywgd2hpY2ggdGhlIGZyYW1ld29yayBhZGRzIGFuZCB0aGUgcmVmZXJlbmNlIGRvZXMgbm90LgogICAgIiIiCiAgICBmcm9tIHRpbnlncmFkLmVuZ2luZS5yZWFsaXplIGltcG9ydCB0aW1lX2NhbGwKICAgIHRyeToKICAgICAgICBsaW5lYXIsIHZhcl92YWxzID0gX2xpbmVhcl9hbmRfdmFycyh0ZW5zb3IpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcmV0dXJuIHsiZXJyb3IiOiBmInt0eXBlKGUpLl9fbmFtZV9ffToge2V9Ils6MjAwXX0KICAgIHRyeToKICAgICAgICBjYWxscyA9IGxpc3QobGluZWFyLnNyYykKICAgICAgICBmb3IgXyBpbiByYW5nZSgyKTogICAgICAgICAgICAgICAgICAgICAjIHR3byB3YXJtIHBhc3NlcwogICAgICAgICAgICBmb3IgYyBpbiBjYWxsczoKICAgICAgICAgICAgICAgIG5leHQodGltZV9jYWxsKGMsIHZhcl92YWxzPXZhcl92YWxzKSkKICAgICAgICB0b3QgPSBbXQogICAgICAgIGZvciBfIGluIHJhbmdlKG1heCgxLCBpdGVycykpOgogICAgICAgICAgICBzID0gMC4wCiAgICAgICAgICAgIGZvciBjIGluIGNhbGxzOgogICAgICAgICAgICAgICAgcyArPSBuZXh0KHRpbWVfY2FsbChjLCB2YXJfdmFscz12YXJfdmFscykpCiAgICAgICAgICAgIHRvdC5hcHBlbmQocyAqIDFlMykKICAgICAgICB0b3Quc29ydCgpCiAgICAgICAgcmV0dXJuIHsibl9rZXJuZWxzIjogbGVuKGNhbGxzKSwgIm1lZGlhbl9tcyI6IHN0YXRpc3RpY3MubWVkaWFuKHRvdCksCiAgICAgICAgICAgICAgICAibWluX21zIjogdG90WzBdLCAibWF4X21zIjogdG90Wy0xXSwKICAgICAgICAgICAgICAgICJtZWRpYW5fcGVyX2tlcm5lbF91cyI6IHN0YXRpc3RpY3MubWVkaWFuKHRvdCkgKiAxZTMgLyBtYXgoMSwgbGVuKGNhbGxzKSl9CiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcmV0dXJuIHsiZXJyb3IiOiBmInt0eXBlKGUpLl9fbmFtZV9ffToge2V9Ils6MjAwXX0KCgpkZWYgYmVuY2hfcmVuZGVyKGJ1aWxkX2lucHV0cywgcmVuZGVyX2ZuLCBkZXYsIGl0ZXJzOiBpbnQsIGxhYmVsOiBzdHIsCiAgICAgICAgICAgICAgICAgd2FybXVwOiBpbnQgPSA1KSAtPiBkaWN0OgogICAgIiIiQnVpbGQgb25jZSwgdGltZSB0aGUgcmVhbGl6ZSwgYW5kIHJlcG9ydCB0aGUgYWxsb2NhdG9yJ3MgcGVhay4KCiAgICBgYnVpbGRfaW5wdXRzKClgIG11c3QgcmV0dXJuIHdoYXRldmVyIGByZW5kZXJfZm5gIG5lZWRzLiBJdCBpcyBjYWxsZWQgaW5zaWRlCiAgICB0aGUgbWVtb3J5IHByb2JlLCBzbyB0aGUgcmVwb3J0ZWQgcGVhayBjb3ZlcnMgdGhlIGlucHV0cyAqYW5kKiBldmVyeQogICAgaW50ZXJtZWRpYXRlIHRoZSByZW5kZXIgYWxsb2NhdGVzIC0tIHdoaWNoIGlzIHRoZSByaWdodCBib3VuZGFyeSwgYmVjYXVzZQogICAgdGhlIHJlZmVyZW5jZSdzIHBlci1rZXJuZWwgbWVhc3VyZW1lbnQgYWxzbyBydW5zIGFnYWluc3QgYSBmdWxseSBhbGxvY2F0ZWQKICAgIGVudmlyb25tZW50IChgYmVuY2guY3VgIGFsbG9jYXRlcyB0aGUgd2hvbGUgc3RhdGUgdXAgZnJvbnQgYW5kIHRpbWVzIGEKICAgIHNpbmdsZSBsYXVuY2ggaW5zaWRlIGl0KS4KICAgICIiIgogICAgb3V0OiBkaWN0ID0geyJsYWJlbCI6IGxhYmVsLCAicmVhbGl6ZWQiOiBGYWxzZSwgImVycm9yIjogTm9uZX0KICAgIGdjLmNvbGxlY3QoKQogICAgciA9IE5vbmUKICAgIHdpdGggTWVtUHJvYmUoZGV2KSBhcyBtcDoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGlucHV0cyA9IGJ1aWxkX2lucHV0cygpCiAgICAgICAgICAgIG91dFsibWVtX2FmdGVyX2lucHV0cyJdID0gbXAuYXNfZGljdCgpCiAgICAgICAgICAgIG1wLnJlc2V0KCkKICAgICAgICAgICAgciA9IHJlbmRlcl9mbigqaW5wdXRzKSBpZiBpc2luc3RhbmNlKGlucHV0cywgdHVwbGUpIGVsc2UgcmVuZGVyX2ZuKGlucHV0cykKICAgICAgICAgICAgciA9IHJbMF0gaWYgaXNpbnN0YW5jZShyLCB0dXBsZSkgZWxzZSByCiAgICAgICAgICAgICMgZGV2aWNlIHRpbWluZyBGSVJTVCwgd2hpbGUgdGhlIGdyYXBoIGlzIHN0aWxsIHVucmVhbGl6ZWQ6CiAgICAgICAgICAgICMgYGxpbmVhcl93aXRoX3ZhcnNgIG9uIGEgcmVhbGl6ZWQgdGVuc29yIHJldHVybnMgYW4gZW1wdHkgTElORUFSLAogICAgICAgICAgICAjIGJlY2F1c2UgdGhlcmUgaXMgbm90aGluZyBsZWZ0IHRvIHJ1bi4KICAgICAgICAgICAgb3V0WyJkZXZpY2VfdGltZSJdID0gbGluZWFyX2RldmljZV90aW1lX21zKAogICAgICAgICAgICAgICAgciwgaXRlcnM9bWluKDUsIG1heCgyLCBpdGVycyAvLyAyKSkpCiAgICAgICAgICAgIHIucmVhbGl6ZSgpCiAgICAgICAgICAgIG91dFsicmVhbGl6ZWQiXSA9IFRydWUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAgICAgICAgICAgICAgICAgIyBPT00gaXMgYSByZXN1bHQsIG5vdCBhIGNyYXNoCiAgICAgICAgICAgIG91dFsiZXJyb3IiXSA9IGYie3R5cGUoZSkuX19uYW1lX199OiB7c3RyKGUpfSJbOjQwMF0KICAgICAgICBfc3luYyhkZXYpCiAgICAgICAgb3V0WyJtZW0iXSA9IG1wLmFzX2RpY3QoKQoKICAgIGlmIG5vdCBvdXRbInJlYWxpemVkIl06CiAgICAgICAgcmV0dXJuIG91dAoKICAgIG91dFsib2JzX2J5dGVzIl0gPSBpbnQoci5uYnl0ZXMoKSkKICAgIG91dFsib2JzX3NoYXBlIl0gPSBsaXN0KHIuc2hhcGUpCiAgICBvdXRbIm9ic19kdHlwZSJdID0gc3RyKHIuZHR5cGUpCiAgICByZXR1cm4gb3V0CgoKZGVmIGQyaF90aW1lKHIsIGRldiwgaXRlcnM6IGludCwgd2FybXVwOiBpbnQgPSAzKSAtPiBkaWN0OgogICAgIiIiVGhlIGNvc3Qgb2YgcmVhZGluZyB0aGUgb2JzZXJ2YXRpb24gYmFjaywgYXMgaXRzIG93biBudW1iZXIuIiIiCiAgICByZXR1cm4gdGltZV9jYWxsKGxhbWJkYTogci5udW1weSgpLCBkZXYsIGl0ZXJzLCB3YXJtdXApCg==", "7ec6927c711e0eec2b4de27d8eb03bb6c2f7e0d660edd8dfa969dba61434c539"], ["minecu_tg/battery.py", "..."], ["minecu_tg/capture_cuda.py", "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJjYXB0dXJlX2N1ZGEucHkgLS0gdGhlIGdlbmVyYXRlZCBDVURBIEMgZm9yIHRoaXMgcmVuZGVyLCBvbiBhIGhvc3Qgd2l0aCBubyBHUFUuCgpXSFkgVEhFIE5VTEwgREVWSUNFLiBUaGUgQ1VEQSAqcmVuZGVyZXIqIGlzIGEgcHVyZSBmdW5jdGlvbiBvZiB0aGUgVU9wIGdyYXBoOwppdCBkb2VzIG5vdCBuZWVkIGEgZGV2aWNlLiBgREVWPU5VTEw6Q1VEQTpzbV83NWAgYXR0YWNoZXMgdGhlIENVREEgcmVuZGVyZXIgdG8KdGlueWdyYWQncyBuby1vcCBkZXZpY2UsIHNvIHRoZSBzY2hlZHVsZSwgdGhlIGZ1c2lvbiwgdGhlIHNjaGVkdWxpbmcgYW5kIHRoZQplbWl0dGVkIEMgYXJlIHRoZSByZWFsIG9uZXMgYW5kIG5vdGhpbmcgaXMgZXhlY3V0ZWQuIGB1c2VfbnZjY2AgaXMgZm9yY2VkIG9uCmJlY2F1c2UgdGhlIGRlZmF1bHQgTlZSVEMgcGF0aCBkbG9wZW4oKXMgbGlibnZydGMsIHdoaWNoIGRvZXMgbm90IGV4aXN0IGhlcmUuCgpUaGlzIGlzIGEgRElGRkVSRU5UIGRldmljZSBzdHJpbmcgZnJvbSBgQ1VEQWAsIHNvIGFueXRoaW5nIG1lYXN1cmVkIHRocm91Z2ggaXQKaXMgbGFiZWxsZWQgYXMgc3VjaC4gVGhlIFQ0IHJ1biBtZWFzdXJlcyB0aGUgc2FtZSBudW1iZXJzIGZyb20gdGhlIHJlYWwgYmFja2VuZAotLSB0aGF0IGlzIHdoYXQgYG1pbmVjdV90Zy9rYWdnbGUvYCBkb2VzIC0tIGFuZCB0aGUgbm90ZXMgcXVvdGUgdGhlIFQ0IG51bWJlcnMKd2hlcmUgdGhleSBleGlzdCBhbmQgdGhlc2Ugb25seSBmb3IgdGhlIHNvdXJjZSB0ZXh0LgoKICAgIHB5dGhvbjMgbWluZWN1X3RnL2NhcHR1cmVfY3VkYS5weSAtLW91dGRpciBvdXQvdGctY29kZWdlbi1zbTc1IC0tc3RlcHMgMSw0LDMyCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJQQVJBTExFTCIsICIwIikKb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJERUJVRyIsICIwIikKIyB0aGUgTlVMTCBhbGxvY2F0b3IgcmVmdXNlcyBjb3B5b3V0OyB0aGlzIHJ1biBuZXZlciByZWFkcyByZXN1bHRzIGJhY2ssIGFuZCB0aGUKIyBmbGFnIG9ubHkga2VlcHMgYSBzdHJheSAubnVtcHkoKSBmcm9tIGFib3J0aW5nIHRoZSBjZW5zdXMKb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJOVUxMX0FMTE9XX0NPUFlPVVQiLCAiMSIpCgpIRVJFID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApzeXMucGF0aC5pbnNlcnQoMCwgc3RyKEhFUkUucGFyZW50KSkKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1hcmNoIiwgZGVmYXVsdD0ic21fNzUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoIiwgdHlwZT1pbnQsIGRlZmF1bHQ9OCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS13b3JsZCIsIHR5cGU9aW50LCBkZWZhdWx0PTE2KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTMyKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXN0ZXBzIiwgZGVmYXVsdD0iMSw0LDgsMzIiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dGRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICAjIHRpbnlncmFkJ3MgdGVuc29yLWNvcmUgdGFibGUgZG9lcyBpbnQoYXJjaFszOl0pIGFuZCBjYW5ub3QgcGFyc2UgYW4KICAgICMgYXJjaGl0ZWN0dXJlLXNwZWNpZmljIHN1ZmZpeCwgc28gaGFuZCBpdCB0aGUgYmFzZSBhcmNoIGFuZCBsZXQgbnZjYyBkbyB0aGUKICAgICMgcmVhbCBjb21waWxlIHdpdGggdGhlIGZ1bGwgbmFtZS4gU2FtZSB0YWJsZSBmb3IgYm90aC4KICAgIGRldl9hcmNoID0gcmUuc3ViKHIiKHNtX1xkKylbYS16XSQiLCByIlwxIiwgYS5hcmNoKQogICAgb3MuZW52aXJvblsiREVWIl0gPSBmIk5VTEw6Q1VEQTp7ZGV2X2FyY2h9IgoKICAgIGZyb20gdGlueWdyYWQucmVuZGVyZXIgaW1wb3J0IGNzdHlsZQogICAgX29yaWcgPSBjc3R5bGUuQ1VEQVJlbmRlcmVyLl9faW5pdF9fCgogICAgZGVmIHBhdGNoZWQoc2VsZiwgdGFyZ2V0LCB1c2VfbnZjYz1GYWxzZSk6CiAgICAgICAgcmV0dXJuIF9vcmlnKHNlbGYsIHRhcmdldCwgdXNlX252Y2M9VHJ1ZSkKCiAgICBjc3R5bGUuQ1VEQVJlbmRlcmVyLl9faW5pdF9fID0gcGF0Y2hlZAoKICAgICMgVGhpcyBob3N0IGhhcyBubyBudmNjLCBhbmQgdGhlIE5VTEwgcHJvZ3JhbSBuZXZlciBleGVjdXRlcyB0aGUgYmluYXJ5LCBzbwogICAgIyB0aGUgY29tcGlsZSBpcyByZXBsYWNlZCBieSBhIGNhcHR1cmUuIFRoZSBTT1VSQ0UgLS0gd2hpY2ggaXMgd2hhdCB0aGlzCiAgICAjIHNjcmlwdCBleGlzdHMgZm9yIC0tIGlzIGZpbmlzaGVkIGJlZm9yZSBjb21waWxlX2NhY2hlZCBpcyBjYWxsZWQsIHNvIHRoZQogICAgIyBnZW5lcmF0ZWQgQyBpcyB0aGUgcmVhbCBvbmUuIEFueSBjdWJpbiBmb3IgdGhpcyBjYXB0dXJlIGhhcyB0byBjb21lIGZyb20KICAgICMgdGhlIFQ0IHJ1biBvciB0aGUgb2ZmbGluZSBhcmNobGFiIHJvdXRlOyB0aGVyZSBpcyBub25lIGhlcmUsIGFuZCB0aGUgbm90ZXMKICAgICMgZG8gbm90IHF1b3RlIFNBU1MgZnJvbSB0aGlzIHBhdGguCiAgICAjCiAgICAjIEluc3RhbGxlZCBoZXJlLCBiZWZvcmUgYW55IHRpbnlncmFkIGltcG9ydCB0aGF0IGNhbiBidWlsZCBhIGRldmljZSwKICAgICMgYmVjYXVzZSB0aGUgcmVuZGVyZXIgaXMgY29uc3RydWN0ZWQgb24gZmlyc3QgdXNlIGFuZCBjb21waWxlcyB0aGVuLgogICAgZnJvbSB0aW55Z3JhZC5kZXZpY2UgaW1wb3J0IENvbXBpbGVyCiAgICBBTExfU1JDOiBsaXN0W3N0cl0gPSBbXQogICAgX29yaWdfY2MgPSBDb21waWxlci5jb21waWxlX2NhY2hlZAoKICAgIGRlZiBfY2FwdHVyZV9vbmx5KHNlbGYsIHNyYywgKmEsICoqayk6CiAgICAgICAgQUxMX1NSQy5hcHBlbmQoc3RyKHNyYykpCiAgICAgICAgcmV0dXJuIGIiIgoKICAgIENvbXBpbGVyLmNvbXBpbGVfY2FjaGVkID0gX2NhcHR1cmVfb25seQoKICAgIGltcG9ydCBudW1weSBhcyBucAogICAgZnJvbSB0aW55Z3JhZCBpbXBvcnQgVGVuc29yLCBkdHlwZXMsIERldmljZSwgR2xvYmFsQ291bnRlcnMKCiAgICBmcm9tIG1pbmVjdV90ZyBpbXBvcnQgY29kZWdlbgogICAgZnJvbSBtaW5lY3VfdGcucmVuZGVyIGltcG9ydCBjYW1lcmFfYmFzaXMsIHJlbmRlcgoKICAgICMgVGhlIHdvcmxkIGlzIGJ1aWx0IGJ5IGBtaW5lY3VfdGcuZ2VuZXJhdGVfd29ybGRgIG9uIHRoZSBDUFUgaW4gYSBjaGlsZAogICAgIyBwcm9jZXNzLCBiZWNhdXNlIHRoZSBOVUxMIGRldmljZSBjYW5ub3QgY29weSByZXN1bHRzIG91dCAob3BzX251bGwucHk6MjgKICAgICMgcmFpc2VzICJubyBjb3B5b3V0IG9uIE5VTEwiKSwgYW5kIHRoZSBpbnB1dHMgaGF2ZSB0byBiZSByZWFsIGJ5dGVzIGJlZm9yZQogICAgIyB0aGV5IGFyZSBoYW5kZWQgdG8gYSBkZXZpY2UgdGhhdCBjYW5ub3QgZXhlY3V0ZS4gQSBzdWJwcm9jZXNzIHJhdGhlciB0aGFuCiAgICAjIGFuIGluLXByb2Nlc3MgREVWIHN3aXRjaCwgYmVjYXVzZSB0aW55Z3JhZCByZXNvbHZlcyBEZXZpY2UuREVGQVVMVCBvbmNlCiAgICAjIGFuZCBjYWNoZXMgaXQuCiAgICBpbXBvcnQgc3VicHJvY2VzcwogICAgY2FjaGUgPSBhLm91dGRpciAvICJpbnB1dHMubnB6IgogICAgYS5vdXRkaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgaWYgbm90IGNhY2hlLmV4aXN0cygpOgogICAgICAgIGNoaWxkID0gKAogICAgICAgICAgICAiaW1wb3J0IHN5czsgc3lzLnBhdGguaW5zZXJ0KDAsICVyKVxuIgogICAgICAgICAgICAiaW1wb3J0IG9zOyBvcy5lbnZpcm9uWydERVYnXT0nQ1BVJ1xuIgogICAgICAgICAgICAiaW1wb3J0IG51bXB5IGFzIG5wXG4iCiAgICAgICAgICAgICJmcm9tIG1pbmVjdV90ZyBpbXBvcnQgZ2VuZXJhdGVfd29ybGRcbiIKICAgICAgICAgICAgIncsIGcsIEIgPSAlZCwgJWQsICVkXG4iCiAgICAgICAgICAgICJ2b3ggPSBucC5hc2NvbnRpZ3VvdXNhcnJheShucC5zdGFjayhbZ2VuZXJhdGVfd29ybGQodywgZykubnVtcHkoKV0qQikpXG4iCiAgICAgICAgICAgICJjYW1zID0gbnAuYXJyYXkoW1t3LzIuMCwgZmxvYXQoZyksIHcvMi4wLCAwLjAwMSooaSUlMTMtNiksIDAuMF0iCiAgICAgICAgICAgICIgZm9yIGkgaW4gcmFuZ2UoQildLCBkdHlwZT1ucC5mbG9hdDMyKVxuIgogICAgICAgICAgICAibnAuc2F2ZXooJXIsIHZveD12b3gsIGNhbXM9Y2FtcylcbiIKICAgICAgICApICUgKHN0cihIRVJFLnBhcmVudCksIGEud29ybGQsIGEud29ybGQgLy8gNCwgYS5iYXRjaCwgc3RyKGNhY2hlKSkKICAgICAgICBzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsICItYyIsIGNoaWxkXSwgY2hlY2s9VHJ1ZSkKICAgIHogPSBucC5sb2FkKGNhY2hlKQogICAgdm94X25wLCBjYW1zID0gelsidm94Il0sIHpbImNhbXMiXQogICAgdm94ID0gVGVuc29yKHZveF9ucCwgZHR5cGU9ZHR5cGVzLmludDgpCiAgICBiYXNpcyA9IGNhbWVyYV9iYXNpcyhUZW5zb3IoY2FtcywgZHR5cGU9ZHR5cGVzLmZsb2F0MzIpLCA3MC4wLCBhLnJlcywgYS5yZXMpLmNvbnRpZ3VvdXMoKQogICAgYmFzaXMucmVhbGl6ZSgpICAgICAgICAjIHJlYWxpemUsIG5ldmVyIHJlYWQgYmFjazogTlVMTCBjYW5ub3QgY29weSBvdXQKCiAgICBzdGVwcyA9IFtpbnQoeCkgZm9yIHggaW4gYS5zdGVwcy5zcGxpdCgiLCIpXQogICAgYyA9IGNvZGVnZW4uY2Vuc3VzKGxhbWJkYSBtczogcmVuZGVyKHZveCwgYmFzaXMsIGEucmVzLCBhLnJlcywgbXMpLAogICAgICAgICAgICAgICAgICAgICAgIHN0ZXBzLCBhLm91dGRpciwgY2FwdHVyZV9vbmx5PVRydWUpCiAgICBjWyJhcmNoIl0gPSBhLmFyY2gKICAgIGNbImRldmljZV9ub3RlIl0gPSAiTlVMTCBkZXZpY2UsIENVREEgcmVuZGVyZXIgYXR0YWNoZWQsIG5vIGhhcmR3YXJlIgoKICAgIGltcG9ydCBqc29uCiAgICAoYS5vdXRkaXIgLyAiY2Vuc3VzLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoYywgaW5kZW50PTIpKQogICAgKGEub3V0ZGlyIC8gImFsbF9zb3VyY2VzLnR4dCIpLndyaXRlX3RleHQoCiAgICAgICAgIlxuXG4vKiA9PT09PSBuZXh0IGtlcm5lbCA9PT09PSAqL1xuXG4iLmpvaW4oQUxMX1NSQykpCiAgICBwcmludChmImNhcHR1cmVkIHtsZW4oQUxMX1NSQyl9IGtlcm5lbCBzb3VyY2VzIGluIHRvdGFsICIKICAgICAgICAgIGYiKHtzdW0obGVuKHMpIGZvciBzIGluIEFMTF9TUkMpfSBieXRlcykgLT4ge2Eub3V0ZGlyfS9hbGxfc291cmNlcy50eHQiKQogICAgcHJpbnQoKQogICAgcHJpbnQoY29kZWdlbi5zdW1tYXJpc2UoYykpCiAgICBwcmludCgpCiAgICBmb3IgciBpbiBjWyJyb3dzIl06CiAgICAgICAgY291bnRzID0gclsibGF1bmNoX2NvdW50c19ieV9wcm9ncmFtIl0KICAgICAgICB0b3AgPSBtYXgoY291bnRzLml0ZW1zKCksIGtleT1sYW1iZGEga3Y6IGt2WzFdKQogICAgICAgIHByaW50KGYiICBtcz17clsnbWF4X3N0ZXBzJ106M2R9ICB7clsna2VybmVsX2xhdW5jaGVzJ106NGR9IGxhdW5jaGVzLCAiCiAgICAgICAgICAgICAgZiJ7clsnZGlzdGluY3RfcHJvZ3JhbXNfbGF1bmNoZWQnXTozZH0gZGlzdGluY3QgcHJvZ3JhbXMsICIKICAgICAgICAgICAgICBmIm1vc3QtcmVwZWF0ZWQ6IHt0b3BbMV06M2R9IHgge3RvcFswXX0iKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgc3lzLmV4aXQobWFpbigpKQo=", "f1957fb652054f63b93e9bbf0d5629391504f7bc2116eef1cc4192f874a97d0b"], ["minecu_tg/analyse_codegen.py", "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJhbmFseXNlX2NvZGVnZW4ucHkgLS0gdHVybiBhIGNhcHR1cmVkIHNjaGVkdWxlIGludG8gYSBwZXIta2VybmVsIHRhYmxlLgoKUmVhZHMgdGhlIGBtYXhfc3RlcHNfKi9rZXJuZWxOTk4uY3VgIGRpcmVjdG9yaWVzIHRoYXQgYGNhcHR1cmVfY3VkYS5weWAgd3JvdGUgYW5kCnJlcG9ydHMsIGZvciBlYWNoIGNhcHR1cmVkIHByb2dyYW06IGl0cyBhcmd1bWVudCBsaXN0LCBpdHMgc2l6ZSwgaG93IG1hbnkgRERBCnN0ZXBzIGl0cyBib2R5IGNvbnRhaW5zLCBhbmQgaG93IG1hbnkgZ2xvYmFsIHN0YXRlIGJ1ZmZlcnMgaXQgdG91Y2hlcy4KClRoZSBzdGVwIGNvdW50IGluc2lkZSBhIHByb2dyYW0gaXMgY291bnRlZCBmcm9tIHRoZSBnZW5lcmF0ZWQgQ1VEQSBpdHNlbGYsIGJ5CnRoZSBzaGFwZSBvZiBvbmUgRERBIGl0ZXJhdGlvbiBpbiB0aGUgZW1pdHRlZCBjb2RlLCBub3QgYnkgZ3Vlc3NpbmcuIEEgc3RlcCBpcwpzaXggbWFza2VkIHNlbGVjdHMgKHRocmVlIGB0X21heGAgYWNjdW11bGF0aW9ucyBhbmQgdGhyZWUgdm94ZWwtY29vcmRpbmF0ZQp1cGRhdGVzKSwgYW5kIHRoZSByZWZlcmVuY2UncyBvd24gbWFjcm8gaGFzIGV4YWN0bHkgdGhvc2Ugc2l4CihrZXJuZWxzLmN1OjY3Mi02NzgpLiBDb3VudGluZyBzaXggcGVyIHN0ZXAgYW5kIGNoZWNraW5nIHRoZSB0b3RhbCBhZ2FpbnN0IHRoZQpsaW5lIGNvdW50IGlzIHRoZSBjcm9zcy1jaGVjayB0aGF0IHRoZSByZWdleCBmb3VuZCB3aGF0IGl0IGNsYWltcyB0by4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgcmUKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCiMgYSBtYXNrZWQgYWNjdW11bGF0ZTogIHggPSAoY29uZCA/IChhICsgYikgOiBhKQpNQVNLRURfQUNDID0gcmUuY29tcGlsZShyImFsdVxkKyA9IFwoYWx1XGQrXD9cKGFsdVxkK1wrYWx1XGQrXCkiKQojIGEgdm94ZWwgcmVhZDogKig8c3RhdGUgYnVmZmVyPiArICguLi4pKQpWT1hFTF9MRCA9IHJlLmNvbXBpbGUociI9IFwoXCpcKGRhdGFcZCtfXGQrXCtcKCIpCiMgYSAzLXdheSBEREEgY29tcGFyZTogKGEgPCBiKSAmIChhIDwgYykKRERBX0NNUCA9IHJlLmNvbXBpbGUociJhbHVcZCsgPSBcKFwoYWx1XGQrPGFsdVxkK1wpJlwoYWx1XGQrPGFsdVxkK1wpXCkiKQoKS0VSTkVMX1NJRyA9IHJlLmNvbXBpbGUociJfX2dsb2JhbF9fIHZvaWQgX19sYXVuY2hfYm91bmRzX19cKChcZCspXClccysoXHcrKVwoIikKCgpkZWYgYW5hbHlzZShwYXRoOiBQYXRoKSAtPiBkaWN0OgogICAgc3JjID0gcGF0aC5yZWFkX3RleHQoKQogICAgbSA9IEtFUk5FTF9TSUcuc2VhcmNoKHNyYykKICAgICMgdGhlIGFyZ3VtZW50IGxpc3QgaXMgaW5zaWRlIHRoZSBzaWduYXR1cmUgbGluZSwgd2hpY2ggY2FuIGJlIHZlcnkgbG9uZwogICAgc2lnX2xpbmUgPSBuZXh0KChsIGZvciBsIGluIHNyYy5zcGxpdGxpbmVzKCkgaWYgIl9fZ2xvYmFsX18gdm9pZCIgaW4gbCksICIiKQogICAgYXJncyA9IHJlLmZpbmRhbGwociIoPzpzaWduZWQgY2hhcnxmbG9hdHxpbnR8Ym9vbHx1aW50KVxzKlwqXHMqKGRhdGFcZCtfXGQrKSIsIHNpZ19saW5lKQogICAgbWFza2VkID0gbGVuKE1BU0tFRF9BQ0MuZmluZGFsbChzcmMpKQogICAgcmV0dXJuIHsKICAgICAgICAiZmlsZSI6IHBhdGgubmFtZSwKICAgICAgICAibGF1bmNoX2JvdW5kcyI6IGludChtLmdyb3VwKDEpKSBpZiBtIGVsc2UgTm9uZSwKICAgICAgICAibmFtZSI6IG0uZ3JvdXAoMikgaWYgbSBlbHNlIE5vbmUsCiAgICAgICAgIm5fYXJncyI6IGxlbihhcmdzKSwKICAgICAgICAic3RhdGVfYnVmZmVycyI6IGxlbihbYSBmb3IgYSBpbiBhcmdzIGlmIGludChhLnNwbGl0KCJfIilbMV0pID4gMTAyNF0pLAogICAgICAgICJsaW5lcyI6IGxlbihzcmMuc3BsaXRsaW5lcygpKSwKICAgICAgICAiYnl0ZXMiOiBsZW4oc3JjKSwKICAgICAgICAibWFza2VkX2FjY3VtdWxhdGVzIjogbWFza2VkLAogICAgICAgICJkZGFfc3RlcHNfZXN0Ijogcm91bmQobWFza2VkIC8gNi4wLCAyKSwKICAgICAgICAiZGRhX3RocmVlX3dheV9jb21wYXJlcyI6IGxlbihEREFfQ01QLmZpbmRhbGwoc3JjKSksCiAgICAgICAgInZveGVsX3JlYWRzIjogbGVuKFZPWEVMX0xELmZpbmRhbGwoc3JjKSksCiAgICB9CgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIpCiAgICBhcC5hZGRfYXJndW1lbnQoInJvb3QiLCB0eXBlPVBhdGgsIGhlbHA9InRoZSAtLW91dGRpciBnaXZlbiB0byBjYXB0dXJlX2N1ZGEucHkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWpzb24iLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYSA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGNlbnN1cyA9IGpzb24ubG9hZHMoKGEucm9vdCAvICJjZW5zdXMuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgb3V0ID0geyJkZXZpY2UiOiBjZW5zdXNbImRldmljZSJdLCAiYXJjaCI6IGNlbnN1cy5nZXQoImFyY2giKSwgInNoYXBlcyI6IHt9fQogICAgZm9yIHIgaW4gY2Vuc3VzWyJyb3dzIl06CiAgICAgICAgZCA9IGEucm9vdCAvIGYibWF4X3N0ZXBzX3tyWydtYXhfc3RlcHMnXTowNGR9IgogICAgICAgIGlmIG5vdCBkLmlzX2RpcigpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJvd3MgPSBbYW5hbHlzZShwKSBmb3IgcCBpbiBzb3J0ZWQoZC5nbG9iKCJrZXJuZWwqLmN1IikpXQogICAgICAgIG91dFsic2hhcGVzIl1bc3RyKHJbIm1heF9zdGVwcyJdKV0gPSB7CiAgICAgICAgICAgICJrZXJuZWxfbGF1bmNoZXMiOiByWyJrZXJuZWxfbGF1bmNoZXMiXSwKICAgICAgICAgICAgImRpc3RpbmN0X3Byb2dyYW1zX2xhdW5jaGVkIjogclsiZGlzdGluY3RfcHJvZ3JhbXNfbGF1bmNoZWQiXSwKICAgICAgICAgICAgIm5ld19kaXN0aW5jdF9wcm9ncmFtcyI6IHJbIm5ld19kaXN0aW5jdF9wcm9ncmFtcyJdLAogICAgICAgICAgICAicHJvZ3JhbXMiOiByb3dzLAogICAgICAgIH0KCiAgICBpZiBhLmpzb246CiAgICAgICAgcHJpbnQoanNvbi5kdW1wcyhvdXQsIGluZGVudD0yKSkKICAgICAgICByZXR1cm4gMAoKICAgIHByaW50KGYiZGV2aWNlPXtvdXRbJ2RldmljZSddfSBhcmNoPXtvdXRbJ2FyY2gnXX0iKQogICAgZm9yIG1zLCBzIGluIG91dFsic2hhcGVzIl0uaXRlbXMoKToKICAgICAgICBwcmludCgpCiAgICAgICAgcHJpbnQoZiI9PT0gbWF4X3N0ZXBzPXttc306IHtzWydrZXJuZWxfbGF1bmNoZXMnXX0gbGF1bmNoZXMsICIKICAgICAgICAgICAgICBmIntzWydkaXN0aW5jdF9wcm9ncmFtc19sYXVuY2hlZCddfSBkaXN0aW5jdCBwcm9ncmFtcyA9PT0iKQogICAgICAgIHByaW50KGYieydmaWxlJzo+MTN9IHsnbmFtZSc6PjEyfSB7J2xkJzo+NH0geydhcmdzJzo+NX0geydzdGF0ZSc6PjZ9ICIKICAgICAgICAgICAgICBmInsnbGluZXMnOj43fSB7J3N0ZXBzfic6Pjd9IHsnY21wMyc6PjV9IHsnbGR2b3gnOj42fSIpCiAgICAgICAgZm9yIHAgaW4gc1sicHJvZ3JhbXMiXToKICAgICAgICAgICAgbm0gPSAocFsibmFtZSJdIG9yICIiKVs6MTJdCiAgICAgICAgICAgIHByaW50KGYie3BbJ2ZpbGUnXTo+MTN9IHtubTo+MTJ9IHtzdHIocFsnbGF1bmNoX2JvdW5kcyddKTo+NH0gIgogICAgICAgICAgICAgICAgICBmIntwWyduX2FyZ3MnXTo+NX0ge3BbJ3N0YXRlX2J1ZmZlcnMnXTo+Nn0ge3BbJ2xpbmVzJ106Pjd9ICIKICAgICAgICAgICAgICAgICAgZiJ7cFsnZGRhX3N0ZXBzX2VzdCddOj43fSB7cFsnZGRhX3RocmVlX3dheV9jb21wYXJlcyddOj41fSAiCiAgICAgICAgICAgICAgICAgIGYie3BbJ3ZveGVsX3JlYWRzJ106PjZ9IikKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK", "0972d45d060a1f6426c1bc6376d59788d145a5563c0cc0c66e8931ddc7382ec7"], ["minecu_tg/sass_matrix.py", "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJzYXNzX21hdHJpeC5weSAtLSB0aGUgaW5zdHJ1Y3Rpb24gbWF0cml4IGZvciBib3RoIHNpZGVzIG9mIHRoZSBjb21wYXJpc29uLgoKT25lIHRhYmxlLCBvbmUgY2F0ZWdvcmlzZXIsIGJvdGggc2lkZXMsIHNvIHRoZSBjb2x1bW5zIG1lYW4gdGhlIHNhbWUgdGhpbmcgaW4KZXZlcnkgcm93LiBUaGUgY2F0ZWdvcmlzZXIgaXMgYHRvb2xzL2FyY2hsYWIvaW5zcGVjdF9wdHgucHk6Q0FURUdPUklFU2AsCmltcG9ydGVkIHJhdGhlciB0aGFuIHJlaW1wbGVtZW50ZWQsIGZvciB0aGUgcmVhc29uIHRoYXQgYSBzZWNvbmQgZGVmaW5pdGlvbiBvZgoiYSB2ZWN0b3Jpc2VkIGxvYWQiIGlzIGhvdyBhIHN0dWR5IGVuZHMgdXAgY29tcGFyaW5nIHR3byBkaWZmZXJlbnQgcXVhbnRpdGllcy4KCklOUFVUUyBhcmUgYGN1b2JqZHVtcCAtc2Fzc2Agb3V0cHV0cyAodGV4dCkgYW5kIGBjdW9iamR1bXAgLXJlcy11c2FnZWAKb3V0cHV0cywgZnJvbSBjdWJpbnMgYnVpbHQgZm9yIG9uZSBhcmNoaXRlY3R1cmUuIEJvdGggc2lkZXMgb2YgdGhpcyBzdHVkeSBhcmUKc21fNzUsIHdoaWNoIGBub3Rlcy8xN2Agc2VjdGlvbiA2IGVzdGFibGlzaGVkIGlzIHJlYWNoYWJsZTogdGhlIGhhbmQtd3JpdHRlbgpga2VybmVscy5jdWAgY29tcGlsZXMgZm9yIHNtXzc1IHdpdGggemVybyBzcGlsbHMgYW5kIG5vIHNtXzgwKyBpbnN0cnVjdGlvbiwgc28KYSBUNCBpcyBhIHZhbGlkIGNhcmQgZm9yIEJPVEggc2lkZXMgYW5kIHRoZSBjb21wYXJpc29uIGlzIG5vdCBvbmUtc2lkZWQuCgpXSEFUIElTIFJFUE9SVEVEIFBFUiBLRVJORUwKICAgIGluc3RydWN0aW9ucyAgICAgICAgc3RhdGljIGNvdW50LCB0aGUgbnVtYmVyIGEgcmVhZGVyIGNhbiBjaGVjayBieSBjb3VudGluZwogICAgcGVyLWtpbmQgICAgICAgICAgICB0aGUgY2F0ZWdvcnkgY2Vuc3VzIChMREcsIFNURywgRkZNQSwgYmFycmllcnMsIC4uLikKICAgIHJlZ2lzdGVycyAgICAgICAgICAgZnJvbSBjdW9iamR1bXAgLXJlcy11c2FnZSBSRUcKICAgIGxvY2FsIGJ5dGVzICAgICAgICAgdGhlIFNQSUxMIGNvbHVtbi4gQW55IG5vbi16ZXJvIHZhbHVlIGhlcmUgaXMgdGhlIHNpbmdsZQogICAgICAgICAgICAgICAgICAgICAgICBtb3N0IGltcG9ydGFudCBudW1iZXIgaW4gdGhlIHRhYmxlOiBpdCBtZWFucyB0aGUKICAgICAgICAgICAgICAgICAgICAgICAgcmVnaXN0ZXIgZmlsZSBjb3VsZCBub3QgaG9sZCB0aGUgbGl2ZSBzZXQgYW5kIHRoZQogICAgICAgICAgICAgICAgICAgICAgICBrZXJuZWwgaXMgcGF5aW5nIG1lbW9yeSB0cmFmZmljIGZvciBpdHMgb3duIGFyaXRobWV0aWMuCiAgICBzaGFyZWQgYnl0ZXMgICAgICAgIHN0YXRpYyBzaGFyZWQgbWVtb3J5CiAgICBiYXJyaWVycyAgICAgICAgICAgIEJBUi5TWU5DIGNvdW50OyB0aGUgcmVmZXJlbmNlIGhhcyBleGFjdGx5IG9uZSwgaW4gdGhlCiAgICAgICAgICAgICAgICAgICAgICAgIGhvdCBrZXJuZWwsIGZvciB0aGUgY2FtZXJhIGJhc2lzIGluIHNoYXJlZCBtZW1vcnkKICAgIGxhdW5jaCBib3VuZHMgICAgICAgX19sYXVuY2hfYm91bmRzX18gaW4gdGhlIHNvdXJjZSwgaS5lLiB0aGUgcmVnaXN0ZXIKICAgICAgICAgICAgICAgICAgICAgICAgYnVkZ2V0IHRoZSBwcm9ncmFtbWVyIGFza2VkIGZvcgoKVGhlIGhvdC1rZXJuZWwgcm93IGlzIHNlbGVjdGVkIGJ5IGEgc3Vic3RyaW5nLCBub3QgYnkgaW5kZXgsIHNvIHRoZSByb3cgaXMKaWRlbnRpZmllZCBieSB3aGF0IGl0IGlzIHJhdGhlciB0aGFuIGJ5IHdoZXJlIGl0IGhhcHBlbnMgdG8gc2l0LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCByZQppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUkVQTyA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQucGFyZW50CkFSQ0hMQUIgPSBSRVBPIC8gInRvb2xzIiAvICJhcmNobGFiIgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKEFSQ0hMQUIpKQoKZnJvbSBpbnNwZWN0X3B0eCBpbXBvcnQgQ0FURUdPUklFUywgcGFyc2Vfc2FzcyAgIyBub3FhOiBFNDAyCgpGVU5DX1JFID0gcmUuY29tcGlsZShyIkZ1bmN0aW9uIDogKFxTKykiKQojIGN1b2JqZHVtcCAtc2FzcyBib2R5OiAiICAgICAgICAvKjAwMDAqLyAgICAgICAgICAgICAgICAgICBNT1YgUjEsIGNbMHgwXVsweDI4XSA7IgojCiMgVGhlIG9mZnNldCBmaWVsZCBpcyA0IGhleCBkaWdpdHMgZm9yIHRoZSBmaXJzdCA0MDk2IGluc3RydWN0aW9ucyBvZiBhIGtlcm5lbAojICgweDAwMDAuLjB4ZmZmMCwgNCBieXRlcyBlYWNoKSBhbmQgNSBvciBtb3JlIGFmdGVyIHRoYXQuIEEgYHs0fWAgcGF0dGVybgojIHRoZXJlZm9yZSByZXBvcnRzIEVYQUNUTFkgNDA5NiBpbnN0cnVjdGlvbnMgZm9yIGFueSBrZXJuZWwgYmlnZ2VyIHRoYW4gdGhhdCwKIyB3aGljaCBsb29rcyBsaWtlIGEgcGxhdXNpYmxlIG51bWJlciBhbmQgaXMgc2lsZW50bHkgd3Jvbmc6IHRocmVlIGtlcm5lbHMgaGVyZQojIHdpdGggMjc3LCAxMDQ1IGFuZCAyMDY5IGdsb2JhbCBsb2FkcyBhbGwgY2FtZSBvdXQgYXMgIjQwOTYgaW5zdHJ1Y3Rpb25zIi4KIyBgezQsOH1gIGlzIHdoYXQgbWFrZXMgdGhlIGNvdW50IG1lYW4gYW55dGhpbmcsIGFuZCBgdG9vbHMvYXJjaGxhYmAncwojIFNBU1NfTElORSBhbHJlYWR5IHVzZXMgezQsNn0gZm9yIHRoZSBzYW1lIHJlYXNvbi4KSU5TVFJfUkUgPSByZS5jb21waWxlKHIiL1wqWzAtOWEtZkEtRl17NCw4fVwqL1xzKyg/OkAhP1x3K1xzKyk/KFtBLVpdW0EtWjAtOV8uXSopIikKIyBgcHR4YXMgLXZgIG91dHB1dCwgd2hpY2ggaXMgdGhlIGF1dGhvcml0YXRpdmUgc291cmNlIGZvciByZWdpc3RlcnMgYW5kIHNwaWxsczoKIyAgIENvbXBpbGluZyBlbnRyeSBmdW5jdGlvbiAnTkFNRScgZm9yICdzbV83NScKIyAgICAgICAwIGJ5dGVzIHN0YWNrIGZyYW1lLCAwIGJ5dGVzIHNwaWxsIHN0b3JlcywgMCBieXRlcyBzcGlsbCBsb2FkcwojICAgVXNlZCAyOCByZWdpc3RlcnMsIHVzZWQgMSBiYXJyaWVycywgMzg4IGJ5dGVzIGNtZW1bMF0KUFRYVl9FTlRSWSA9IHJlLmNvbXBpbGUociJDb21waWxpbmcgZW50cnkgZnVuY3Rpb24gJyhbXiddKyknIGZvciAnKFxTKyknIikKUFRYVl9GUkFNRSA9IHJlLmNvbXBpbGUociIoXGQrKSBieXRlcyBzdGFjayBmcmFtZSwgKFxkKykgYnl0ZXMgc3BpbGwgc3RvcmVzLCAiCiAgICAgICAgICAgICAgICAgICAgICAgIHIiKFxkKykgYnl0ZXMgc3BpbGwgbG9hZHMiKQpQVFhWX1VTRUQgPSByZS5jb21waWxlKHIiVXNlZCAoXGQrKSByZWdpc3RlcnMsIHVzZWQgKFxkKykgYmFycmllcnMiKQoKCmRlZiBzcGxpdF9mdW5jdGlvbnMoc2Fzczogc3RyKSAtPiBkaWN0W3N0ciwgc3RyXToKICAgICIiImN1b2JqZHVtcCAtc2FzcyB0ZXh0IC0+IHtmdW5jdGlvbiBuYW1lOiB0aGF0IGZ1bmN0aW9uJ3MgYm9keX0uIiIiCiAgICBvdXQ6IGRpY3Rbc3RyLCBzdHJdID0ge30KICAgIGN1ciA9IE5vbmUKICAgIGZvciBsaW5lIGluIHNhc3Muc3BsaXRsaW5lcygpOgogICAgICAgIG0gPSBGVU5DX1JFLnNlYXJjaChsaW5lKQogICAgICAgIGlmIG06CiAgICAgICAgICAgIGN1ciA9IG0uZ3JvdXAoMSkKICAgICAgICAgICAgb3V0W2N1cl0gPSBbXQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIGN1ciBpcyBub3QgTm9uZToKICAgICAgICAgICAgb3V0W2N1cl0uYXBwZW5kKGxpbmUpCiAgICByZXR1cm4ge2s6ICJcbiIuam9pbih2KSBmb3IgaywgdiBpbiBvdXQuaXRlbXMoKX0KCgpkZWYgb3Bjb2Rlcyhib2R5OiBzdHIpIC0+IGxpc3Rbc3RyXToKICAgIHJldHVybiBJTlNUUl9SRS5maW5kYWxsKGJvZHkpCgoKZGVmIHBhcnNlX3B0eGFzX3YodGV4dDogc3RyKSAtPiBkaWN0W3N0ciwgZGljdF06CiAgICAiIiJgcHR4YXMgLXZgIHRleHQgLT4ge2Z1bmN0aW9uIG5hbWU6IHtyZWdpc3RlcnMsIHNwaWxsXyosIGJhcnJpZXJzLCAuLi59fS4KCiAgICBNYXRjaGVkIEJZIE5BTUUsIG5vdCBieSBwb3NpdGlvbi4gQm90aCB0aGUgbWFuZ2xlZCBuYW1lIGFuZCB0aGUgcGxhaW4gbmFtZQogICAgYXJlIGluZGV4ZWQsIGJlY2F1c2UgYC1zYXNzYCBwcmludHMgdGhlIG1hbmdsZWQgbmFtZSBhbmQgdGhlIGxhdW5jaGVyCiAgICB0YWJsZXMgYW5kIHRoZSBub3RlcyB1c2UgdGhlIHBsYWluIG9uZSwgYW5kIHBhaXJpbmcgdHdvIGxpc3RzIGJ5IGluZGV4IGlzCiAgICBob3cgYSByZWdpc3RlciBjb3VudCBlbmRzIHVwIGF0dGFjaGVkIHRvIHRoZSB3cm9uZyBrZXJuZWwuCiAgICAiIiIKICAgIG91dDogZGljdFtzdHIsIGRpY3RdID0ge30KICAgIGN1ciA9IE5vbmUKICAgIGZvciBsaW5lIGluIHRleHQuc3BsaXRsaW5lcygpOgogICAgICAgIG0gPSBQVFhWX0VOVFJZLnNlYXJjaChsaW5lKQogICAgICAgIGlmIG06CiAgICAgICAgICAgIGN1ciA9IHsiYXJjaCI6IG0uZ3JvdXAoMil9CiAgICAgICAgICAgIG91dFttLmdyb3VwKDEpXSA9IGN1cgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIGN1ciBpcyBOb25lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG0gPSBQVFhWX0ZSQU1FLnNlYXJjaChsaW5lKQogICAgICAgIGlmIG06CiAgICAgICAgICAgIGN1clsic3RhY2tfZnJhbWUiXSA9IGludChtLmdyb3VwKDEpKQogICAgICAgICAgICBjdXJbInNwaWxsX3N0b3JlcyJdID0gaW50KG0uZ3JvdXAoMikpCiAgICAgICAgICAgIGN1clsic3BpbGxfbG9hZHMiXSA9IGludChtLmdyb3VwKDMpKQogICAgICAgICAgICAjIHRoZSBzcGlsbCBjb2x1bW46IGFueSBub24temVybyBoZXJlIG1lYW5zIHRoZSByZWdpc3RlciBmaWxlIGNvdWxkCiAgICAgICAgICAgICMgbm90IGhvbGQgdGhlIGxpdmUgc2V0IGFuZCB0aGUga2VybmVsIGlzIHBheWluZyBtZW1vcnkgdHJhZmZpYyBmb3IKICAgICAgICAgICAgIyBpdHMgb3duIGFyaXRobWV0aWMKICAgICAgICAgICAgY3VyWyJsb2NhbF9ieXRlcyJdID0gaW50KG0uZ3JvdXAoMikpICsgaW50KG0uZ3JvdXAoMykpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbSA9IFBUWFZfVVNFRC5zZWFyY2gobGluZSkKICAgICAgICBpZiBtOgogICAgICAgICAgICBjdXJbInJlZ2lzdGVycyJdID0gaW50KG0uZ3JvdXAoMSkpCiAgICAgICAgICAgIGN1clsiYmFycmllcnMiXSA9IGludChtLmdyb3VwKDIpKQogICAgcmV0dXJuIG91dAoKCmRlZiByb3cobmFtZTogc3RyLCBib2R5OiBzdHIsIHJlczogZGljdCB8IE5vbmUpIC0+IGRpY3Q6CiAgICBvcHMgPSBvcGNvZGVzKGJvZHkpCiAgICBzdCA9IHBhcnNlX3Nhc3MoYm9keSkKICAgIHI6IGRpY3QgPSB7Imtlcm5lbCI6IG5hbWUsICJpbnN0cnVjdGlvbnMiOiBsZW4ob3BzKX0KICAgIGZvciBjYXQsIF8sIF8gaW4gQ0FURUdPUklFUzoKICAgICAgICByW2NhdF0gPSBzdFsiY2F0ZWdvcmllcyJdLmdldChjYXQsIDApCiAgICByWyJic3NfYnN5bmNfcGFpcnMiXSA9IG9wcy5jb3VudCgiQlNTWSIpCiAgICByWyJicmFuY2giXSA9IG9wcy5jb3VudCgiQlJBIikKICAgIHJbIm11ZnVfcmNwIl0gPSBvcHMuY291bnQoIk1VRlUuUkNQIikKICAgIHJbIm11ZnVfcnNxIl0gPSBvcHMuY291bnQoIk1VRlUuUlNRIikKICAgIHJbImlzZXRwX3UzMiJdID0gc3VtKDEgZm9yIG8gaW4gb3BzIGlmIG8uc3RhcnRzd2l0aCgiSVNFVFAiKSBhbmQgIlUzMiIgaW4gbykKICAgIHJbImxkZ19lX3M4Il0gPSBzdW0oMSBmb3IgbyBpbiBvcHMgaWYgby5zdGFydHN3aXRoKCJMREcuRSIpIGFuZCAiLlM4IiBpbiBvKQogICAgclsic3RnX3U4Il0gPSBzdW0oMSBmb3IgbyBpbiBvcHMgaWYgby5zdGFydHN3aXRoKCJTVEcuRS5VOCIpKQogICAgclsiZ2xvYmFsX2xkX3dpZHRocyJdID0gc3RbImdsb2JhbF9sZF93aWR0aHMiXQogICAgclsiZ2xvYmFsX3N0X3dpZHRocyJdID0gc3RbImdsb2JhbF9zdF93aWR0aHMiXQogICAgclsiYW55XzEyOGJpdCJdID0gc3RbImFueV8xMjhiaXQiXQogICAgaWYgcmVzOgogICAgICAgIGZvciBrIGluICgicmVnaXN0ZXJzIiwgImJhcnJpZXJzIiwgInNwaWxsX3N0b3JlcyIsICJzcGlsbF9sb2FkcyIsCiAgICAgICAgICAgICAgICAgICJsb2NhbF9ieXRlcyIsICJzdGFja19mcmFtZSIpOgogICAgICAgICAgICByW2tdID0gcmVzLmdldChrKQogICAgZWxzZToKICAgICAgICBmb3IgayBpbiAoInJlZ2lzdGVycyIsICJiYXJyaWVycyIsICJzcGlsbF9zdG9yZXMiLCAic3BpbGxfbG9hZHMiLAogICAgICAgICAgICAgICAgICAibG9jYWxfYnl0ZXMiLCAic3RhY2tfZnJhbWUiKToKICAgICAgICAgICAgcltrXSA9IE5vbmUKICAgIHJldHVybiByCgoKZGVmIHN1bW1hcmlzZShyOiBkaWN0KSAtPiBzdHI6CiAgICBkZWYgX24odiwgd2lkdGg9NCk6CiAgICAgICAgcmV0dXJuIGYie3Y6e3dpZHRofX0iIGlmIGlzaW5zdGFuY2UodiwgaW50KSBlbHNlIGYieyc/Jzo+e3dpZHRofX0iCgogICAgcmV0dXJuIChmIntyWydrZXJuZWwnXVs6NDRdOjQ0c30gaW5zdHI9e3JbJ2luc3RydWN0aW9ucyddOjVkfSAiCiAgICAgICAgICAgIGYicmVnPXtfbihyLmdldCgncmVnaXN0ZXJzJykpfSAiCiAgICAgICAgICAgIGYic3BpbGw9e19uKHIuZ2V0KCdsb2NhbF9ieXRlcycpKX0gIgogICAgICAgICAgICBmInNtZW09e19uKHIuZ2V0KCdzaGFyZWRfYnl0ZXMnKSwgNSl9ICIKICAgICAgICAgICAgZiJMREc9e3JbJ2dsb2JhbF9sZCddOjRkfSBTVEc9e3JbJ2dsb2JhbF9zdCddOjRkfSAiCiAgICAgICAgICAgIGYiRkZNQT17clsnc2NhbGFyX2ZmbWEnXTo0ZH0gQkFSPXtfbihyLmdldCgnYmFycmllcnMnKSwgMyl9ICIKICAgICAgICAgICAgZiJCU1NZPXtyWydic3NfYnN5bmNfcGFpcnMnXTo0ZH0iKQoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNhc3MiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUsIGhlbHA9ImN1b2JqZHVtcCAtc2FzcyB0ZXh0IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1yZXMiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJwdHhhcyAtdiB0ZXh0IChyZWdpc3RlcnMsIHNwaWxscywgYmFycmllcnMpLCBtYXRjaGVkIGJ5ICIKICAgICAgICAgICAgICAgICAgICAgICAgICJmdW5jdGlvbiBuYW1lIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWxlY3QiLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ic3Vic3RyaW5nOyBvbmx5IGZ1bmN0aW9ucyBjb250YWluaW5nIGl0IGFyZSByZXBvcnRlZCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tanNvbiIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhID0gYXAucGFyc2VfYXJncygpCgogICAgc2FzcyA9IGEuc2Fzcy5yZWFkX3RleHQoKQogICAgcmVzID0gcGFyc2VfcHR4YXNfdihhLnJlcy5yZWFkX3RleHQoKSkgaWYgYS5yZXMgYW5kIGEucmVzLmV4aXN0cygpIGVsc2Uge30KICAgIGZ1bmNzID0gc3BsaXRfZnVuY3Rpb25zKHNhc3MpCiAgICBuYW1lcyA9IGxpc3QoZnVuY3MpCiAgICByb3dzID0gW10KICAgIGZvciBuLCBib2R5IGluIGZ1bmNzLml0ZW1zKCk6CiAgICAgICAgaWYgYS5zZWxlY3QgYW5kIGEuc2VsZWN0IG5vdCBpbiBuOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJvd3MuYXBwZW5kKHJvdyhuLCBib2R5LCByZXMuZ2V0KG4pKSkKCiAgICBpZiBhLmpzb246CiAgICAgICAgcHJpbnQoanNvbi5kdW1wcyh7InNhc3MiOiBzdHIoYS5zYXNzKSwgImZ1bmN0aW9ucyI6IG5hbWVzLCAicm93cyI6IHJvd3N9LAogICAgICAgICAgICAgICAgICAgICAgICAgaW5kZW50PTIpKQogICAgICAgIHJldHVybiAwCgogICAgcHJpbnQoZiJzYXNzOiB7YS5zYXNzfSIpCiAgICBwcmludChmInsna2VybmVsJzo0NHN9IHsnaW5zdHInOj4xMH0geydyZWcnOj41fSB7J3NwaWxsJzo+NX0geydzbWVtJzo+NX0gIgogICAgICAgICAgZiJ7J0xERyc6PjR9IHsnU1RHJzo+NH0geydGRk1BJzo+NH0geydCQVInOj4zfSB7J0JTU1knOj40fSIpCiAgICBmb3IgciBpbiByb3dzOgogICAgICAgIHByaW50KHN1bW1hcmlzZShyKSkKICAgIGlmIG5vdCByb3dzOgogICAgICAgIHByaW50KGYiICAobm8gZnVuY3Rpb24gbWF0Y2hlZCAtLXNlbGVjdCB7YS5zZWxlY3Qhcn07ICIKICAgICAgICAgICAgICBmIntsZW4obmFtZXMpfSBmdW5jdGlvbnMgaW4gdGhlIGZpbGUpIikKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK", "dedf5437ebfbaf0b574203f7dd5960d895cf2c6aeb260936b4834aed795a23af"], ["tools/archlab/inspect_ptx.py", "..."]]
BATCHES = [512, 1024, 2048, 4096, 8192, 16384, 32768]
TIME_BATCHES = [256, 512, 1024, 2048]
CODEGEN_STEPS = "1,4,16,32"
DUMP_BATCH = 4096
ITERS = 8
WARMUP = 5

lines = []
def say(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    lines.append(s)

def die(code, why):
    say("")
    say("FATAL: " + why)
    open(f"{OUT}/SUMMARY.txt", "w").write("\n".join(lines) + f"\nFATAL {why}\n")
    sys.exit(code)

def sh(cmd, **kw):
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    return p.returncode, p.stdout, p.stderr

say("=" * 78)
say("minecu_tg -- TinyGrad vs the hand-written CUDA, on this T4")
say("=" * 78)

# ------------------------------------------------------------- provenance
say("")
say("--- provenance: every payload is hash-checked before anything is built ---")
for name, b64, want in PAYLOADS:
    data = base64.b64decode(b64)
    got = hashlib.sha256(data).hexdigest()
    if got != want:
        die(9, f"{name} sha256 {got} != {want}; refusing to measure a source I "
               f"cannot identify. Regenerate with minecu_tg/kaggle/run.sh.")
    dest = f"{OUT}/{name}"                 # name is repo-relative, so the
    os.makedirs(os.path.dirname(dest), exist_ok=True)   # package stays a package
    open(dest, "wb").write(data)
    say(f"  {name:34s} {len(data):8d} B  sha256 {got[:16]}  OK")

# ------------------------------------------------------------------- pip
# ORDERING, and it is load-bearing. The package is written first because
# `import minecu_tg.render` needs both the package and tinygrad; the import
# CHECK is deferred until after the pip install for the same reason. Checking it
# in between, which is where it sat once, fails with ModuleNotFoundError:
# No module named 'tinygrad' -- a real error, reported in the right place, but
# one this notebook did not need to have.
sys.path.insert(0, OUT)

say("")
say("--- tinygrad ---")
rc, out, err = sh([sys.executable, "-m", "pip", "install", "-q", "tinygrad"])
say(f"  pip install tinygrad rc={rc}")
if rc != 0:
    say("  " + (err or out)[-400:].replace("\n", "\n  "))
try:
    import tinygrad
    say(f"  tinygrad at {tinygrad.__file__}")
except Exception as e:
    die(12, f"tinygrad did not import: {e}")

# now that tinygrad exists, prove the package we just laid down is importable.
# This is the check that would have caught the flattened-payload bug, which
# killed the previous push at this exact point AFTER the reference side had
# already spent its time.
try:
    import importlib
    _m = importlib.import_module("minecu_tg.render")
    for _s in ("blocks", "world", "cpu_reference", "codegen", "bench",
               "battery", "capture_cuda", "analyse_codegen", "sass_matrix"):
        importlib.import_module(f"minecu_tg.{_s}")
    say(f"  import minecu_tg.render OK, from {_m.__file__}")
    say("  all 9 minecu_tg submodules import OK")
except Exception as e:
    die(11, f"the minecu_tg package did not import from {OUT}: {e}")

# ----------------------------------------------------------------- build
say("")
say("--- build the hand-written side for sm_75 ---")
NVCC = None
for p in ("/usr/local/cuda/bin/nvcc", "nvcc"):
    rc, _, _ = sh([p, "--version"])
    if rc == 0:
        NVCC = p
        break
if NVCC is None:
    die(10, "no nvcc on this runner; the T4 image is supposed to have one")
say("  " + "\n  ".join(sh([NVCC, "--version"])[1].strip().splitlines()[-2:]))

# -arch=sm_75 explicit: the T4 is sm_75 and this run exists to produce numbers
# for that architecture, not for whatever nvcc would default to.
#
# refdrv.cu and bench.cu need OPPOSITE treatment, and getting this wrong is a
# LINK error rather than a compile error, so it is worth writing down. The first
# push of this notebook cost one run to it:
#   * refdrv.cu `#include`s kernels.cu, so it is compiled ALONE. Passing
#     kernels.cu on the command line as well compiles it a second time and the
#     linker reports `multiple definition of launch_render_uint8_prebasis' and
#     eight others. The local pre-flight in run.sh already compiled refdrv.cu
#     alone and PASSED, because it was not running the notebook's command -- so
#     both are now the same command, and the pre-flight is a real gate rather
#     than a decorative one.
#   * bench.cu does NOT include kernels.cu; it declares the `extern "C"`
#     launchers and links against them, so it needs kernels.cu on the command
#     line. That is how minecu_ref/build.sh builds it, and the reference lane's
#     binary is built the same way here.
for label, srcs, outbin, extra in (
    ("refdrv", [f"{OUT}/refdrv.cu"], f"{OUT}/refdrv_sm_75",
     ['-DMINECU_KERNELS="%s/kernels.cu"' % OUT]),
    ("bench",  [f"{OUT}/kernels.cu", f"{OUT}/bench.cu"],  f"{OUT}/bench_sm_75", []),
):
    t0 = time.time()
    rc, out, err = sh([NVCC, "-O3", "--use_fast_math", "-std=c++17", "-arch=sm_75",
                       "-diag-suppress", "177", *extra, *srcs, "-o", outbin])
    say(f"  nvcc {label:8s} rc={rc} in {time.time()-t0:.1f}s -> {os.path.basename(outbin)}")
    if err.strip():
        for ln in err.strip().splitlines()[-8:]:
            say("  | " + ln)
    if rc != 0:
        die(11, f"nvcc failed for {label}")

# prove the binary really is sm_75 with no sm_80+ instructions
rc, sass, _ = sh(["/usr/local/cuda/bin/cuobjdump", "-sass", f"{OUT}/refdrv_sm_75"])
gates = ["HMMA", "HGMMA", "LDGSTS", "REDUX", "ELECT", "BAR.RED", "F2FP"]
say("  SASS census of the reference binary: " + "  ".join(
    f"{g}={sass.count(g)}" for g in gates))
if any(sass.count(g) for g in gates):
    die(13, "the reference binary contains sm_80+ instructions; it would not run "
            "on a T4 and the comparison would be meaningless")

# ------------------------------------------------- the reference battery
say("")
say("--- the hand-written side, per batch (cudaEvent, this card, this run) ---")
ref = {}
for b in BATCHES:
    t0 = time.time()
    rc, out, err = sh([f"{OUT}/refdrv_sm_75", "--batch", str(b), "--world", "16",
                       "--res", "32", "--max-steps", "32", "--iters", str(ITERS),
                       "--warmup", str(WARMUP)])
    for ln in out.splitlines():
        say("  " + ln)
    if rc != 0:
        say(f"  batch {b}: refdrv rc={rc}: {(err or '')[-200:]}")
    d = {"batch": b, "rc": rc}
    for ln in out.splitlines():
        m = re.match(r"\s*RENDER_US\s+(\S+)\s+(\S+)\s+(\S+)\s+(\d+)", ln)
        if m: d["render_us"] = [float(m.group(i)) for i in (1, 2, 3)]; d["render_iters"] = int(m.group(4))
        m = re.match(r"\s*STEP_US\s+(\S+)\s+(\S+)\s+(\S+)\s+(\d+)", ln)
        if m: d["step_us"] = [float(m.group(i)) for i in (1, 2, 3)]
    ref[b] = d
    say(f"  batch {b:6d}  render {d.get('render_us',[0])[0]:9.1f} us  "
        f"step {d.get('step_us',[0])[0]:9.1f} us  ({time.time()-t0:.1f}s)")
json.dump(ref, open(f"{OUT}/ref_timing.json", "w"), indent=2)

# the reference's own device-measured DDA statistics at the dump batch, so the
# two sides' workloads can be compared rather than assumed equal
say("")
say(f"--- the reference's own DDA statistics at batch {DUMP_BATCH} ---")
rc, out, err = sh([f"{OUT}/bench_sm_75", "--batch", str(DUMP_BATCH), "--world", "16",
                   "--res", "32", "--max-steps", "32", "--iters", "10",
                   "--graph-iters", "10"])
open(f"{OUT}/log_bench_dda.txt", "w").write(out + ("\n[stderr]\n" + err if err else ""))
for ln in out.splitlines():
    if ("DDA" in ln or "selftest" in ln or "mean" in ln or "solid" in ln
            or "launch_step_uint8_prebasis" in ln or "render uint8 prebasis" in ln):
        say("  " + ln)
say(f"  (full log: log_bench_dda.txt, rc={rc})")

# ------------------------------------------------------------- the dump
say("")
say(f"--- the hand-written side dumps its settled state at batch {DUMP_BATCH} ---")
os.makedirs(f"{OUT}/refstate", exist_ok=True)
rc, out, err = sh([f"{OUT}/refdrv_sm_75", "--batch", str(DUMP_BATCH), "--world", "16",
                   "--res", "32", "--max-steps", "32", "--iters", str(ITERS),
                   "--warmup", str(WARMUP), "--dump", "--outdir", f"{OUT}/refstate"])
for ln in out.splitlines():
    say("  " + ln)
if rc != 0:
    die(14, f"refdrv --dump failed rc={rc}: {(err or '')[-300:]}")
open(f"{OUT}/log_refdump.txt", "w").write(out)

# ------------------------------------------------------- the TinyGrad side
say("")
say("=" * 78)
say("  the TinyGrad side")
say("=" * 78)
rc, out, err = sh([sys.executable, f"{OUT}/minecu_tg/battery.py",
                   "--out", f"{OUT}/tg", "--world", "16", "--res", "32",
                   "--max-steps", "32",
                   "--census-batch", "64", "--census-steps", CODEGEN_STEPS,
                   "--mem-batches", ",".join(str(b) for b in BATCHES),
                   "--time-batches", ",".join(str(b) for b in TIME_BATCHES),
                   "--iters", str(ITERS)], timeout=2400)
say(out)
if err.strip():
    say("  [battery stderr tail]")
    for ln in err.strip().splitlines()[-20:]:
        say("  " + ln)
if rc not in (0, 2):
    say(f"  battery.py rc={rc}")

# --------------------------------------------- observation equivalence
say("")
say("--- observation equivalence, on the state the reference itself produced ---")
import numpy as np
RS = f"{OUT}/refstate"
def load(name, dtype):
    return np.fromfile(f"{RS}/{name}.bin", dtype=dtype)
try:
    vox = load("voxels", np.int8).reshape(DUMP_BATCH, 16, 16, 16)
    basis = load("basis13", np.float32).reshape(DUMP_BATCH, 13)
    ref_obs = load("obs_u8", np.uint8).reshape(DUMP_BATCH, 32, 32, 3)
    say(f"  voxels  {vox.shape}  basis {basis.shape}  obs {ref_obs.shape}")
except Exception as e:
    die(15, f"could not read the reference dump: {e}")

from tinygrad import Tensor, dtypes, Device
from minecu_tg.render import render, camera_basis

t0 = time.time()
voxT = Tensor(vox, dtype=dtypes.int8)
basisT = Tensor(basis, dtype=dtypes.float32)
obs, steps = render(voxT, basisT, 32, 32, 32, count_steps=True)
tg_obs = obs.numpy()
tg_steps = steps.numpy().reshape(DUMP_BATCH, 32, 32)
say(f"  rendered on {Device.DEFAULT} in {time.time()-t0:.1f}s")

diff = (tg_obs.astype(np.int32) != ref_obs.astype(np.int32))
bad_px = int(diff.any(axis=3).sum())
bad_b = int(diff.any(axis=3).any(axis=2).any(axis=1).sum())
npx = DUMP_BATCH * 32 * 32
say(f"  pixels differing from the reference : {bad_px} / {npx} "
    f"({100.0 * bad_px / npx:.4f}%)")
say(f"  envs with at least one differing pixel: {bad_b} / {DUMP_BATCH}")
say(f"  max channel difference              : {int(np.abs(diff).max())}")
say(f"  distinct colours, reference         : {len(np.unique(ref_obs.reshape(-1, 3), axis=0))}")
say(f"  distinct colours, tinygrad          : {len(np.unique(tg_obs.reshape(-1, 3), axis=0))}")
sm = int((tg_steps == 32).sum())
say(f"  tinygrad DDA: mean {tg_steps.mean():.3f}  min {int(tg_steps.min())}  "
    f"max {int(tg_steps.max())}  at cap {100.0 * sm / npx:.1f}%  "
    f"first voxel {100.0 * int((tg_steps == 1).sum()) / npx:.1f}%")
import hashlib as _h
say(f"  obs sha256 reference {  _h.sha256(np.ascontiguousarray(ref_obs)).hexdigest()[:32]}")
say(f"  obs sha256 tinygrad  {_h.sha256(np.ascontiguousarray(tg_obs)).hexdigest()[:32]}")
json.dump({"pixels_differing": bad_px, "pixels": npx,
           "envs_differing": bad_b, "batch": DUMP_BATCH,
           "dda_mean": float(tg_steps.mean()),
           "dda_at_cap_frac": float(sm / npx),
           "ref_obs_sha256": _h.sha256(np.ascontiguousarray(ref_obs)).hexdigest(),
           "tg_obs_sha256": _h.sha256(np.ascontiguousarray(tg_obs)).hexdigest()},
          open(f"{OUT}/obs_equivalence.json", "w"), indent=2)

# ------------------------------------------- generated CUDA, compiled
say("")
say("--- the generated CUDA, compiled for sm_75 ---")
CG = f"{OUT}/tg_codegen"
rc, out, err = sh([sys.executable, f"{OUT}/minecu_tg/capture_cuda.py",
                   "--arch", "sm_75", "--batch", "8", "--world", "16",
                   "--res", "32", "--steps", CODEGEN_STEPS,
                   "--outdir", CG], timeout=1200)
say(out)
if err.strip():
    for ln in err.strip().splitlines()[-10:]:
        say("  " + ln)

rc, out, err = sh([sys.executable, f"{OUT}/minecu_tg/analyse_codegen.py", CG])
say(out)

# per-kernel nvcc + ptxas -v for the hot shape, so registers and spills are real
say("")
say("--- nvcc -arch=sm_75 and ptxas -v on the generated kernels at max_steps=32 ---")
res_rows = []
msdir = f"{CG}/max_steps_0032"
if os.path.isdir(msdir):
    for f in sorted(os.listdir(msdir)):
        if not f.endswith(".cu"):
            continue
        stem = f[:-3]
        rc, o, e = sh(["/usr/local/cuda/bin/nvcc", "-arch=sm_75", "--use_fast_math",
                       "-std=c++17", "-cubin", f"{msdir}/{f}",
                       "-o", f"{CG}/{stem}.cubin"])
        if rc != 0:
            say(f"  {f}: nvcc FAILED")
            for ln in (e or "").splitlines()[-4:]:
                say("    " + ln)
            continue
        rc, o, e = sh(["/usr/local/cuda/bin/cuobjdump", "-sass", f"{CG}/{stem}.cubin"])
        open(f"{CG}/{stem}.sass", "w").write(o)
        rc2, o2, e2 = sh(["/usr/local/cuda/bin/nvcc", "-arch=sm_75", "--use_fast_math",
                           "-std=c++17", "-ptx", f"{msdir}/{f}", "-o", f"{CG}/{stem}.ptx"])
        rc3, o3, e3 = sh(["/usr/local/cuda/bin/ptxas", "-arch=sm_75", "-v",
                           f"{CG}/{stem}.ptx", "-o", f"{CG}/{stem}.ptxas.cubin"])
        open(f"{CG}/{stem}.ptxas.txt", "w").write(o3 + (e3 or ""))
        reg = re.search(r"Used (\d+) registers", o3 + (e3 or ""))
        spill = re.search(r"(\d+) bytes spill stores, (\d+) bytes spill loads",
                          o3 + (e3 or ""))
        frame = re.search(r"(\d+) bytes stack frame", o3 + (e3 or ""))
        ninstr = len(re.findall(r"/\*[0-9a-f]{4}\*/", o))
        res_rows.append({"file": f,
                         "instructions": ninstr,
                         "registers": int(reg.group(1)) if reg else None,
                         "spill_stores": int(spill.group(1)) if spill else None,
                         "spill_loads": int(spill.group(2)) if spill else None,
                         "stack_frame": int(frame.group(1)) if frame else None})
        say(f"  {f:14s} instr={ninstr:5d}  reg={reg.group(1) if reg else '?':>4}"
            f"  spill_st={spill.group(1) if spill else '?':>5}"
            f"  spill_ld={spill.group(2) if spill else '?':>5}"
            f"  frame={frame.group(1) if frame else '?':>5}")
    json.dump(res_rows, open(f"{CG}/ptxas_resources.json", "w"), indent=2)

# ------------------------------------------------------------- the summary
say("")
say("=" * 78)
say("SUMMARY")
say("=" * 78)

say("")
say("THE SCHEDULE (the prediction under test)")
say("  A Python loop over DDA steps unrolls at trace time. The question is")
say("  whether tinygrad fuses across the unrolled body or emits one kernel per")
say("  step. These are the counts, per max_steps, on this card:")
try:
    cens = json.load(open(f"{OUT}/tg/report.json"))["census"]
    say(f"  {'max_steps':>10} {'launches':>9} {'distinct':>9} {'launches-8':>11}")
    for r in cens["rows"]:
        say(f"  {r['max_steps']:>10} {r['kernel_launches']:>9} "
            f"{r['distinct_programs_launched']:>9} "
            f"{r['kernel_launches'] - 8:>11}")
except Exception as e:
    say(f"  census unavailable: {e}")

say("")
say("PERFORMANCE, both sides, same card, same run, same world, max_steps=32")
say("  ref render / ref step are cudaEvent medians in MICROSECONDS. tg is the")
say("  sum of per-kernel device times for the whole generated graph.")
say(f"  {'batch':>7} {'ref render us':>14} {'ref step us':>13} "
    f"{'tg device ms':>13} {'tg/ref':>8} {'tg peak MiB':>12} {'tg state':>22}")
try:
    rep = json.load(open(f"{OUT}/tg/report.json"))
    tg = {r["batch"]: r for r in rep.get("timing", []) if r.get("batch")}
    mem = {r["batch"]: r for r in rep.get("memory", []) if r.get("batch")}
    for b in sorted(set(list(TIME_BATCHES) + list(BATCHES))):
        r = ref.get(b, {})
        t = tg.get(b, {})
        rr = r.get("render_us", [None])[0]
        rs = r.get("step_us", [None])[0] if r.get("step_us") else None
        dm = (t.get("device_time") or {}).get("median_ms")
        peak = (t.get("mem") or {}).get("peak_live_bytes", 0) / 2**20
        ratio = (dm * 1000.0 / rr) if (rr and dm) else None
        m = mem.get(b)
        if m is None:
            state = "not attempted"
        elif m.get("realized"):
            state = f"fits, {m['mem']['peak_live_bytes']/2**20:.0f} MiB"
        else:
            state = "DOES NOT ALLOCATE"
        say(f"  {b:>7} {round(rr,1) if rr else '-':>14} {round(rs,1) if rs else '-':>13} "
            f"{round(dm,3) if dm else '-':>13} {round(ratio,2) if ratio else '-':>8} "
            f"{round(peak,1) if peak else '-':>12} {state:>22}")
except Exception as e:
    say(f"  performance table unavailable: {e}")

say("")
say("WHAT TO READ")
say("  * A framework emits a graph of kernels; the 4.55 M baseline is a")
say("    whole-step number over 7 launches. Dividing one by the other is a")
say("    category error. The defensible comparison is render against render at")
say("    the same batch, and the batch table above is the same-card, same-run")
say("    version of that.")
say("  * Every timing is printed next to the DDA step count of its own run.")
say("    A throughput number without the workload behind it is not a")
say("    measurement; see minecu_ref/buried_agent_fix.md for what that costs.")
say("  * No number here is transcribed from any notes file. Every figure is")
say("    produced by an instrument in this notebook on this card.")

open(f"{OUT}/SUMMARY.txt", "w").write("\n".join(lines) + "\n")
say("")
say("wrote /kaggle/working/SUMMARY.txt")
